<a href="https://colab.research.google.com/github/chantakeru16-ship-it/staff-break-monitor/blob/main/BTAxDashboardxAcsenda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install streamlit pandas numpy plotly pyngrok

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
%%writefile app.py
import os
import streamlit as st
import pandas as pd
import plotly.express as px

# This is additional requirement to import for part G5-Lazy Text Reward Farming - BERNIE, ERIKO, JULLIE
import numpy as np
import plotly.graph_objects as go
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# This is additional requirement to import for part G3-Utilization Analysis - ANGELA, JAIME, JUNNAR
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json, re, warnings, os, base64
warnings.filterwarnings('ignore')

# ⚠️ IMPORTANT:
# Only edit your assigned group section.
# Do NOT modify Dashboard, Sidebar, CSS, Imports, or another group's section.


# =============================================================
# BTA DATA ANALYTICS DASHBOARD - CLEAN GOOGLE COLAB VERSION
# =============================================================
# Google Drive folder structure:
#
# MyDrive/BTAxDashboardxAcsenda/
# ├── Main CSV Files/
# │   ├── dim_questions_202605121902.csv
# │   ├── dim_surveys_202605121902.csv
# │   ├── dim_users_202605121902.csv
# │   ├── fct_survey_responses_202605121902.csv
# │   └── wp_users_202605121903.csv
# ├── G1/
# ├── G2/
# ├── G3/
# ├── G4/
# └── G5/
#
# This clean version only loads the MAIN CSV files.
# G1-G5 tabs are placeholders for classmates' future files/code.
# =============================================================


# Main Google Drive folder name used to locate all dashboard files.
DRIVE_FOLDER_NAME = "BTAxDashboardxAcsenda"


# Finds the BTA project folder in Google Drive, even if it was moved inside another folder.
def find_drive_base():
    expected_path = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"

    if os.path.exists(expected_path):
        return expected_path

    mydrive = "/content/drive/MyDrive"
    if os.path.exists(mydrive):
        for root, dirs, files in os.walk(mydrive):
            for folder in dirs:
                if folder.lower() == DRIVE_FOLDER_NAME.lower():
                    return os.path.join(root, folder)

    return expected_path


DRIVE_BASE = find_drive_base()
MAIN_CSV_FOLDER = os.path.join(DRIVE_BASE, "Main CSV Files")

# Sets the Streamlit page title, icon, layout, and sidebar behavior.
st.set_page_config(
    page_title="BTA Data Analytics Dashboard",
    page_icon="🌿",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =============================================================
# DATA LOADING
# =============================================================

# Stores any missing required CSV filenames so the dashboard can show a clear warning.
missing_required_files = []


# Lists possible locations where a required CSV file may be found.
def possible_paths(filename):
    return [
        os.path.join(MAIN_CSV_FOLDER, filename),
        os.path.join(DRIVE_BASE, filename),
        filename,
        os.path.join("src", filename),
    ]


# Reads one required CSV file and returns an empty dataframe if the file is missing or unreadable.
def read_required_csv(filename):
    for path in possible_paths(filename):
        if os.path.exists(path):
            try:
                return pd.read_csv(path)
            except Exception as e:
                st.error(f"Could not read {filename}: {e}")
                return pd.DataFrame()

    missing_required_files.append(filename)
    return pd.DataFrame()


# Loads all main BTA datasets once and caches them to improve dashboard performance.
@st.cache_data(show_spinner=False)
def load_main_data(_drive_base, _main_csv_folder):
    questions = read_required_csv("dim_questions_202605121902.csv")
    surveys = read_required_csv("dim_surveys_202605121902.csv")
    users = read_required_csv("dim_users_202605121902.csv")
    fct = read_required_csv("fct_survey_responses_202605121902.csv")
    wp = read_required_csv("wp_users_202605121903.csv")

    if "created_at" in fct.columns:
        fct["created_at"] = pd.to_datetime(fct["created_at"], errors="coerce", utc=True)

    if "user_registered" in wp.columns:
        wp["user_registered"] = pd.to_datetime(wp["user_registered"], errors="coerce", utc=True)

    for col in ["reward_points"]:
        if col in fct.columns:
            fct[col] = pd.to_numeric(fct[col], errors="coerce")

    for col in ["lifetime_points_earned", "current_points_balance"]:
        if col in users.columns:
            users[col] = pd.to_numeric(users[col], errors="coerce")

    return questions, surveys, users, fct, wp


questions, surveys, users, fct, wp = load_main_data(DRIVE_BASE, MAIN_CSV_FOLDER)


# =============================================================
# GLOBAL FILTER HELPERS
# =============================================================

def first_existing_col(df, possible_names):
    """Return the first matching column name from a dataframe."""
    if df is None or df.empty:
        return None

    lower_map = {str(col).lower().strip(): col for col in df.columns}
    for name in possible_names:
        key = name.lower().strip()
        if key in lower_map:
            return lower_map[key]
    return None


# Standardizes filter values by replacing blanks or missing values with "Unknown".
def clean_filter_value(series):
    return series.fillna("Unknown").astype(str).str.strip().replace("", "Unknown")


# Identifies unknown or internal user types that can be excluded from dashboard metrics.
def is_unknown_value(series):
    cleaned = clean_filter_value(series).str.lower()
    return cleaned.isin([
        "unknown", "unk", "none", "null", "nan", "n/a", "na", "", "not available",
        "admin", "administrator", "intern", "security intern", "brand manager", "brand managers"
    ])


# Detect important column names automatically because different files may use different naming conventions.
USER_KEY_COL = first_existing_col(users, ["user_key", "user_id", "wp_user_id", "id"])
FCT_USER_KEY_COL = first_existing_col(fct, ["user_key", "user_id", "wp_user_id", "id"])

ROLE_COL = first_existing_col(users, [
    "bta_user_role", "user_role", "role", "member_role", "profile_role", "bta_role"
])

PROVINCE_COL = first_existing_col(users, [
    "province", "Province", "user_province", "bta_province", "state", "region", "location_province"
])

DATE_COL = first_existing_col(fct, [
    "created_at", "attempt_start", "submitted_at", "completed_at", "response_created_at", "date"
])

if DATE_COL and DATE_COL in fct.columns:
    fct[DATE_COL] = pd.to_datetime(fct[DATE_COL], errors="coerce", utc=True)

# =============================================================
# BTA TYPOGRAPHY HELPERS
# =============================================================
# BTA brand guideline:
# - Kelly Bold = display typeface for headings, short statements, and taglines.
# - Montserrat = body copy and subheadings.
#
# Recommended Google Drive folder:
# /content/drive/MyDrive/BTAxDashboardxAcsenda/Fonts/
#   Kelly-Bold.otf
#   Montserrat-Regular.ttf
#   Montserrat-Medium.ttf
#   Montserrat-SemiBold.ttf
#   Montserrat-Bold.ttf
#
# This also searches the whole BTAxDashboardxAcsenda folder, so it can find
# your folder even if it is named "Montserrat fonts".


def find_font_file(possible_names):
    """Find a font file from Drive, app folder, src folder, or nested folders."""
    search_roots = [
        DRIVE_BASE,
        os.path.join(DRIVE_BASE, "Fonts"),
        os.path.join(DRIVE_BASE, "Montserrat fonts"),
        os.path.join(DRIVE_BASE, "Montserrat Fonts"),
        ".",
        "src",
    ]

    for root in search_roots:
        for name in possible_names:
            path = os.path.join(root, name)
            if os.path.exists(path):
                return path

    if os.path.exists(DRIVE_BASE):
        target_names = {name.lower().replace(" ", "").replace("_", "-") for name in possible_names}
        for root, dirs, files in os.walk(DRIVE_BASE):
            for file in files:
                normalized = file.lower().replace(" ", "").replace("_", "-")
                if normalized in target_names:
                    return os.path.join(root, file)

    return None


def font_face_css(font_family, font_path, font_weight="normal", font_style="normal"):
    """Convert a local font file into CSS @font-face using base64."""
    if not font_path or not os.path.exists(font_path):
        return ""

    try:
        ext = os.path.splitext(font_path)[1].lower()
        mime = "font/otf" if ext == ".otf" else "font/ttf"
        fmt = "opentype" if ext == ".otf" else "truetype"

        with open(font_path, "rb") as font_file:
            encoded_font = base64.b64encode(font_file.read()).decode()

        return f"""
        @font-face {{
            font-family: '{font_family}';
            src: url(data:{mime};base64,{encoded_font}) format('{fmt}');
            font-weight: {font_weight};
            font-style: {font_style};
            font-display: swap;
        }}
        """
    except Exception:
        return ""


KELLY_BOLD_PATH = find_font_file([
    "Kelly-Bold.otf", "Kelly Bold.otf", "KellyBold.otf",
    "Kelly-Bold.ttf", "Kelly Bold.ttf", "KellyBold.ttf",
])

MONTSERRAT_REGULAR_PATH = find_font_file([
    "Montserrat-Regular.ttf", "Montserrat Regular.ttf",
    "Montserrat-Regular.otf", "Montserrat Regular.otf",
])

MONTSERRAT_MEDIUM_PATH = find_font_file([
    "Montserrat-Medium.ttf", "Montserrat Medium.ttf",
    "Montserrat-Medium.otf", "Montserrat Medium.otf",
])

MONTSERRAT_SEMIBOLD_PATH = find_font_file([
    "Montserrat-SemiBold.ttf", "Montserrat SemiBold.ttf",
    "Montserrat-Semibold.ttf", "Montserrat Semibold.ttf",
    "Montserrat-SemiBold.otf", "Montserrat SemiBold.otf",
])

MONTSERRAT_BOLD_PATH = find_font_file([
    "Montserrat-Bold.ttf", "Montserrat Bold.ttf",
    "Montserrat-Bold.otf", "Montserrat Bold.otf",
])

BTA_FONT_CSS = ""
BTA_FONT_CSS += font_face_css("KellyBold", KELLY_BOLD_PATH, "700")
BTA_FONT_CSS += font_face_css("Montserrat", MONTSERRAT_REGULAR_PATH, "400")
BTA_FONT_CSS += font_face_css("Montserrat", MONTSERRAT_MEDIUM_PATH, "500")
BTA_FONT_CSS += font_face_css("Montserrat", MONTSERRAT_SEMIBOLD_PATH, "600")
BTA_FONT_CSS += font_face_css("Montserrat", MONTSERRAT_BOLD_PATH, "700")

# Safe fallbacks if a file is missing.
DISPLAY_FONT_FAMILY = "'KellyBold', Georgia, serif"
BODY_FONT_FAMILY = "'Montserrat', Arial, sans-serif"

# =============================================================
# STYLE
# =============================================================

# Applies consistent BTA styling to Plotly charts across the dashboard.
def style_fig(fig, height=380):
    fig.update_layout(
        template="plotly_white",
        height=height,
        paper_bgcolor="#f8f3e8",
        plot_bgcolor="#f8f3e8",
        font=dict(family="Montserrat", color="#000000", size=13),
        margin=dict(l=25, r=35, t=35, b=35),

        xaxis=dict(
            gridcolor="rgba(0,0,0,0.14)",
            zerolinecolor="rgba(0,0,0,0.14)",
            linecolor="#E26655",
            tickfont=dict(family="Montserrat", color="#000000"),
            title_font=dict(family="Montserrat", color="#000000")
        ),

        yaxis=dict(
            gridcolor="rgba(0,0,0,0.14)",
            zerolinecolor="rgba(0,0,0,0.14)",
            linecolor="#E26655",
            tickfont=dict(family="Montserrat", color="#000000"),
            title_font=dict(family="Montserrat", color="#000000")
        ),

        legend=dict(font=dict(family="Montserrat", color="#000000", size=12))
    )
    return fig


st.markdown("""
<style>
""" + BTA_FONT_CSS + """
@import url('https://fonts.googleapis.com/icon?family=Material+Icons');
@import url('https://fonts.googleapis.com/css2?family=Material+Symbols+Rounded');

:root {
    --bta-offwhite: #f8f3e8;
    --bta-red: #e26655;
    --bta-coral: #e6866f;
    --bta-orange: #e38f55;
    --bta-yellow: #f4b35e;
    --bta-sage: #b0bfa5;
    --bta-purple: #838db9;
    --bta-magenta: #9f83b8;
    --bta-black: #000000;
}

* {
    font-family: 'Montserrat', sans-serif !important;
}

[data-testid="stAppViewContainer"] {
    background-color: #f8f3e8;
}

[data-testid="stHeader"] {
    background: rgba(248,243,232,0);
}

/* KEEP STREAMLIT SIDEBAR ARROW VISIBLE */
[data-testid="collapsedControl"],
[data-testid="stSidebarCollapseButton"] {
    display: flex !important;
    visibility: visible !important;
    opacity: 1 !important;
}

/* FIX ICON FONT SO keyboard_double_arrow_right DOES NOT SHOW AS TEXT */
[data-testid="collapsedControl"] span,
[data-testid="stSidebarCollapseButton"] span {
    font-family: 'Material Symbols Rounded', 'Material Icons' !important;
}

/* SIDEBAR */
[data-testid="stSidebar"] {
    background: #F4B35E;
    border-right: none;
}

[data-testid="stSidebar"] p,
[data-testid="stSidebar"] label,
[data-testid="stSidebar"] span,
[data-testid="stSidebar"] h1,
[data-testid="stSidebar"] h2,
[data-testid="stSidebar"] h3,
[data-testid="stSidebar"] h4 {
    color: #000000 !important;
}

.block-container {
    padding-top: 35px;
    max-width: 1250px;
}

.logo-title {
    font-family: 'KellyBold', Georgia, serif !important;
    font-size: 24px;
    font-weight: normal;
    color: #000000 !important;
    line-height: 1.2;
    text-align: center;
}

.logo-subtitle {
    font-size: 11px;
    font-weight: 600;
    color: #000000 !important;
    margin-top: 3px;
    text-align: center;
}

.side-section {
    font-family: 'KellyBold', Georgia, serif !important;
    font-size: 14px;
    font-weight: normal;
    color: #000000 !important;
    letter-spacing: 1.5px;
    margin-top: 28px;
    margin-bottom: 12px;
}

.side-active {
    font-family: 'Montserrat', sans-serif !important;
    background: #f8f3e8;
    border-radius: 14px;
    padding: 12px 14px;
    font-weight: 900;
    color: #000000 !important;
    margin-bottom: 8px;
}

/* SIDEBAR ANALYTICS CLICK CARDS - CLEAN VERSION */
.side-item {
    padding: 9px 4px;
    color: #000000 !important;
    font-size: 13px;
    font-weight: 700;
}

.bta-side-details {
    background: rgba(248,243,232,0.50);
    border-radius: 14px;
    border: 1px solid rgba(255,255,255,0.55);
    margin-bottom: 10px;
    box-shadow: 0px 4px 12px rgba(0,0,0,0.06);
    overflow: hidden;
}

.bta-side-details:hover {
    background: rgba(248,243,232,0.78);
    box-shadow: 0px 6px 18px rgba(0,0,0,0.12);
}

.bta-side-details summary {
    list-style: none !important;
    cursor: pointer;
    padding: 12px 14px;
    color: #000000 !important;
    font-family: 'Montserrat', sans-serif !important;
    font-size: 13px !important;
    font-weight: 900 !important;
    display: flex;
    align-items: center;
    gap: 8px;
    line-height: 1.25;
}

.bta-side-details summary::-webkit-details-marker {
    display: none !important;
}

.bta-side-details summary::after {
    content: '+';
    margin-left: auto;
    font-size: 18px;
    font-weight: 900;
    color: #000000;
}

.bta-side-details[open] summary::after {
    content: '−';
}

.bta-side-icon {
    font-size: 17px;
    min-width: 22px;
    display: inline-block;
}

.sidebar-insight-box {
    background: #FFF8EF;
    border-radius: 14px;
    padding: 13px 14px;
    margin: 0 10px 12px 10px;
    border-left: 5px solid #F4B35E;
    box-shadow: 0px 4px 14px rgba(0,0,0,0.10);
    color: #000000 !important;
    font-family: 'Montserrat', sans-serif !important;
    font-size: 12px;
    line-height: 1.55;
    font-weight: 600;
}

.sidebar-insight-title {
    font-family: 'Montserrat', sans-serif !important;
    font-size: 13px;
    font-weight: 900;
    margin-bottom: 9px;
    color: #000000 !important;
}

.sidebar-business-impact {
    margin-top: 11px;
    padding: 9px 10px;
    border-radius: 12px;
    background: rgba(244,179,94,0.35);
    font-weight: 900;
    color: #000000 !important;
}

.side-note, .side-stat {
    background: rgba(248,243,232,0.50);
    border-radius: 14px;
    padding: 13px;
    margin-bottom: 8px;
}

.side-stat-label {
    font-size: 11px;
    color: #000000 !important;
    font-weight: 700;
}

.side-stat-value {
    font-size: 16px;
    color: #000000 !important;
    font-weight: 900;
}

/* CLICKABLE SIDEBAR DASHBOARD BUTTON */
[data-testid="stSidebar"] .stButton button {
    background: #f8f3e8 !important;
    color: #000000 !important;
    border-radius: 18px !important;
    border: 1px solid rgba(255,255,255,0.70) !important;
    min-height: 60px !important;
    font-family: 'Montserrat', sans-serif !important;
    font-size: 20px !important;
    font-weight: 900 !important;
    text-align: center !important;
    justify-content: flex-start !important;
    padding-left: 18px !important;
    box-shadow: 0px 4px 12px rgba(0,0,0,0.08) !important;
    transition: all 0.2s ease !important;
}

[data-testid="stSidebar"] .stButton button:hover {
    background: #FFFFFFCC !important;
    transform: translateY(-1px);
    box-shadow: 0px 6px 18px rgba(0,0,0,0.14) !important;
    border: 2px solid #FFFFFF !important;
}

[data-testid="stSidebar"] .stButton button:active {
    transform: scale(0.98);
}

/* TOP NAVIGATION BUTTONS */
/* Change the 6 colors here anytime */
:root {
    --nav-dashboard: #e26655;
    --nav-g1: #e6866f;
    --nav-g2: #e38f55;
    --nav-g3: #f4b35e;
    --nav-g4: #b0bfa5;
    --nav-g5: #838db9;

    /* Active/selected button colors */
    --nav-dashboard-active: #e26655;
    --nav-g1-active: #e6866f;
    --nav-g2-active: #e38f55;
    --nav-g3-active: #f4b35e;
    --nav-g4-active: #b0bfa5;
    --nav-g5-active: #838db9;
}

div[role="radiogroup"] {
    display: flex;
    gap: 14px;
    flex-wrap: wrap;
    align-items: center;
}

div[role="radiogroup"] label {
    border-radius: 999px !important;
    padding: 9px 26px !important;
    border: 2px solid rgba(255,255,255,0.70) !important;
    box-shadow: 0px 4px 12px rgba(0,0,0,0.08);
    transition: 0.2s ease;
}

/* Different color per top button */
div[role="radiogroup"] label:nth-of-type(1) { background: var(--nav-dashboard) !important; }
div[role="radiogroup"] label:nth-of-type(2) { background: var(--nav-g1) !important; }
div[role="radiogroup"] label:nth-of-type(3) { background: var(--nav-g2) !important; }
div[role="radiogroup"] label:nth-of-type(4) { background: var(--nav-g3) !important; }
div[role="radiogroup"] label:nth-of-type(5) { background: var(--nav-g4) !important; }
div[role="radiogroup"] label:nth-of-type(6) { background: var(--nav-g5) !important; }

/* Selected/active button style */
div[role="radiogroup"] label:has(input:checked) {
    transform: translateY(-1px);
    border: 3px solid #FFFFFF !important;
    box-shadow: 0px 6px 18px rgba(0,0,0,0.16);
}

div[role="radiogroup"] label:nth-of-type(1):has(input:checked) { background: var(--nav-dashboard-active) !important; }
div[role="radiogroup"] label:nth-of-type(2):has(input:checked) { background: var(--nav-g1-active) !important; }
div[role="radiogroup"] label:nth-of-type(3):has(input:checked) { background: var(--nav-g2-active) !important; }
div[role="radiogroup"] label:nth-of-type(4):has(input:checked) { background: var(--nav-g3-active) !important; }
div[role="radiogroup"] label:nth-of-type(5):has(input:checked) { background: var(--nav-g4-active) !important; }
div[role="radiogroup"] label:nth-of-type(6):has(input:checked) { background: var(--nav-g5-active) !important; }



/* Hand cursor everywhere in dropdown */

div[data-baseweb="select"],
div[data-baseweb="select"] *,
div[data-baseweb="select"] svg,
div[data-baseweb="select"] span {
    cursor: pointer !important;
}

/* Arrow icon specifically */
div[data-baseweb="select"] svg {
    cursor: pointer !important;
}

/* Dropdown menu items */
ul[role="listbox"] li,
ul[role="listbox"] div {
    cursor: pointer !important;
}

/* Hover effect */
div[data-baseweb="select"]:hover {
    border: 1px solid #E26655 !important;
    box-shadow: 0px 4px 12px rgba(226,102,85,0.18) !important;
}


/* Hide radio circle */
div[role="radiogroup"] input[type="radio"] {
    display: none !important;
}

div[role="radiogroup"] label p {
    color: #000000 !important;
    font-family: 'Montserrat', sans-serif !important;
    font-weight: 900 !important;
    font-size: 13px !important;
}

[data-testid="stRadio"] > label {
    display: none;
}

.main-title {
    font-family: 'KellyBold', serif !important;
    font-size: 42px;
    font-weight: normal;
    color: #e36654 !important;
    margin-bottom: 6px;
}

.main-subtitle {
    font-family: 'Montserrat', sans-serif !important;
    font-size: 14px;
    font-weight: 600;
    color: #000000 !important;
    margin-bottom: 28px;
}

.kpi-card, .gray-box, .info-card, .placeholder-card {
    border-radius: 18px;
    box-shadow: 0px 4px 18px rgba(0,0,0,0.08);
}

.kpi-card {
    padding: 25px 30px;
    min-height: 130px;
}

.kpi-label {
    font-family: 'Montserrat', sans-serif !important;
    color: #000000 !important;
    font-size: 14px;
    font-weight: 600;
}

.kpi-value {
    font-family: 'KellyBold', Georgia, serif !important;
    color: #000000 !important;
    font-size: 38px;
    font-weight: normal;
    margin-top: 6px;
}

.kpi-note {
    color: #000000 !important;
    font-size: 12px;
    margin-top: 8px;
    font-weight: 600;
}

.gray-box {
    padding: 24px;
    margin-top: 20px;
    background: #FFFFFF80;
}

.info-card {
    background: rgba(248,243,232,0.75);
    border-left: 8px solid #ffe9cc;
    border-radius: 18px;
    padding: 34px 36px;
    min-height: 310px;
    margin-top: 24px;
    box-shadow: 0px 4px 18px rgba(0,0,0,0.08);
}

.info-card p {
    color: #000000 !important;
    font-size: 15px;
    line-height: 1.7;
    font-weight: 500;
}

.info-card p b {
    font-weight: 900;
}

.info-card .chart-title {
    font-size: 30px;
    margin-bottom: 18px;
}

.placeholder-card {
    padding: 26px;
    margin-top: 16px;
    background: #f8f3e8;
    border-left: 7px solid #e38f55;
}

.placeholder-card p {
    color: #000000 !important;
    font-size: 15px;
    margin: 0;
}

.chart-title {
    font-family: 'KellyBold', Georgia, serif !important;
    color: #000000 !important;
    font-size: 28px;
    font-weight: normal;
    margin-top: 0;
    margin-bottom: 10px;
}

.green-text {
    font-family: 'KellyBold', Georgia, serif !important;
    color: #000000 !important;
    font-size: 38px;
    font-weight: normal;
}

.small-dark {
    color: #000000 !important;
    font-size: 13px;
    font-weight: 800;
}

.stSelectbox label {
    color: #000000 !important;
    font-weight: 700 !important;
}

[data-testid="stSidebar"] .stSelectbox label {
    color: #000000 !important;
}

[data-testid="metric-container"] {
    background: #f8f3e8;
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0px 4px 18px rgba(0,0,0,0.08);
}

[data-testid="metric-container"] * {
    color: #000000 !important;
}

.tooltip-icon {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    width: 17px;
    height: 17px;
    margin-left: 6px;
    border-radius: 50%;
    background: rgba(0,0,0,0.18);
    color: #000000 !important;
    font-size: 11px;
    font-weight: 900;
    cursor: help;
}

.executive-summary-card {
    background: rgba(248,243,232,0.75);
    border-left: 8px solid #ffe9cc;
    border-radius: 18px;
    padding: 22px 26px;
    margin-bottom: 22px;
    box-shadow: 0px 4px 18px rgba(0,0,0,0.08);
}

.executive-summary-title {
    font-family: 'KellyBold', Georgia, serif !important;
    font-size: 30px;
    font-weight: normal;
    color: #000000 !important;
    margin-bottom: 10px;
}

.executive-summary-text {
    font-size: 15px;
    line-height: 1.7;
    font-weight: 600;
    color: #000000 !important;
}

.filter-pill {
    display: inline-block;
    background: rgba(176,191,165,0.60);
    color: #000000 !important;
    padding: 7px 12px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 800;
    margin-right: 8px;
    margin-top: 10px;
}

</style>
""", unsafe_allow_html=True)


# =============================================================
# TOP NAVIGATION UPDATE
# Dashboard button is visible again in the top navigation.
# G1-G5 names use shorter labels.
# Hover over each tab to see the analytics insight.
# =============================================================

st.markdown("""
<style>

/* Sticky top navigation while scrolling */
[data-testid="stRadio"] {
    position: sticky !important;
    top: 0px !important;
    z-index: 9999 !important;
    background: #f8f3e8 !important;
    padding: 105px 0 14px 0 !important;
    margin-top: -95px !important;
    margin-bottom: 18px !important;
    border-bottom: 1px solid rgba(0,0,0,0.08) !important;
}

/* Dashboard button is visible again in the top navigation */
div[role="radiogroup"] label:nth-of-type(1) { background: #e26655 !important; }
div[role="radiogroup"] label:nth-of-type(2) { background: #e6866f !important; }
div[role="radiogroup"] label:nth-of-type(3) { background: #e38f55 !important; }
div[role="radiogroup"] label:nth-of-type(4) { background: #f4b35e !important; }
div[role="radiogroup"] label:nth-of-type(5) { background: #b0bfa5 !important; }
div[role="radiogroup"] label:nth-of-type(6) { background: #838db9 !important; }

div[role="radiogroup"] label {
    position: relative !important;
    cursor: pointer !important;
}

/* Analytics insight tooltip on top tabs */
div[role="radiogroup"] label:nth-of-type(2)::after,
div[role="radiogroup"] label:nth-of-type(3)::after,
div[role="radiogroup"] label:nth-of-type(4)::after,
div[role="radiogroup"] label:nth-of-type(5)::after,
div[role="radiogroup"] label:nth-of-type(6)::after {
    visibility: hidden;
    opacity: 0;
    position: absolute;
    left: 50%;
    top: auto !important;
    bottom: calc(100% + 12px) !important;
    transform: translateX(-50%) !important;
    width: 300px;
    background: #2c2c2c;
    color: #ffffff;
    border-radius: 12px;
    padding: 12px 14px;
    font-size: 12px;
    font-weight: 600;
    line-height: 1.55;
    text-align: left;
    white-space: pre-line;
    z-index: 10000;
    box-shadow: 0px 8px 22px rgba(0,0,0,0.22);
    transition: opacity 0.2s ease;
    pointer-events: none;
}

div[role="radiogroup"] label:nth-of-type(2):hover::after,
div[role="radiogroup"] label:nth-of-type(3):hover::after,
div[role="radiogroup"] label:nth-of-type(4):hover::after,
div[role="radiogroup"] label:nth-of-type(5):hover::after,
div[role="radiogroup"] label:nth-of-type(6):hover::after {
    visibility: visible;
    opacity: 1;
}


/* Small arrow under tooltip */
div[role="radiogroup"] label:nth-of-type(2)::before,
div[role="radiogroup"] label:nth-of-type(3)::before,
div[role="radiogroup"] label:nth-of-type(4)::before,
div[role="radiogroup"] label:nth-of-type(5)::before,
div[role="radiogroup"] label:nth-of-type(6)::before {
    content: "";
    visibility: hidden;
    opacity: 0;
    position: absolute;
    left: 50%;
    bottom: calc(100% + 4px);
    transform: translateX(-50%);
    border-left: 8px solid transparent;
    border-right: 8px solid transparent;
    border-top: 8px solid #2c2c2c;
    z-index: 10001;
    transition: opacity 0.2s ease;
    pointer-events: none;
}

div[role="radiogroup"] label:nth-of-type(2):hover::before,
div[role="radiogroup"] label:nth-of-type(3):hover::before,
div[role="radiogroup"] label:nth-of-type(4):hover::before,
div[role="radiogroup"] label:nth-of-type(5):hover::before,
div[role="radiogroup"] label:nth-of-type(6):hover::before {
    visibility: visible;
    opacity: 1;
}

div[role="radiogroup"] label:nth-of-type(2)::after {
    content: "Retention\A Tracks how many members return after signup.\A Shows loyalty, drop-off, and retention patterns.";
}

div[role="radiogroup"] label:nth-of-type(3)::after {
    content: "Cadence\A Measures participation rhythm.\A Shows average days between surveys and inactivity risk.";
}

div[role="radiogroup"] label:nth-of-type(4)::after {
    content: "Points\A Monitors reward activity and point patterns.\A Helps detect unusual points behavior and member value.";
}

div[role="radiogroup"] label:nth-of-type(5)::after {
    content: "Variance Analysis\A Measures response consistency.\A Helps find stable answers, high variance, and data reliability issues.";
}

div[role="radiogroup"] label:nth-of-type(6)::after {
    content: "Lazy-Text + Reward Farming \A Flags low-effort text responses.\A Helps identify weak answers and possible reward farming behavior.";
}

</style>
""", unsafe_allow_html=True)

# =============================================================
# TOP RIGHT BTA WATERMARK
# =============================================================

# Searches for an optional BTA dashboard watermark image.
watermark_candidates = [
    os.path.join(DRIVE_BASE, "BTA Data Analytics Dashboard.png"),
    "BTA Data Analytics Dashboard.png",
    os.path.join("src", "BTA Data Analytics Dashboard.png"),
]

watermark_path = next(
    (p for p in watermark_candidates if os.path.exists(p)),
    None
)

if watermark_path:
    with open(watermark_path, "rb") as img_file:
        watermark_base64 = base64.b64encode(img_file.read()).decode()

    st.markdown(
        f"""
        <style>

        .bta-watermark {{
            position: fixed;
            top: 70px;
            right: 35px;
            width: 340px;
            opacity: 0.06;
            z-index: 0;
            pointer-events: none;
        }}

        .bta-watermark img {{
            width: 100%;
            height: auto;
        }}

        </style>

        <div class="bta-watermark">
            <img src="data:image/png;base64,{watermark_base64}">
        </div>
        """,
        unsafe_allow_html=True
    )

# =============================================================
# MISSING MAIN FILE MESSAGE
# =============================================================

if missing_required_files:
    with st.expander("⚠️ Missing Main CSV Files", expanded=True):
        st.error("The dashboard cannot find the required Main CSV Files.")
        st.write("Dashboard is checking this folder:")
        st.code(MAIN_CSV_FOLDER)
        st.write("Missing files:")
        for item in missing_required_files:
            st.write(f"- {item}")

# =============================================================
# SIDEBAR
# =============================================================

# Searches for the BTA logo image to display in the sidebar.
logo_candidates = [
    os.path.join(DRIVE_BASE, "Logo.png"),
    os.path.join(DRIVE_BASE, "logo.png"),
    "Logo.png",
    "logo.png",
    os.path.join("src", "Logo.png"),
]

logo_path = next((path for path in logo_candidates if os.path.exists(path)), None)

if logo_path:
    left_logo, center_logo, right_logo = st.sidebar.columns([1, 3, 1])
    with center_logo:
        st.image(logo_path, width=180)
else:
    st.sidebar.markdown("<h1 style='text-align:center;'>🌿</h1>", unsafe_allow_html=True)

st.sidebar.markdown("""
<div class="logo-title">BTA Analytics Dashboard</div>
<div class="logo-subtitle">Survey Intelligence Platform</div>
""", unsafe_allow_html=True)

# -----------------------------
# GLOBAL FILTERS REQUESTED BY BTA
# -----------------------------
# Builds Role filter options from the users dataset.
roles = ["All"]
if ROLE_COL:
    roles += sorted(clean_filter_value(users[ROLE_COL]).unique().tolist())

# Builds Province filter options from the users dataset.
provinces = ["All"]
if PROVINCE_COL:
    provinces += sorted(clean_filter_value(users[PROVINCE_COL]).unique().tolist())

st.sidebar.markdown('<div class="side-section">FILTERS</div>', unsafe_allow_html=True)
selected_role = st.sidebar.selectbox("Role", roles)
selected_province = st.sidebar.selectbox("Province", provinces)
exclude_unknown_users = st.sidebar.checkbox("Exclude Unknown / Internal Users", value=True)
period = st.sidebar.selectbox("Time Period", ["All Time", "This Year", "This Month", "Custom Range"])

custom_start = None
custom_end = None
if period == "Custom Range":
    custom_dates = st.sidebar.date_input("Select date range", value=[])
    if isinstance(custom_dates, tuple) and len(custom_dates) == 2:
        custom_start, custom_end = custom_dates

# =============================================================
# MAIN METRICS
# =============================================================

# Creates filtered copies of the main datasets so the original data remains unchanged.
filtered_users = users.copy()
filtered_fct = fct.copy()

# 1) Global Unknown-user filter
# This removes Unknown / internal roles from the users table and all survey activity.
if exclude_unknown_users and ROLE_COL and USER_KEY_COL:
    filtered_users = filtered_users[~is_unknown_value(filtered_users[ROLE_COL])].copy()

# 2) Role filter
if selected_role != "All" and ROLE_COL:
    filtered_users = filtered_users[clean_filter_value(filtered_users[ROLE_COL]) == selected_role].copy()

# 3) Province filter
if selected_province != "All" and PROVINCE_COL:
    filtered_users = filtered_users[clean_filter_value(filtered_users[PROVINCE_COL]) == selected_province].copy()

# Apply user-based filters to fact table
if USER_KEY_COL and FCT_USER_KEY_COL:
    valid_user_keys = filtered_users[USER_KEY_COL].dropna().unique()
    filtered_fct = filtered_fct[filtered_fct[FCT_USER_KEY_COL].isin(valid_user_keys)].copy()

# 4) Time filter for survey activity
if DATE_COL and DATE_COL in filtered_fct.columns and not filtered_fct.empty:
    max_date = filtered_fct[DATE_COL].max()

    if period == "This Year":
        filtered_fct = filtered_fct[filtered_fct[DATE_COL].dt.year == max_date.year]

    elif period == "This Month":
        filtered_fct = filtered_fct[
            (filtered_fct[DATE_COL].dt.year == max_date.year) &
            (filtered_fct[DATE_COL].dt.month == max_date.month)
        ]

    elif period == "Custom Range" and custom_start and custom_end:
        start_ts = pd.to_datetime(custom_start).tz_localize("UTC")
        end_ts = pd.to_datetime(custom_end).tz_localize("UTC") + pd.Timedelta(days=1)
        filtered_fct = filtered_fct[
            (filtered_fct[DATE_COL] >= start_ts) &
            (filtered_fct[DATE_COL] < end_ts)
        ]


# =============================================================
# GLOBAL FILTER FUNCTIONS FOR ALL TABS
# =============================================================

def apply_global_user_filter(df, user_col="user_key"):
    """
    Applies the main sidebar Role, Province, and Unknown/Internal user filter
    to any group dataframe that has a user_key column.
    """
    if df is None or df.empty:
        return df

    temp = df.copy()

    if USER_KEY_COL and user_col in temp.columns and USER_KEY_COL in filtered_users.columns:
        valid_keys = filtered_users[USER_KEY_COL].dropna().astype(str).unique()
        temp[user_col] = temp[user_col].astype(str)
        temp = temp[temp[user_col].isin(valid_keys)]

    return temp


def apply_global_date_filter(df, date_col):
    """
    Applies the main sidebar Time Period filter to any group dataframe
    that has a usable date column.
    """
    if df is None or df.empty or date_col not in df.columns:
        return df

    temp = df.copy()
    temp[date_col] = pd.to_datetime(temp[date_col], errors="coerce", utc=True)

    if temp[date_col].dropna().empty:
        return temp

    max_date = temp[date_col].max()

    if period == "This Year":
        temp = temp[temp[date_col].dt.year == max_date.year]

    elif period == "This Month":
        temp = temp[
            (temp[date_col].dt.year == max_date.year) &
            (temp[date_col].dt.month == max_date.month)
        ]

    elif period == "Custom Range" and custom_start and custom_end:
        start_ts = pd.to_datetime(custom_start).tz_localize("UTC")
        end_ts = pd.to_datetime(custom_end).tz_localize("UTC") + pd.Timedelta(days=1)
        temp = temp[(temp[date_col] >= start_ts) & (temp[date_col] < end_ts)]

    return temp


# Calculates dashboard KPI values after global filters are applied.
total_profiles = filtered_users[USER_KEY_COL].nunique() if USER_KEY_COL and USER_KEY_COL in filtered_users.columns else len(filtered_users)
total_responders = filtered_fct[FCT_USER_KEY_COL].nunique() if FCT_USER_KEY_COL and FCT_USER_KEY_COL in filtered_fct.columns else 0
total_responses = len(filtered_fct)

if "survey_key" in filtered_fct.columns:
    total_surveys = filtered_fct["survey_key"].nunique()
elif "survey_key" in surveys.columns:
    total_surveys = surveys["survey_key"].nunique()
else:
    total_surveys = len(surveys)

total_questions = questions["question_key"].nunique() if "question_key" in questions.columns else len(questions)
coverage_rate = (total_responders / total_profiles * 100) if total_profiles else 0

activity_start = filtered_fct[DATE_COL].min() if DATE_COL and DATE_COL in filtered_fct.columns and not filtered_fct.empty else pd.NaT
activity_end = filtered_fct[DATE_COL].max() if DATE_COL and DATE_COL in filtered_fct.columns and not filtered_fct.empty else pd.NaT

attempt_cols = [col for col in [FCT_USER_KEY_COL, "survey_id", "survey_key"] if col and col in filtered_fct.columns]
attempts = filtered_fct.drop_duplicates(attempt_cols).copy() if len(attempt_cols) >= 2 else filtered_fct.copy()
total_attempts = len(attempts)

if ROLE_COL and USER_KEY_COL and USER_KEY_COL in filtered_users.columns:
    role_counts = clean_filter_value(filtered_users[ROLE_COL]).value_counts().reset_index()
    role_counts.columns = ["Role", "Users"]
else:
    role_counts = pd.DataFrame({"Role": ["No role column found"], "Users": [0]})

gap_df = pd.DataFrame()
if DATE_COL and FCT_USER_KEY_COL and DATE_COL in attempts.columns and FCT_USER_KEY_COL in attempts.columns:
    gap_df = attempts.copy()
    gap_df[DATE_COL] = pd.to_datetime(gap_df[DATE_COL], errors="coerce", utc=True)
    gap_df = gap_df.dropna(subset=[DATE_COL])
    gap_df = gap_df.sort_values([FCT_USER_KEY_COL, DATE_COL])
    gap_df["previous_attempt"] = gap_df.groupby(FCT_USER_KEY_COL)[DATE_COL].shift(1)
    gap_df["gap_days"] = (gap_df[DATE_COL] - gap_df["previous_attempt"]).dt.days
    gap_df = gap_df.dropna(subset=["gap_days"])

avg_gap = gap_df["gap_days"].mean() if not gap_df.empty else 0
median_gap = gap_df["gap_days"].median() if not gap_df.empty else 0
long_gaps = (gap_df["gap_days"] >= 60).sum() if not gap_df.empty else 0

# Executive summary helper values
top_role = "N/A"
if ROLE_COL and not filtered_users.empty:
    role_summary = clean_filter_value(filtered_users[ROLE_COL]).value_counts()
    if not role_summary.empty:
        top_role = role_summary.index[0]

top_province = "N/A"
if PROVINCE_COL and not filtered_users.empty:
    province_summary = clean_filter_value(filtered_users[PROVINCE_COL]).value_counts()
    if not province_summary.empty:
        top_province = province_summary.index[0]

unknown_profiles_count = 0
if ROLE_COL and not users.empty:
    unknown_profiles_count = int(is_unknown_value(users[ROLE_COL]).sum())

valid_profiles_count = len(users) - unknown_profiles_count if not users.empty else 0

filter_status = "Unknown/internal users excluded" if exclude_unknown_users else "Unknown/internal users included"

if pd.notna(activity_start) and pd.notna(activity_end):
    activity_window_label = f"{activity_start.date()} → {activity_end.date()}"
else:
    activity_window_label = "N/A"

st.sidebar.markdown(f"""
<div class="side-section">DATASET SNAPSHOT</div>

<div class="side-stat">
    <div class="side-stat-label">👥 Profiles</div>
    <div class="side-stat-value">{total_profiles:,.0f}</div>
</div>

<div class="side-stat">
    <div class="side-stat-label">📊 Responders</div>
    <div class="side-stat-value">{total_responders:,.0f}</div>
</div>

<div class="side-stat">
    <div class="side-stat-label">💬 Responses</div>
    <div class="side-stat-value">{total_responses:,.0f}</div>
</div>

<div class="side-stat">
    <div class="side-stat-label">📋 Surveys</div>
    <div class="side-stat-value">{total_surveys:,.0f}</div>
</div>

<div class="side-stat">
    <div class="side-stat-label">📝 Questions</div>
    <div class="side-stat-value">{total_questions:,.0f}</div>
</div>

<div class="side-section">COVERAGE</div>

<div class="side-note">
<b>{coverage_rate:.1f}% Coverage</b><br>
{total_responders:,.0f} of {total_profiles:,.0f} users responded.<br><br>
Activity Window:<br>
{activity_start.date() if pd.notna(activity_start) else 'N/A'} → {activity_end.date() if pd.notna(activity_end) else 'N/A'}<br><br>
Province: {selected_province}<br>
Unknown Filter: {'On' if exclude_unknown_users else 'Off'}
</div>
""", unsafe_allow_html=True)

# =============================================================
# PAGE NAVIGATION
# =============================================================

# Defines the dashboard pages/tabs.
pages = ["Dashboard", "G1", "G2", "G3", "G4", "G5"]

# Sets user-friendly names for each group tab.
page_labels = {
    "Dashboard": "Dashboard",
    "G1": "Retention",
    "G2": "Cadence",
    "G3": "Points",
    "G4": "Variance",
    "G5": "Text",
}

if "selected_page" not in st.session_state:
    st.session_state["selected_page"] = "Dashboard"

if st.session_state["selected_page"] not in pages:
    st.session_state["selected_page"] = "Dashboard"

page = st.radio(
    "Navigation",
    pages,
    index=pages.index(st.session_state["selected_page"]),
    horizontal=True,
    label_visibility="collapsed",
    format_func=lambda x: page_labels.get(x, x),
    key="top_navigation_radio"
)

st.session_state["selected_page"] = page

# Creates a standard placeholder card for group sections that are not fully loaded yet.
def group_placeholder(title, subtitle, folder_name):
    st.markdown(f'<div class="main-title">{title}</div>', unsafe_allow_html=True)
    st.markdown(f'<div class="main-subtitle">{subtitle}</div>', unsafe_allow_html=True)
    st.markdown(
        f"""
        <div class="placeholder-card">
            <p><b>Note:</b> Put {folder_name} files in Google Drive folder <b>BTAxDashboardxAcsenda/{folder_name}</b>.</p>
        </div>
        """,
        unsafe_allow_html=True
    )

# =============================================================
# DASHBOARD HOME
# =============================================================

# Renders the main dashboard home page.
if page == "Dashboard":

    st.markdown('<div class="main-title">BTA Data Analytics Dashboard</div>', unsafe_allow_html=True)


    st.markdown(f"""
    <div class="executive-summary-card">
        <div class="executive-summary-title">Executive Summary</div>
        <div class="executive-summary-text">
            Based on the selected filters, <b>{total_responders:,.0f}</b> users completed survey activity from
            <b>{activity_window_label}</b>. The current coverage rate is <b>{coverage_rate:.1f}%</b>,
            with <b>{total_responses:,.0f}</b> total responses and <b>{total_attempts:,.0f}</b> survey attempts.
            The most common active role is <b>{top_role}</b>, and the top province is <b>{top_province}</b>.
            Data quality filter status: <b>{filter_status}</b>.
        </div>
        <div>
            <span class="filter-pill">Role: {selected_role}</span>
            <span class="filter-pill">Province: {selected_province}</span>
            <span class="filter-pill">Time: {period}</span>
            <span class="filter-pill">Unknown Filter: {"On" if exclude_unknown_users else "Off"}</span>
        </div>
    </div>
    """, unsafe_allow_html=True)

    k1, k2, k3 = st.columns(3)

    with k1:
        st.markdown(f"""
        <div class="kpi-card" style="background:#FFFFFF80;">
            <div class="kpi-label">Total Profiles <span class="tooltip-icon" title="Number of user profiles after applying role, province, time, and unknown-user filters.">?</span></div>
            <div class="kpi-value">{total_profiles:,.0f}</div>
            <div class="kpi-note">Member profiles in dim_users</div>
        </div>
        """, unsafe_allow_html=True)

    with k2:
        st.markdown(f"""
        <div class="kpi-card" style="background:#FFFFFF80;">
            <div class="kpi-label">Survey Responders <span class="tooltip-icon" title="Number of unique users who answered at least one survey after filters are applied.">?</span></div>
            <div class="kpi-value">{total_responders:,.0f}</div>
            <div class="kpi-note">Users with survey activity</div>
        </div>
        """, unsafe_allow_html=True)

    with k3:
        st.markdown(f"""
        <div class="kpi-card" style="background:#FFFFFF80;">
            <div class="kpi-label">Coverage Rate <span class="tooltip-icon" title="Survey responders divided by total filtered profiles. This shows how much of the user base is engaging with surveys.">?</span></div>
            <div class="kpi-value">{coverage_rate:.1f}%</div>
            <div class="kpi-note">Responders ÷ profiles</div>
        </div>
        """, unsafe_allow_html=True)

    left, right = st.columns([2.1, 1])

    with left:
        st.markdown('<div class="gray-box" style="background:#FFFFFF80;"><div class="chart-title">Member Role Distribution</div>', unsafe_allow_html=True)

        fig_donut = px.pie(
            role_counts,
            names="Role",
            values="Users",
            hole=0.55
        )
        fig_donut.update_traces(
            textinfo="percent+label",
            textfont=dict(family="Montserrat", color="#000000", size=13),
            marker=dict(colors=["#e26655", "#e6866f", "#e38f55", "#f4b35e", "#FFFFFF80", "#838db9", "#9f83b8"])
        )
        fig_donut = style_fig(fig_donut, 330)
        st.plotly_chart(fig_donut, use_container_width=True, key="dashboard_role_donut")

        st.markdown('</div>', unsafe_allow_html=True)

    with right:
        st.markdown(f"""
        <div class="gray-box" style="background:#FFFFFF80;">
            <div class="small-dark">Survey Responses</div>
            <div class="green-text">{total_responses:,.0f}</div>
        """, unsafe_allow_html=True)

        if DATE_COL and DATE_COL in filtered_fct.columns and not filtered_fct.empty:
            monthly = filtered_fct.copy()
            monthly["month"] = monthly[DATE_COL].dt.strftime("%b")
            monthly_counts = monthly.groupby("month").size().reset_index(name="Responses")

            month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
            monthly_counts["month"] = pd.Categorical(monthly_counts["month"], categories=month_order, ordered=True)
            monthly_counts = monthly_counts.sort_values("month")

            fig_line = px.line(monthly_counts, x="month", y="Responses", markers=True)
            fig_line.update_traces(line_color="#F4B35E", marker_color="#E26655")
            fig_line = style_fig(fig_line, 250)
            st.plotly_chart(fig_line, use_container_width=True, key="dashboard_monthly_line")
        else:
            st.info("Monthly survey trend will appear when created_at data is available.")

        st.markdown('</div>', unsafe_allow_html=True)

    s1, s2 = st.columns(2)

    with s1:
        st.markdown(f"""
        <div class="info-card">
            <div class="chart-title">Project Overview</div>
            <p><b>Selected Role:</b> {selected_role}</p>
            <p><b>Province:</b> {selected_province}</p>
            <p><b>Time Period:</b> {period}</p>
            <p><b>Unknown Users Removed:</b> {"Yes" if exclude_unknown_users else "No"}</p>
            <p><b>Unknown/Internal Profiles Detected:</b> {unknown_profiles_count:,.0f}</p>
            <p><b>Valid Profiles Before Other Filters:</b> {valid_profiles_count:,.0f}</p>
            <p><b>Activity Start:</b> {activity_start.date() if pd.notna(activity_start) else 'N/A'}</p>
            <p><b>Activity End:</b> {activity_end.date() if pd.notna(activity_end) else 'N/A'}</p>
            <p><b>Total Surveys:</b> {total_surveys:,.0f}</p>
            <p><b>Total Questions:</b> {total_questions:,.0f}</p>
            <p><b>Survey Attempts:</b> {total_attempts:,.0f}</p>
        </div>
        """, unsafe_allow_html=True)

    with s2:
        st.markdown(f"""
        <div class="info-card">
            <div class="chart-title">Key Findings</div>
            <p><b>Coverage:</b> {total_responders:,.0f} of {total_profiles:,.0f} users answered at least one survey.</p>
            <p><b>Average Gap:</b> {avg_gap:.1f} days</p>
            <p><b>Median Gap:</b> {median_gap:.1f} days</p>
            <p><b>60+ Day Gaps:</b> {long_gaps:,.0f}</p>
            <p><b>Dashboard Use:</b> Use G1–G5 tabs to organize each group analysis module.</p>
        </div>
        """, unsafe_allow_html=True)




# =============================================================
# GROUP PLACEHOLDER PAGES
# =============================================================

# =============================================================
# G1 · COHORT RETENTION
# TEAM MEMBERS: YUSRI, SANDRA, JAY
#
# ⬇️ G1 PASTE YOUR FINAL CHART CODE BELOW THIS SECTION ⬇️
# =============================================================
elif page == "G1":

    # =============================================================
    # G1 · COHORT RETENTION  |  TEAM: YUSRI, SANDRA, JAY
    # =============================================================

    import numpy as np
    import plotly.graph_objects as go

    # ── BTA Brand Book v3 Palette ─────────────────────────────────────────────
    BTA_BG         = "#f8f3e8"
    BTA_DARK       = "#2c2c2c"
    BTA_ORANGE     = "#e38f54"
    BTA_CORAL      = "#e36654"
    BTA_YELLOW     = "#f5b25e"
    BTA_SAGE       = "#b0bfa6"
    BTA_PERIWINKLE = "#828cba"
    BTA_LAVENDER   = "#9e82b8"
    BTA_ORANGE_DK  = "#c8501b"
    BTA_CORAL_DK   = "#c8291b"
    BTA_SAGE_DK    = "#788e6c"

    # ── CSS ───────────────────────────────────────────────────────────────────
    st.markdown(f"""
    <style>
    .g1-tooltip-wrap {{
        position: relative;
        display: inline-block;
    }}
    .g1-tooltip-wrap .g1-tooltip-text {{
        visibility: hidden;
        opacity: 0;
        width: 270px;
        background: {BTA_DARK};
        color: #fff;
        font-size: 12px;
        line-height: 1.55;
        border-radius: 10px;
        padding: 11px 14px;
        position: absolute;
        z-index: 9999;
        bottom: 130%;
        left: 50%;
        transform: translateX(-50%);
        transition: opacity 0.2s ease;
        font-family: Montserrat, sans-serif;
        font-weight: 500;
        box-shadow: 0 4px 16px rgba(0,0,0,0.22);
    }}
    .g1-tooltip-wrap:hover .g1-tooltip-text {{
        visibility: visible;
        opacity: 1;
    }}
    .g1-info-icon {{
        display: inline-flex;
        align-items: center;
        justify-content: center;
        width: 17px;
        height: 17px;
        border-radius: 50%;
        background: rgba(44,44,44,0.13);
        color: {BTA_DARK};
        font-size: 10px;
        font-weight: 800;
        cursor: help;
        margin-left: 5px;
        font-family: Montserrat, sans-serif;
        vertical-align: middle;
    }}
    .g1-kpi-card {{
        background: {BTA_BG};
        border-radius: 16px;
        padding: 20px 16px 16px;
        box-shadow: 0 2px 14px rgba(44,44,44,0.09);
        min-height: 148px;
        position: relative;
    }}
    .g1-kpi-label {{
        font-size: 10px;
        font-weight: 700;
        color: {BTA_DARK};
        text-transform: uppercase;
        letter-spacing: 0.8px;
        font-family: Montserrat, sans-serif;
        display: flex;
        align-items: center;
        gap: 4px;
    }}
    .g1-kpi-value {{
        font-size: 40px;
        font-weight: 900;
        line-height: 1.1;
        margin: 10px 0 8px;
        font-family: Montserrat, sans-serif;
    }}
    .g1-kpi-status {{
        font-size: 10px;
        font-weight: 700;
        font-family: Montserrat, sans-serif;
    }}
    .g1-chart-insight {{
        background: rgba(44,44,44,0.04);
        border-radius: 10px;
        padding: 13px 16px;
        margin-top: 6px;
        margin-bottom: 6px;
        font-size: 13px;
        color: {BTA_DARK};
        font-family: Montserrat, sans-serif;
        line-height: 1.6;
        display: flex;
        align-items: flex-start;
        gap: 10px;
    }}
    .g1-insight-icon {{
        display: inline-flex;
        align-items: center;
        justify-content: center;
        min-width: 22px;
        height: 22px;
        border-radius: 50%;
        background: {BTA_ORANGE};
        color: #fff;
        font-size: 12px;
        font-weight: 800;
        margin-top: 1px;
        font-family: Montserrat, sans-serif;
    }}
    .g1-section-divider {{
        border: none;
        border-top: 1px solid rgba(44,44,44,0.1);
        margin: 28px 0 20px;
    }}
    .g1-province-table {{
        width: 100%;
        border-collapse: collapse;
        font-family: Montserrat, sans-serif;
        font-size: 13px;
    }}
    .g1-province-table th {{
        background: {BTA_ORANGE};
        color: #fff;
        padding: 10px 14px;
        text-align: left;
        font-weight: 700;
        font-size: 11px;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }}
    .g1-province-table td {{
        padding: 9px 14px;
        border-bottom: 1px solid rgba(44,44,44,0.08);
        color: {BTA_DARK};
    }}
    .g1-province-table tr:nth-child(even) td {{
        background: rgba(44,44,44,0.03);
    }}
    .g1-atrisk-card {{
        background: #FFF4E5;
        border-left: 5px solid {BTA_CORAL};
        border-radius: 10px;
        padding: 16px 20px;
        margin-bottom: 10px;
        font-family: Montserrat, sans-serif;
        font-size: 13px;
        color: {BTA_DARK};
        line-height: 1.6;
    }}
    </style>
    """, unsafe_allow_html=True)

    # ── TITLE & DESCRIPTION ───────────────────────────────────────────────────
    st.markdown('<div class="main-title">Retention</div>', unsafe_allow_html=True)
    st.markdown(
        f'<div class="main-subtitle" style="color:{BTA_DARK};opacity:0.65;">'
        'How many members return to complete surveys after signing up? '
        'Covers verified Budtenders, Consumers, and Retail Managers only. '
        'Internal users excluded.</div>',
        unsafe_allow_html=True
    )

    # ── BUILD DATA ────────────────────────────────────────────────────────────
    @st.cache_data(show_spinner=False)
    def g1_build(_fct, _users, _wp):

        fct_g1   = _fct.copy()
        users_g1 = _users.copy()
        wp_g1    = _wp.copy()

        if fct_g1.empty or users_g1.empty or wp_g1.empty:
            try:
                import glob as _glob

                _roots = [
                    "/content/drive/MyDrive/BTA_Project_Data",
                    "/content/drive/MyDrive/BTAxDashboardxAcsenda/Main CSV Files",
                    "/content/drive/MyDrive/BTAxDashboardxAcsenda",
                    "/content/drive/MyDrive",
                ]

                def _find(pattern):
                    for r in _roots:
                        m = _glob.glob(os.path.join(r, pattern))
                        if m:
                            return pd.read_csv(m[0])
                    return pd.DataFrame()

                if fct_g1.empty:
                    fct_g1 = _find("fct_survey_respon*.csv")

                if users_g1.empty:
                    users_g1 = _find("dim_users_*.csv")

                if wp_g1.empty:
                    wp_g1 = _find("wp_users_*.csv")

            except Exception:
                pass

        created_col = next(
            (c for c in fct_g1.columns if "created" in c.lower()),
            None
        )

        fct_g1["created_at_clean"] = (
            pd.to_datetime(
                fct_g1[created_col],
                utc=True,
                errors="coerce"
            )
            if created_col else pd.NaT
        )

        registered_col = next(
            (c for c in wp_g1.columns if "registered" in c.lower()),
            None
        )

        wp_g1["user_registered_clean"] = (
            pd.to_datetime(
                wp_g1[registered_col],
                utc=True,
                errors="coerce"
            )
            if registered_col else pd.NaT
        )

        wp_id_col = (
            next((c for c in wp_g1.columns if c.lower()=="id"), None)
            or
            next((c for c in wp_g1.columns if "id" in c.lower()), None)
        )

        if wp_id_col and wp_id_col != "id":
            wp_g1["id"] = wp_g1[wp_id_col]

        if "bta_user_role" not in users_g1.columns:

            rc = next(
                (c for c in users_g1.columns if "role" in c.lower()),
                None
            )

            users_g1["bta_user_role"] = (
                users_g1[rc]
                if rc else
                "Unknown Role"
            )

        users_g1["bta_user_role"] = (
            users_g1["bta_user_role"]
            .fillna("Unknown Role")
        )

        if "wp_user_id" not in users_g1.columns:

            wf = next(
                (
                    c for c in users_g1.columns
                    if "wp" in c.lower()
                    and "id" in c.lower()
                ),
                None
            )

            users_g1["wp_user_id"] = (
                users_g1[wf]
                if wf else None
            )

        if "fact_key" not in fct_g1.columns:
            fct_g1["fact_key"] = range(len(fct_g1))

        if "user_key" not in fct_g1.columns:

            uk = next(
                (
                    c for c in fct_g1.columns
                    if "user" in c.lower()
                    and "key" in c.lower()
                ),
                None
            )

            fct_g1["user_key"] = (
                fct_g1[uk]
                if uk else None
            )

        users_g1 = users_g1[
            users_g1["bta_user_role"] != "Unknown Role"
        ].copy()

        if "province" not in users_g1.columns:

            pc = next(
                (
                    c for c in users_g1.columns
                    if "province" in c.lower()
                    or "region" in c.lower()
                    or "state" in c.lower()
                ),
                None
            )

            users_g1["province"] = (
                users_g1[pc]
                if pc else "Unknown"
            )

        users_g1["province"] = (
            users_g1["province"]
            .fillna("Unknown")
        )

        wp_g1["cohort_month"] = (
            wp_g1["user_registered_clean"]
            .dt.to_period("M")
            .astype(str)
        )

        fct_g1["activity_month"] = (
            fct_g1["created_at_clean"]
            .dt.to_period("M")
            .astype(str)
        )

        # -----------------------------
        # Registered cohort baseline
        # -----------------------------
        registered_baselines = (
            wp_g1
            .merge(
                users_g1[
                    [
                        "wp_user_id",
                        "bta_user_role",
                        "province"
                    ]
                ],
                left_on="id",
                right_on="wp_user_id",
                how="inner"
            )
            .groupby(
                [
                    "cohort_month",
                    "bta_user_role",
                    "province"
                ]
            )["id"]
            .nunique()
            .reset_index(
                name="total_registered_pool"
            )
        )

        # -----------------------------
        # Monthly responses
        # -----------------------------
        active_sessions = (
            fct_g1
            .groupby(
                [
                    "user_key",
                    "activity_month"
                ]
            )["fact_key"]
            .count()
            .reset_index(
                name="responses_that_month"
            )
        )

        # -----------------------------
        # Cohort activity spine
        # -----------------------------
        g1_spine = (
            active_sessions
            .merge(
                users_g1[
                    [
                        "user_key",
                        "wp_user_id",
                        "bta_user_role",
                        "province"
                    ]
                ],
                on="user_key",
                how="inner"
            )
            .merge(
                wp_g1[
                    [
                        "id",
                        "cohort_month"
                    ]
                ],
                left_on="wp_user_id",
                right_on="id",
                how="inner"
            )
        )

        g1_spine["activity_dt"] = pd.to_datetime(
            g1_spine["activity_month"] + "-01",
            utc=True
        )

        g1_spine["signup_dt"] = pd.to_datetime(
            g1_spine["cohort_month"] + "-01",
            utc=True
        )

        g1_spine["month_offset"] = (
            (g1_spine["activity_dt"].dt.year -
            g1_spine["signup_dt"].dt.year) * 12
            +
            (g1_spine["activity_dt"].dt.month -
            g1_spine["signup_dt"].dt.month)
        )

        g1_spine = g1_spine[
            g1_spine["month_offset"] >= 0
        ].copy()

        cohort_activity = (
            g1_spine
            .groupby(
                [
                    "cohort_month",
                    "month_offset",
                    "bta_user_role",
                    "province"
                ]
            )["user_key"]
            .nunique()
            .reset_index(
                name="active_responders"
            )
        )

        # -----------------------------
        # Final retention mart
        # -----------------------------
        marts = cohort_activity.merge(
            registered_baselines,
            on=[
                "cohort_month",
                "bta_user_role",
                "province"
            ],
            how="left"
        )

        marts["retention_rate_pct"] = (
            marts["active_responders"]
            /
            marts["total_registered_pool"]
            *
            100
        ).round(2)

        marts.loc[
            marts["total_registered_pool"] < 5,
            "retention_rate_pct"
        ] = None

        return marts, g1_spine, users_g1, wp_g1


    g1_marts, g1_spine, users_g1_clean, wp_g1 = g1_build(
        filtered_fct,
        filtered_users,
        wp
    )

    g1_filtered = g1_marts.copy()
    spine_filt = g1_spine.copy()

    # ── KPI COMPUTATIONS ─────────────────────────────────────────────────────
    month0      = g1_filtered[g1_filtered["month_offset"] == 0]
    month0_rate = round(month0["active_responders"].sum() / month0["total_registered_pool"].sum() * 100, 1) \
                  if month0["total_registered_pool"].sum() > 0 else 0

    month3      = g1_filtered[g1_filtered["month_offset"] == 3]
    month3_rate = round(month3["active_responders"].sum() / month3["total_registered_pool"].sum() * 100, 1) \
                  if month3["total_registered_pool"].sum() > 0 else 0

    last_active       = spine_filt.groupby("user_key")["month_offset"].max()
    avg_dropoff_month = round(last_active.mean(), 1) if not last_active.empty else 0

    cohort_month0 = (
        g1_filtered[g1_filtered["month_offset"] == 0]
        .groupby("cohort_month")
        .apply(lambda x: round(x["active_responders"].sum() / x["total_registered_pool"].sum() * 100, 1)
               if x["total_registered_pool"].sum() > 0 else 0)
        .reset_index(name="month0_rate")
        .sort_values("cohort_month")
    )
    cohort_change       = round(cohort_month0.iloc[-1]["month0_rate"] - cohort_month0.iloc[0]["month0_rate"], 1) \
                          if len(cohort_month0) >= 2 else 0
    cohort_change_label = f"+{cohort_change}%" if cohort_change >= 0 else f"{cohort_change}%"
    cohort_change_color = BTA_SAGE_DK if cohort_change >= 0 else BTA_CORAL_DK

    activity_per_user = spine_filt.groupby("user_key")["activity_month"].nunique()
    total_with_wp     = len(activity_per_user)
    retained_users    = (activity_per_user >= 2).sum()
    retained_rate     = round(retained_users / total_with_wp * 100, 1) if total_with_wp > 0 else 0

    # ── 5 KPI CARDS ───────────────────────────────────────────────────────────
    def g1_kpi(col, label, value, tooltip, status, status_color, accent):
        with col:
            st.markdown(f"""
            <div class="g1-kpi-card" style="border-top:5px solid {accent};">
                <div class="g1-kpi-label">
                    {label}
                    <span class="g1-tooltip-wrap">
                        <span class="g1-info-icon">i</span>
                        <span class="g1-tooltip-text">{tooltip}</span>
                    </span>
                </div>
                <div class="g1-kpi-value" style="color:{accent};">{value}</div>
                <div class="g1-kpi-status" style="color:{status_color};">● {status}</div>
            </div>""", unsafe_allow_html=True)

    k1, k2, k3, k4, k5 = st.columns(5)
    g1_kpi(k1, "First-Month Engagement", f"{month0_rate}%",
           "% of new members who completed a survey in the same month they joined. Target: 40%+.",
           "Needs attention" if month0_rate < 40 else "On track",
           BTA_CORAL_DK if month0_rate < 40 else BTA_SAGE_DK, BTA_ORANGE)

    g1_kpi(k2, "3-Month Retention", f"{month3_rate}%",
           "Of members who joined 3+ months ago, % still completing surveys at month 3. Target: 25%+.",
           "Needs attention" if month3_rate < 25 else "On track",
           BTA_CORAL_DK if month3_rate < 25 else BTA_SAGE_DK, BTA_YELLOW)

    g1_kpi(k3, "Avg. Active Months", f"M{avg_dropoff_month}",
           "The average month number when members stop responding. Higher is better. Target: Month 3+.",
           "Longer = stronger loyalty", BTA_DARK, BTA_SAGE)

    g1_kpi(k4, "Cohort Trend", cohort_change_label,
           "Compares first-month engagement of the newest vs earliest signup group. Positive = improving. Target: positive.",
           "Improving" if cohort_change >= 0 else "Declining",
           cohort_change_color, cohort_change_color)

    g1_kpi(k5, "True Repeat Members", f"{retained_rate}%",
           f"{retained_users} members responded in 2+ separate months — genuine repeat engagement. Target: 30%+.",
           "Needs attention" if retained_rate < 30 else "On track",
           BTA_CORAL_DK if retained_rate < 30 else BTA_SAGE_DK, BTA_PERIWINKLE)

    st.markdown("<br>", unsafe_allow_html=True)

    # ── EXECUTIVE VERDICT ─────────────────────────────────────────────────────
    verdict_color = BTA_ORANGE if month0_rate < 40 or month3_rate < 25 else BTA_SAGE_DK
    verdict_label = "🟠 AMBER" if month0_rate < 40 or month3_rate < 25 else "🟢 GREEN"
    verdict_msg   = (
        f"First-month engagement is <b>{'below' if month0_rate < 40 else 'meeting'} target</b> "
        f"({month0_rate}% vs 40% goal). "
        f"3-month retention is <b>{'below' if month3_rate < 25 else 'meeting'} benchmark</b> "
        f"({month3_rate}% vs 25% goal). "
        f"Repeat participation at {retained_rate}% — "
        f"{'moderate, room to grow' if retained_rate < 50 else 'healthy'}. "
        f"<b>Priority:</b> Strengthen onboarding and introduce a month-2 re-engagement campaign."
    )
    st.markdown(f"""
    <div style="background:#FFF4E5;border-left:6px solid {verdict_color};
                border-radius:10px;padding:16px 20px;
                margin-top:4px;margin-bottom:24px;
                color:{BTA_DARK};font-family:Montserrat,sans-serif;line-height:1.7;">
        <b>{verdict_label} — Retention Performance</b><br>{verdict_msg}
    </div>
    """, unsafe_allow_html=True)

    # ── HELPER: insight box ───────────────────────────────────────────────────
    def g1_insight(text):
        st.markdown(f"""
        <div class="g1-chart-insight">
            <span class="g1-insight-icon">i</span>
            <span>{text}</span>
        </div>
        """, unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)

    # ── CHART 1: RETENTION HEATMAP ────────────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_DARK};">Retention Heatmap</div>', unsafe_allow_html=True)

    # Heatmap-only cohort retention mart (keeps denominator fixed by signup cohort)
    heatmap_pool = wp_g1.groupby("cohort_month")["id"].nunique().reset_index(name="total_registered_pool")
    heatmap_activity = g1_spine.groupby(["cohort_month", "month_offset"])["user_key"].nunique().reset_index(name="active_responders")
    heatmap_mart = heatmap_activity.merge(heatmap_pool, on="cohort_month", how="left")
    heatmap_mart["retention_rate_pct"] = (heatmap_mart["active_responders"] / heatmap_mart["total_registered_pool"] * 100).round(2)

    # Hide very small cohorts
    heatmap_mart.loc[heatmap_mart["total_registered_pool"] < 5, "retention_rate_pct"] = None

    pivot = heatmap_mart.pivot_table(
        index="cohort_month",
        columns="month_offset",
        values="retention_rate_pct",
        aggfunc="mean"
    )

    fig_hm = go.Figure(
        go.Heatmap(
            z=pivot.values,
            x=[f"Month {c}" for c in pivot.columns],
            y=pivot.index.tolist(),
            colorscale=[
                [0, BTA_BG],
                [0.4, BTA_YELLOW],
                [0.75, BTA_ORANGE],
                [1, BTA_ORANGE_DK]
            ],
            zmin=0,
            zmax=100,
            text=[
                [
                    f"{v:.0f}%" if (v is not None and not np.isnan(v)) else "–"
                    for v in row
                ]
                for row in pivot.values
            ],
            texttemplate="%{text}",
            colorbar=dict(
                title="Retention %",
                tickfont=dict(color=BTA_DARK)
            ),
            hovertemplate=(
                "<b>Signup group:</b> %{y}<br>"
                "<b>%{x}</b><br>"
                "<b>Still active:</b> %{z:.1f}%"
                "<extra></extra>"
            )
        )
    )

    fig_hm = style_fig(fig_hm, 480)
    fig_hm.update_layout(
        xaxis_title="Months Since Joining",
        yaxis_title="Signup Month",
        font=dict(color=BTA_DARK),
        paper_bgcolor=BTA_BG,
        plot_bgcolor=BTA_BG,
        dragmode="zoom"
    )

    st.plotly_chart(fig_hm, use_container_width=True, key="g1_heatmap")

    g1_insight(
        "<b>Rows</b> = groups of members who joined in the same month. "
        "<b>Columns</b> = months since joining. "
        "Each cell shows the percentage of the original signup group that was still completing surveys during that month. "
        "The denominator remains fixed based on the original cohort size. "
        "Darker cells indicate stronger retention. "
        "Cells marked '–' have fewer than 5 members and are hidden to avoid misleading results. "
        "Drag to zoom into any area."
    )

    st.markdown('<hr class="g1-section-divider">', unsafe_allow_html=True)

    # ── CHART 2: RETENTION CURVES ─────────────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_DARK};">Retention Over Time</div>', unsafe_allow_html=True)
    combined = (
        g1_filtered
        .groupby(["cohort_month","month_offset"])
        .agg(active_responders=("active_responders","sum"),
             total_registered_pool=("total_registered_pool","sum"))
        .reset_index()
    )
    combined["retention_rate_pct"] = (
        combined["active_responders"] / combined["total_registered_pool"] * 100
    ).round(2)
    cohort_colors = [BTA_ORANGE, BTA_CORAL, BTA_YELLOW, BTA_SAGE,
                     BTA_PERIWINKLE, BTA_LAVENDER, BTA_ORANGE_DK]
    fig_curve = go.Figure()
    for i, cohort in enumerate(sorted(combined["cohort_month"].unique())):
        d     = combined[combined["cohort_month"] == cohort].sort_values("month_offset")
        color = cohort_colors[i % len(cohort_colors)]
        fig_curve.add_trace(go.Scatter(
            x=d["month_offset"], y=d["retention_rate_pct"],
            mode="lines+markers+text", name=cohort,
            line=dict(color=color, width=2.5), marker=dict(color=color, size=8),
            text=[f"{v:.0f}%" for v in d["retention_rate_pct"]],
            textposition="top center", textfont=dict(size=10, color=color),
            hovertemplate=f"<b>{cohort}</b><br>Month %{{x}}<br>%{{y:.1f}}% still active<extra></extra>"
        ))
    fig_curve.add_hline(y=50, line_dash="dash", line_color=BTA_SAGE,
                        annotation_text="50% benchmark", annotation_font_color=BTA_DARK)
    fig_curve = style_fig(fig_curve, 500)
    fig_curve.update_layout(
        xaxis_title="Months Since Joining", yaxis_title="% Still Active",
        yaxis_range=[0, 115],
        legend=dict(orientation="h", y=-0.14, font=dict(size=11, color=BTA_DARK)),
        font=dict(color=BTA_DARK), paper_bgcolor=BTA_BG, plot_bgcolor=BTA_BG, dragmode="zoom"
    )
    st.plotly_chart(fig_curve, use_container_width=True, key="g1_curve")
    g1_insight(
        "Each line is one signup group tracked over time. "
        "A flat line = strong engagement. A steep drop = members disengaging quickly. "
        "Lines above the dashed 50% marker mean more than half that group is still active. "
        "Drag to zoom."
    )

    st.markdown('<hr class="g1-section-divider">', unsafe_allow_html=True)

    # ── CHART 3: ROLE COMPARISON ──────────────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_DARK};">Budtender vs. Consumer Retention</div>', unsafe_allow_html=True)
    role_agg = (
        g1_marts
        .groupby(["bta_user_role","month_offset"])
        .agg(active_responders=("active_responders","sum"),
             total_registered_pool=("total_registered_pool","sum"))
        .reset_index()
    )
    role_agg["retention_rate_pct"] = (
        role_agg["active_responders"] / role_agg["total_registered_pool"] * 100
    ).round(2)
    role_styles = {
        "Budtender"      : (BTA_ORANGE_DK, "circle"),
        "Consumer"       : (BTA_PERIWINKLE, "square"),
        "Retail Manager" : (BTA_SAGE_DK,   "triangle-up"),
    }
    fig_role = go.Figure()
    for role, (color, symbol) in role_styles.items():
        d = role_agg[role_agg["bta_user_role"] == role].sort_values("month_offset")
        if d.empty: continue
        fig_role.add_trace(go.Scatter(
            x=d["month_offset"], y=d["retention_rate_pct"],
            mode="lines+markers+text", name=role,
            line=dict(color=color, width=3),
            marker=dict(color=color, size=10, symbol=symbol),
            text=[f"{v:.0f}%" for v in d["retention_rate_pct"]],
            textposition="top center", textfont=dict(size=10, color=color),
            hovertemplate=f"<b>{role}</b><br>Month %{{x}}<br>%{{y:.1f}}% still active<extra></extra>"
        ))
    fig_role.add_hline(y=50, line_dash="dash", line_color=BTA_SAGE)
    fig_role = style_fig(fig_role, 500)
    fig_role.update_layout(
        xaxis_title="Months Since Joining", yaxis_title="% Still Active",
        yaxis_range=[0, 120],
        legend=dict(orientation="h", y=-0.12, font=dict(size=12, color=BTA_DARK)),
        font=dict(color=BTA_DARK), paper_bgcolor=BTA_BG, plot_bgcolor=BTA_BG, dragmode="zoom"
    )
    st.plotly_chart(fig_role, use_container_width=True, key="g1_role_chart")
    g1_insight(
        "One line per member type. A higher line = that group stays more engaged over time. "
        "Use this to tailor campaigns — if one group drops off faster, they need different outreach or incentives."
    )

    st.markdown('<hr class="g1-section-divider">', unsafe_allow_html=True)

    # ── PROVINCE RETENTION RANKING TABLE ─────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_DARK};">Province Retention Ranking</div>', unsafe_allow_html=True)
    st.markdown(
        f"<p style='font-size:13px;color:{BTA_DARK};opacity:0.65;margin-bottom:12px;'>"
        "Ranked by average Month-0 retention rate. Provinces with fewer than 5 members are excluded.</p>",
        unsafe_allow_html=True
    )

    prov_retention = (
        g1_marts[g1_marts["month_offset"] == 0]
        .groupby("province")
        .apply(lambda x: pd.Series({
            "members"       : int(x["total_registered_pool"].sum()),
            "active_month0" : int(x["active_responders"].sum()),
            "retention_pct" : round(x["active_responders"].sum() / x["total_registered_pool"].sum() * 100, 1)
                              if x["total_registered_pool"].sum() > 0 else 0
        }))
        .reset_index()
    )
    prov_retention = prov_retention[
        (prov_retention["members"] >= 5) &
        (~prov_retention["province"].isin(["Unknown", ""]))
    ].sort_values("retention_pct", ascending=False).reset_index(drop=True)

    prov_retention["rank"]  = prov_retention.index + 1
    prov_retention["trend"] = prov_retention["retention_pct"].apply(
        lambda v: "🟢 Strong" if v >= 40 else ("🟡 Moderate" if v >= 20 else "🔴 Needs attention")
    )

    if not prov_retention.empty:
        rows_html = ""
        for _, row in prov_retention.iterrows():
            rows_html += f"""
            <tr>
                <td>#{int(row['rank'])}</td>
                <td><b>{row['province']}</b></td>
                <td>{int(row['members'])}</td>
                <td>{int(row['active_month0'])}</td>
                <td><b>{row['retention_pct']}%</b></td>
                <td>{row['trend']}</td>
            </tr>"""
        st.markdown(f"""
        <table class="g1-province-table">
            <thead><tr>
                <th>Rank</th><th>Province</th><th>Members</th>
                <th>Active (Month 0)</th><th>Retention Rate</th><th>Status</th>
            </tr></thead>
            <tbody>{rows_html}</tbody>
        </table>
        """, unsafe_allow_html=True)
    else:
        st.info("Not enough province data to rank — fewer than 5 members per province after filtering.")

    st.markdown("<br>", unsafe_allow_html=True)
    g1_insight(
        "Ranks each province by how many new members engaged in their first month. "
        "🟢 Strong = 40%+ first-month engagement. 🟡 Moderate = 20–39%. 🔴 Needs attention = below 20%. "
        "Use this to identify where local outreach or targeted campaigns could have the most impact."
    )

    st.markdown('<hr class="g1-section-divider">', unsafe_allow_html=True)

    # ── CHART 4: ACTIVITY CALENDAR ────────────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_DARK};">When Members Are Most Active</div>',
                unsafe_allow_html=True)
    import glob as _glob

    # Honor the globally filtered dataset directly
    fct_cal = filtered_fct.copy()

    # Fallback to file search ONLY if no filtered dataframe structure exists at all
    if fct_cal.empty:
        for _root in ["/content/drive/MyDrive/BTA_Project_Data",
                      "/content/drive/MyDrive/BTAxDashboardxAcsenda/Main CSV Files",
                      "/content/drive/MyDrive"]:
            _m = _glob.glob(os.path.join(_root, "fct_survey_respon*.csv"))
            if _m:
                fct_cal = pd.read_csv(_m[0])
                break

    _cal_col = next((c for c in fct_cal.columns if "created" in c.lower()), None)
    if _cal_col and not fct_cal.empty:
        fct_cal["created_at"] = pd.to_datetime(fct_cal[_cal_col], utc=True, errors="coerce")
        fct_cal = fct_cal.dropna(subset=["created_at"])
        fct_cal["date_only"] = fct_cal["created_at"].dt.date
        daily = fct_cal.groupby("date_only").size().reset_index(name="responses")
        daily["date_only"] = pd.to_datetime(daily["date_only"])
        daily["week"]      = daily["date_only"].dt.isocalendar().week.astype(int)
        daily["day_name"]  = daily["date_only"].dt.day_name()
        day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

        cal_pivot = daily.pivot_table(
            index="day_name", columns="week",
            values="responses", aggfunc="sum"
        ).reindex(day_order)

        fig_cal = go.Figure(go.Heatmap(
            z=cal_pivot.values,
            x=[str(w) for w in cal_pivot.columns],
            y=cal_pivot.index.tolist(),
            colorscale=[[0, BTA_BG],[0.4, BTA_YELLOW],[0.75, BTA_ORANGE],[1, BTA_CORAL]],
            colorbar=dict(title="Responses", tickfont=dict(color=BTA_DARK)),
            hovertemplate="<b>Day:</b> %{y}<br><b>Week:</b> %{x}<br><b>Responses:</b> %{z}<extra></extra>"
        ))
        fig_cal = style_fig(fig_cal, 420)
        fig_cal.update_layout(xaxis_title="Week Number", yaxis_title="",
                              font=dict(color=BTA_DARK), paper_bgcolor=BTA_BG,
                              plot_bgcolor=BTA_BG, dragmode="zoom")
        st.plotly_chart(fig_cal, use_container_width=True, key="g1_calendar")
        g1_insight(
            "<b>Rows</b> = days of the week. <b>Columns</b> = week numbers across the year. "
            "Darker = more survey responses that day. "
            "Use this to schedule new surveys and member reminders when activity is already highest."
        )
    else:
        st.info("Activity calendar unavailable — matching filter selections yielded no records.")

    st.markdown('<hr class="g1-section-divider">', unsafe_allow_html=True)

    # ── AT-RISK MEMBERS SUMMARY ───────────────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_CORAL_DK};">⚠️ At-Risk Members</div>',
                unsafe_allow_html=True)
    st.markdown(
        f"<p style='font-size:13px;color:{BTA_DARK};opacity:0.65;margin-bottom:12px;'>"
        "Members showing signs of disengagement who may need direct outreach.</p>",
        unsafe_allow_html=True
    )

    # Build filtered summary off the active spine matrix
    active_user_keys = set(spine_filt["user_key"].unique())
    responses_per_user = (
        spine_filt.groupby("user_key")
        .agg(
            total_responses   = ("responses_that_month", "sum"),
            last_active_month = ("activity_month",       "max"),
            months_active     = ("activity_month",       "nunique"),
            first_active_month= ("activity_month",       "min"),
        )
        .reset_index()
    )

    member_table = users_g1_clean[["user_key","bta_user_role","province"]].copy()
    member_table = member_table.merge(responses_per_user, on="user_key", how="left")
    member_table["total_responses"]    = member_table["total_responses"].fillna(0).astype(int)
    member_table["months_active"]      = member_table["months_active"].fillna(0).astype(int)
    member_table["last_active_month"]  = member_table["last_active_month"].fillna("Never")
    member_table["first_active_month"] = member_table["first_active_month"].fillna("Never")
    member_table["responding"]         = member_table["user_key"].isin(active_user_keys)

    all_months = sorted(spine_filt["activity_month"].unique()) if not spine_filt.empty else []
    latest_month = all_months[-1] if all_months else None

    never_responded = member_table[~member_table["responding"]].copy()
    one_time_only   = member_table[(member_table["months_active"] == 1)].copy()

    lapsing = pd.DataFrame()
    if latest_month:
        lapsing = member_table[
            (member_table["responding"]) &
            (member_table["last_active_month"] < latest_month) &
            (member_table["months_active"] >= 1)
        ].copy()

    ar1, ar2, ar3 = st.columns(3)
    with ar1:
        st.markdown(f"""
        <div class="g1-atrisk-card">
            <b style="font-size:28px;color:{BTA_CORAL_DK};">{len(never_responded)}</b><br>
            <b>Never Responded</b><br>
            Registered but have never completed a survey.
        </div>""", unsafe_allow_html=True)
    with ar2:
        st.markdown(f"""
        <div class="g1-atrisk-card">
            <b style="font-size:28px;color:{BTA_ORANGE};">{len(one_time_only)}</b><br>
            <b>One-Time Only</b><br>
            Responded once and never came back.
        </div>""", unsafe_allow_html=True)
    with ar3:
        st.markdown(f"""
        <div class="g1-atrisk-card">
            <b style="font-size:28px;color:{BTA_YELLOW};">{len(lapsing)}</b><br>
            <b>Going Quiet</b><br>
            Were active but haven't responded in the most recent month.
        </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    at_risk_view = st.radio(
        "View at-risk group",
        ["Never Responded", "One-Time Only", "Going Quiet"],
        horizontal=True,
        key="g1_atrisk_view"
    )

    if at_risk_view == "Never Responded":
        ar_df = never_responded[["user_key","bta_user_role","province"]].copy()
        ar_df.columns = ["Member ID","Role","Province"]
        ar_df["Status"] = "❌ Never responded"
    elif at_risk_view == "One-Time Only":
        ar_df = one_time_only[["user_key","bta_user_role","province","first_active_month","total_responses"]].copy()
        ar_df.columns = ["Member ID","Role","Province","Only Active Month","Responses"]
        ar_df["Status"] = "⚠️ One-time only"
    else:
        ar_df = lapsing[["user_key","bta_user_role","province","last_active_month","months_active","total_responses"]].copy()
        ar_df.columns = ["Member ID","Role","Province","Last Active Month","Months Active","Responses"]
        ar_df["Status"] = "🟡 Going quiet"

    st.caption(f"{len(ar_df):,} members in this group")
    st.dataframe(ar_df, use_container_width=True, hide_index=True, height=320)
    st.download_button(
        label=f"⬇ Download {at_risk_view} list",
        data=ar_df.to_csv(index=False).encode(),
        file_name=f"g1_atrisk_{at_risk_view.lower().replace(' ','_')}.csv",
        mime="text/csv",
        key="g1_atrisk_dl"
    )

    st.markdown('<hr class="g1-section-divider">', unsafe_allow_html=True)

    # ── MEMBER ENGAGEMENT SUMMARY TABLE ──────────────────────────────────────
    st.markdown(f'<div class="chart-title" style="color:{BTA_DARK};">Member Engagement Summary</div>',
                unsafe_allow_html=True)
    st.markdown(
        f"<p style='font-size:13px;color:{BTA_DARK};opacity:0.65;margin-bottom:12px;'>"
        "Full member list showing who is active and who is not — use this to plan direct outreach.</p>",
        unsafe_allow_html=True
    )

    member_table["engagement_status"] = member_table["responding"].map(
        {True: "✅ Active", False: "❌ Not responding"}
    )
    display_cols = member_table[["user_key","bta_user_role","province",
                                  "total_responses","months_active",
                                  "last_active_month","engagement_status"]].copy()
    display_cols.columns = ["Member ID","Role","Province","Total Responses",
                             "Months Active","Last Active Month","Status"]

    tc1, tc2 = st.columns(2)
    with tc1:
        status_filter = st.selectbox("Filter by status",
            ["All Members","✅ Active only","❌ Not responding only"], key="g1_table_status")
    with tc2:
        role_filter_table = st.selectbox("Filter by role",
            ["All Roles"] + sorted(display_cols["Role"].dropna().unique().tolist()),
            key="g1_table_role")

    df_show = display_cols.copy()
    if status_filter == "✅ Active only":
        df_show = df_show[df_show["Status"] == "✅ Active"]
    elif status_filter == "❌ Not responding only":
        df_show = df_show[df_show["Status"] == "❌ Not responding"]
    if role_filter_table != "All Roles":
        df_show = df_show[df_show["Role"] == role_filter_table]

    st.caption(
        f"Showing {len(df_show):,} members — "
        f"{(display_cols['Status'] == '✅ Active').sum():,} active · "
        f"{(display_cols['Status'] == '❌ Not responding').sum():,} not responding"
    )
    st.dataframe(
        df_show.sort_values(["Status","Total Responses"], ascending=[True, False]),
        use_container_width=True, hide_index=True, height=420
    )

    st.download_button(
        label="⬇ Download full member engagement list as CSV",
        data=df_show.to_csv(index=False).encode(),
        file_name="g1_member_engagement.csv",
        mime="text/csv",
        key="g1_download"
    )

    st.markdown("<br>", unsafe_allow_html=True)

    # ── ABOUT THIS DATA ───────────────────────────────────────────────────────
    st.markdown(f"""
    <div style="background:{BTA_BG};border-left:5px solid {BTA_ORANGE};border-radius:8px;
                padding:14px 18px;color:{BTA_DARK};font-size:12px;line-height:1.7;
                font-family:Montserrat,sans-serif;">
        <b>About this data</b> &nbsp;·&nbsp;
        Figures reflect survey participation only — not logins or purchases.
        Internal users (interns, admins, brand managers) are excluded from all numbers.
        Members without a matched signup date (~11%) are excluded from retention calculations
        only — they still appear in the member table.
        Cells with fewer than 5 members are hidden in charts to avoid misleading percentages.
        Province rankings exclude groups with fewer than 5 members.
    </div>
    """, unsafe_allow_html=True)

# =============================================================
# G2 · SURVEY CADENCE
# TEAM MEMBERS: MARK, NINA, YOSHIKO
# =============================================================
elif page == "G2":

    # -----------------------------
    # BTA brand color palette used for the G2 page design.
    # These colors are applied to KPI cards, chart colors, borders, and text.
    # -----------------------------
    BTA_BG = "#F8F3E8"
    BTA_RED = "#E26655"
    BTA_CORAL = "#E6866F"
    BTA_ORANGE = "#E38F55"
    BTA_YELLOW = "#F4B35E"
    BTA_SAGE = "#B0BFA5"
    BTA_BLUE = "#838DB9"
    BTA_PURPLE = "#9F83B8"
    BTA_DARK = "#263126"
    BTA_MUTED = "#5A4A3E"

    # -----------------------------
    # Custom font setup for G2 headings.
    # The code checks several possible Google Drive locations for the Kelly-Bold font.
    # If the font is found, it is converted to base64 so Streamlit can use it in CSS.
    # -----------------------------
    # Kelly Bold font for G2 heading/title text
    import base64

    KELLY_FONT_CSS = ""
    kelly_font_paths = [
        "/content/drive/MyDrive/BTAxDashboardxAcsenda/Kelly-Bold.otf",
        "/content/drive/MyDrive/BTAxDashboardxAcsenda/Fonts/Kelly-Bold.otf",
        "/content/drive/MyDrive/BTAxDashboardxAcsenda/fonts/Kelly-Bold.otf",
        "/content/drive/MyDrive/Kelly-Bold.otf",
    ]

    for font_path in kelly_font_paths:
        if os.path.exists(font_path):
            with open(font_path, "rb") as font_file:
                font_data = base64.b64encode(font_file.read()).decode()
            KELLY_FONT_CSS = f"""
            @font-face {{
                font-family: 'KellyBold';
                src: url(data:font/otf;base64,{font_data}) format('opentype');
                font-weight: 900;
                font-style: normal;
            }}
            """
            break

    # -----------------------------
    # CSS styling for the G2 tab.
    # This controls the page title, subtitle, KPI cards, chart titles,
    # insight boxes, notes, and radio button styling.
    # -----------------------------
    st.markdown("""
    <style>
    """ + KELLY_FONT_CSS + """
        .g2-title {
            font-family: 'KellyBold', Georgia, serif !important;
            font-size: 42px;
            font-weight: normal !important;
            color: #E26655;
            margin-bottom: 5px;
        }

        .g2-subtitle {
            font-size: 17px;
            color: #5A4A3E;
            margin-bottom: 25px;
        }

        .g2-kpi-card {
            background: rgba(255,255,255,0.60);
            border-radius: 18px;
            padding: 20px 16px;
            text-align: center;
            min-height: 135px;
            box-shadow: 0 10px 22px rgba(0,0,0,0.08);
            border-top: 7px solid #E6866F;
        }

        .g2-kpi-label {
            color: #5A4A3E;
            font-size: 14px;
            font-weight: 700;
            margin-bottom: 8px;
        }

        .g2-kpi-value {
            color: #263126;
            font-size: 30px;
            font-weight: 900;
            line-height: 1.1;
            white-space: nowrap;
        }

        .g2-tooltip {
            cursor: help;
            color: #E26655;
            font-weight: 900;
        }

        .g2-insight {
            background: rgba(255,255,255,0.60);
            border-left: 8px solid #E38F55;
            border-radius: 18px;
            padding: 20px 24px;
            color: #263126;
            font-size: 15px;
            line-height: 1.7;
            margin-top: 10px;
            margin-bottom: 25px;
            box-shadow: 0 10px 22px rgba(0,0,0,0.08);
        }

        .g2-chart-title {
            font-family: 'KellyBold', Georgia, serif !important;
            font-size: 24px;
            font-weight: normal !important;
            color: #E26655;
            margin-top: 30px;
            margin-bottom: 10px;
        }

        .g2-dataset-title {
            font-family: 'KellyBold', Georgia, serif !important;
            font-size: 30px;
            font-weight: normal !important;
            color: #E26655;
            margin-top: 20px;
            margin-bottom: 15px;
        }

        .js-plotly-plot .plotly .gtitle {
            font-family: 'KellyBold', Georgia, serif !important;
            font-weight: normal !important;
        }

        .g2-note {
            background: rgba(255,255,255,0.55);
            border-left: 6px solid #B0BFA5;
            border-radius: 14px;
            padding: 15px 18px;
            margin-top: 10px;
            margin-bottom: 18px;
            color: #263126;
            font-size: 14px;
            line-height: 1.6;
        }

        div[data-testid="stRadio"] > div {
            display: flex;
            justify-content: center;
            gap: 28px;
            margin-top: 18px;
            margin-bottom: 28px;
        }

        div[data-testid="stRadio"] label {
            background: linear-gradient(135deg, #FF3D2E, #FFB238);
            padding: 13px 28px;
            border-radius: 28px;
            font-weight: 800;
            color: black;
            box-shadow: 0 8px 18px rgba(0,0,0,0.15);
            border: 3px solid white;
        }

        div[data-testid="stRadio"] label:hover {
            transform: translateY(-2px);
            cursor: pointer;
        }
    </style>
    """, unsafe_allow_html=True)


    # -----------------------------
    # Load the prepared G2 survey cadence dataset from Google Drive.
    # This CSV contains the processed survey attempt and gap information.
    # -----------------------------
    g2 = pd.read_csv(
        "/content/drive/MyDrive/BTAxDashboardxAcsenda/G2/g2_survey_cadence_feature.csv"
    )

    # Convert attempt start and end columns into datetime format.
    # This allows the dashboard to apply time period filters correctly.
    g2["attempt_start"] = pd.to_datetime(g2["attempt_start"], errors="coerce")
    g2["attempt_end"] = pd.to_datetime(g2["attempt_end"], errors="coerce")
    # Convert the days_between_attempts column into a numeric value.
    # Invalid or missing values are converted to NaN instead of breaking the app.
    g2["days_between_attempts"] = pd.to_numeric(
        g2["days_between_attempts"],
        errors="coerce"
    )

    # Apply MAIN SIDEBAR filters to G2.
    # No separate G2 role/province/date filters are used.
    g2 = apply_global_user_filter(g2, "user_key")
    g2 = apply_global_date_filter(g2, "attempt_start")

    # Prepare variables for possible role and province columns.
    # These are checked dynamically because different datasets may use different column names.
    role_col = None
    province_col = None

    possible_role_cols = ["role", "user_role", "user_type", "member_type", "account_type"]
    possible_province_cols = ["province", "user_province", "state", "region"]

    # Find the first role column that exists in the dataset.
    for col in possible_role_cols:
        if col in g2.columns:
            role_col = col
            break

    # Find the first province/location column that exists in the dataset.
    for col in possible_province_cols:
        if col in g2.columns:
            province_col = col
            break

    # Create a filtered copy of the G2 dataset after applying the global sidebar filters.
    filtered_g2 = g2.copy()



    # Keep only rows with valid gap values for gap-based calculations and charts.
    gaps = filtered_g2.dropna(subset=["days_between_attempts"]).copy()


    # -----------------------------
    # Calculate KPI values shown at the top of the G2 dashboard.
    # These values update based on the global filters selected in the sidebar.
    # -----------------------------
    total_attempts = len(filtered_g2)
    total_users = filtered_g2["user_key"].nunique() if "user_key" in filtered_g2.columns else 0
    median_gap = gaps["days_between_attempts"].median() if len(gaps) > 0 else 0
    avg_gap = gaps["days_between_attempts"].mean() if len(gaps) > 0 else 0

    # Count users with a 60+ day inactivity gap.
    # If the dataset already has a churn flag, use it; otherwise calculate it from days_between_attempts.
    if "churn_60_plus_gap" in filtered_g2.columns:
        churn_users = filtered_g2[filtered_g2["churn_60_plus_gap"] == True]["user_key"].nunique()
    else:
        churn_users = gaps[gaps["days_between_attempts"] >= 60]["user_key"].nunique()

    churn_rate = (churn_users / total_users * 100) if total_users > 0 else 0

    # Reusable function for displaying KPI cards with a label, value, tooltip, and brand color.
    def g2_kpi_card(label, value, tooltip, border_color):
        st.markdown(
            f"""
            <div class="g2-kpi-card" style="border-top-color:{border_color};">
                <div class="g2-kpi-label">
                    {label}
                    <span class="g2-tooltip" title="{tooltip}"> ⓘ</span>
                </div>
                <div class="g2-kpi-value">{value}</div>
            </div>
            """,
            unsafe_allow_html=True
        )

    # Create four columns so the KPI cards appear side by side.
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        g2_kpi_card(
            "Total Attempts",
            f"{total_attempts:,}",
            "Total number of survey attempts after filters are applied.",
            BTA_RED
        )

    with col2:
        g2_kpi_card(
            "Responding Users",
            f"{total_users:,}",
            "Number of unique users who completed at least one survey attempt.",
            BTA_ORANGE
        )

    with col3:
        g2_kpi_card(
            "Median Gap",
            f"{median_gap:.2f} days",
            "Middle value of days between survey attempts.",
            BTA_SAGE
        )

    with col4:
        g2_kpi_card(
            "60+ Day Gap Users",
            f"{churn_users:,}",
            "Users who had at least one inactivity gap of 60 days or more.",
            BTA_BLUE
        )

    st.markdown("<br>", unsafe_allow_html=True)

    # Display an executive summary that explains the main G2 findings in plain language.
    st.markdown(f"""
    <div class="g2-insight">
        <b>Executive Insight</b><br>
        After applying the selected filters, G2 found <b>{total_attempts:,}</b> survey attempts
        from <b>{total_users:,}</b> responding users.
        The median return gap is <b>{median_gap:.2f} days</b>, while
        <b>{churn_users:,}</b> users show a 60+ day inactivity gap
        ({churn_rate:.1f}% of responding users).
        This helps BTA identify when users may be disengaging.
    </div>
    """, unsafe_allow_html=True)

    # Reusable chart styling function.
    # This keeps all G2 Plotly charts visually consistent with the BTA brand.
    def style_g2_chart(fig, height=430):
        fig.update_layout(
            template="plotly_white",
            height=height,
            paper_bgcolor="rgba(255,255,255,0.55)",
            plot_bgcolor="rgba(255,255,255,0.35)",
            font=dict(color=BTA_DARK, size=13),
            title=dict(
                font=dict(color=BTA_RED, size=20, family="KellyBold, Georgia, serif"),
                x=0.02
            ),
            margin=dict(l=40, r=35, t=65, b=50),
            xaxis=dict(
                gridcolor="rgba(90,74,62,0.15)",
                zerolinecolor="rgba(90,74,62,0.15)",
                linecolor=BTA_MUTED,
                tickfont=dict(color=BTA_MUTED),
                titlefont=dict(color=BTA_MUTED)
            ),
            yaxis=dict(
                gridcolor="rgba(90,74,62,0.15)",
                zerolinecolor="rgba(90,74,62,0.15)",
                linecolor=BTA_MUTED,
                tickfont=dict(color=BTA_MUTED),
                titlefont=dict(color=BTA_MUTED)
            )
        )
        return fig

    # If filters remove all records, show a warning instead of displaying empty charts.
    if filtered_g2.empty:
        st.warning("No G2 data available after applying the selected filters.")

    else:

        # -----------------------------
        # Chart 1: Inter-Attempt Gap Buckets
        # Groups survey return gaps into simple time ranges for easier interpretation.
        # -----------------------------
        st.markdown('<div class="g2-chart-title">1. Inter-Attempt Gap Buckets</div>', unsafe_allow_html=True)

        # Replaces the old box plot with a clearer bucket chart.
        # This is easier to explain in a demo because it shows how many survey returns
        # happened within each engagement window.
        gap_bucket_labels = [
            "Same day",
            "1–7 days",
            "8–14 days",
            "15–30 days",
            "31–60 days",
            "60+ days"
        ]

        if gaps.empty:
            st.info("No inter-attempt gap data available after filters are applied.")
        else:
            # Create gap buckets based on the number of days between survey attempts.
            gaps["gap_bucket"] = pd.cut(
                gaps["days_between_attempts"],
                bins=[-0.01, 0, 7, 14, 30, 60, float("inf")],
                labels=gap_bucket_labels
            )

            # Count how many repeat attempts fall into each gap bucket.
            gap_bucket_summary = (
                gaps.groupby("gap_bucket", observed=False)
                .size()
                .reset_index(name="number_of_attempts")
            )

            gap_bucket_summary["gap_bucket"] = pd.Categorical(
                gap_bucket_summary["gap_bucket"],
                categories=gap_bucket_labels,
                ordered=True
            )

            gap_bucket_summary = gap_bucket_summary.sort_values("gap_bucket")

            # Build the bar chart for the inter-attempt gap bucket summary.
            fig_gap_buckets = px.bar(
                gap_bucket_summary,
                x="gap_bucket",
                y="number_of_attempts",
                title="Inter-Attempt Gap Buckets",
                text="number_of_attempts",
                labels={
                    "gap_bucket": "Days Between Survey Attempts",
                    "number_of_attempts": "Number of Repeat Attempts"
                },
                color_discrete_sequence=[BTA_CORAL]
            )

            fig_gap_buckets.update_traces(
                marker=dict(
                    color=BTA_CORAL,
                    line=dict(color=BTA_RED, width=1)
                ),
                textposition="outside"
            )

            fig_gap_buckets.update_xaxes(title="Days Between Survey Attempts")
            fig_gap_buckets.update_yaxes(title="Number of Repeat Attempts")
            fig_gap_buckets = style_g2_chart(fig_gap_buckets, 450)

            st.plotly_chart(fig_gap_buckets, use_container_width=True)

        st.markdown("""
        <div class="g2-note">
            <b>What this means:</b>
            This chart groups return gaps into simple time ranges.
            Same-day and 1–7 day returns show stronger engagement, while 31–60 day and 60+ day gaps may show users who are becoming inactive.
        </div>
        """, unsafe_allow_html=True)

        # -----------------------------
        # Chart 2: Survey Attempts per User
        # Shows how many surveys each user completed.
        # -----------------------------
        st.markdown('<div class="g2-chart-title">2. Survey Attempts per User</div>', unsafe_allow_html=True)

        # Count total survey attempts for each user.
        attempts_per_user = (
            filtered_g2.groupby("user_key")
            .size()
            .reset_index(name="total_attempts")
        )

        # Build a histogram to show the distribution of attempts per user.
        fig_attempts = px.histogram(
            attempts_per_user,
            x="total_attempts",
            nbins=25,
            title="Survey Attempts per User",
            labels={"total_attempts": "Number of Attempts per User"},
            color_discrete_sequence=[BTA_YELLOW]
        )

        fig_attempts.update_traces(
            marker=dict(
                color=BTA_YELLOW,
                line=dict(color=BTA_ORANGE, width=1)
            )
        )

        fig_attempts.update_xaxes(title="Number of Attempts per User")
        fig_attempts.update_yaxes(title="Number of Users")
        fig_attempts = style_g2_chart(fig_attempts, 420)

        st.plotly_chart(fig_attempts, use_container_width=True)

        st.markdown("""
        <div class="g2-note">
            <b>What this means:</b>
            This chart shows whether users are completing surveys once, a few times,
            or repeatedly. More repeat attempts show stronger participation habits.
        </div>
        """, unsafe_allow_html=True)

        # -----------------------------
        # Chart 3: 60+ Day Gap Churn Indicator
        # Shows how many users have a long inactivity gap.
        # -----------------------------
        st.markdown('<div class="g2-chart-title">3. 60+ Day Gap Churn Indicator</div>', unsafe_allow_html=True)

        # If the churn flag is missing, create it using a 60-day gap rule.
        if "churn_60_plus_gap" not in filtered_g2.columns:
            filtered_g2["churn_60_plus_gap"] = filtered_g2["days_between_attempts"] >= 60

        # Count unique users with and without a 60+ day gap.
        churn_summary = (
            filtered_g2.groupby("churn_60_plus_gap")["user_key"]
            .nunique()
            .reset_index()
        )

        churn_summary["churn_status"] = churn_summary["churn_60_plus_gap"].map({
            False: "No 60+ Day Gap",
            True: "Has 60+ Day Gap"
        })

        # Build the churn indicator bar chart.
        fig_churn = px.bar(
            churn_summary,
            x="churn_status",
            y="user_key",
            title="Users With and Without 60+ Day Gaps",
            text="user_key",
            labels={
                "churn_status": "Churn Status",
                "user_key": "Number of Users"
            },
            color="churn_status",
            color_discrete_map={
                "No 60+ Day Gap": BTA_SAGE,
                "Has 60+ Day Gap": BTA_RED
            }
        )

        fig_churn.update_traces(
            marker_line_color=BTA_DARK,
            marker_line_width=1,
            textposition="outside"
        )

        fig_churn.update_xaxes(title="Churn Status")
        fig_churn.update_yaxes(title="Number of Users")
        fig_churn = style_g2_chart(fig_churn, 420)

        st.plotly_chart(fig_churn, use_container_width=True)

        st.markdown("""
        <div class="g2-note">
            <b>What this means:</b>
            Users with a 60+ day gap may need re-engagement.
            This helps BTA identify members who are not participating consistently.
        </div>
        """, unsafe_allow_html=True)

        # -----------------------------
        # Chart 4: Average Gap by User Activity Level
        # Compares return gaps across low, medium, and high activity users.
        # -----------------------------
        st.markdown('<div class="g2-chart-title">4. Average Gap by User Activity Level</div>', unsafe_allow_html=True)

        # Summarize each user by total attempts and average days between attempts.
        user_summary = (
            filtered_g2.groupby("user_key")
            .agg(
                total_attempts=("survey_id", "count"),
                avg_gap=("days_between_attempts", "mean")
            )
            .reset_index()
        )

        # Categorize users into activity levels based on their number of attempts.
        def activity_level(attempts):
            if attempts <= 2:
                return "1–2 attempts"
            elif attempts <= 5:
                return "3–5 attempts"
            elif attempts <= 10:
                return "6–10 attempts"
            else:
                return "10+ attempts"

        user_summary["activity_level"] = user_summary["total_attempts"].apply(activity_level)

        activity_order = [
            "1–2 attempts",
            "3–5 attempts",
            "6–10 attempts",
            "10+ attempts"
        ]

        # Calculate the average gap for each activity level.
        activity_gap = (
            user_summary.groupby("activity_level")["avg_gap"]
            .mean()
            .reset_index()
        )

        activity_gap["activity_level"] = pd.Categorical(
            activity_gap["activity_level"],
            categories=activity_order,
            ordered=True
        )

        activity_gap = activity_gap.sort_values("activity_level")

        # Build the average gap by activity level bar chart.
        fig_activity = px.bar(
            activity_gap,
            x="activity_level",
            y="avg_gap",
            title="Average Gap by User Activity Level",
            text="avg_gap",
            color_discrete_sequence=[BTA_PURPLE],
            labels={
                "activity_level": "User Activity Level",
                "avg_gap": "Average Days Between Attempts"
            }
        )

        fig_activity.update_traces(
            marker=dict(
                color=BTA_PURPLE,
                line=dict(color=BTA_BLUE, width=1)
            ),
            texttemplate="%{text:.2f}",
            textposition="outside"
        )

        fig_activity.update_xaxes(title="User Activity Level")
        fig_activity.update_yaxes(title="Average Days Between Attempts")
        fig_activity = style_g2_chart(fig_activity, 420)

        st.plotly_chart(fig_activity, use_container_width=True)

        st.markdown("""
        <div class="g2-note">
            <b>What this means:</b>
            This compares low-activity and high-activity users.
            If low-activity users have longer gaps, BTA can target them with reminders or engagement campaigns.
        </div>
        """, unsafe_allow_html=True)

    # Add a divider before showing the filtered dataset table.
    st.divider()

    # Display the filtered dataset used in the current G2 view.
    st.markdown("""
    <h2 class="g2-dataset-title">Filtered Cadence Dataset</h2>
    """, unsafe_allow_html=True)
    st.dataframe(filtered_g2, use_container_width=True)

    # Allow users to download the filtered G2 dataset as a CSV file.
    st.download_button(
        label="Download Filtered Cadence Dataset",
        data=filtered_g2.to_csv(index=False),
        file_name="filtered_g2_survey_cadence_feature.csv",
        mime="text/csv"
    )


# =============================================================
# G3 · UTILIZATION ANALYSIS
# TEAM MEMBERS: ANGELA, JAIME, JUNNAR
#
# ⬇️ G3 PASTE YOUR FINAL CHART CODE BELOW THIS SECTION ⬇️
# =============================================================
elif page == "G3":

    # ── G3 header ─────────────────────────────────────────────
    st.markdown('<div class="main-title">Points & Utilization</div>', unsafe_allow_html=True)
    st.markdown('<div class="main-subtitle">Points earned · Points spent · Utilization · Rewards · Roles · Activity level</div>', unsafe_allow_html=True)

    G3_ROLE_COLORS = {
        'Budtender':     '#E07A30',
        'Consumer':      '#2E86AB',
        'Store Manager': '#27AE60',
        'Unknown':       '#8E9E96',
    }

    # ── Shared G3 chart style (identical to G1 style_fig) ─────
    def g3_style(fig, height=380):
        fig.update_layout(
            template="plotly_white",
            height=height,
            paper_bgcolor="#F1F2EF",
            plot_bgcolor="#F1F2EF",
            font=dict(color="#2E342A", size=13),
            margin=dict(l=35, r=35, t=55, b=45),
            xaxis=dict(gridcolor="#CBD1C7", zerolinecolor="#CBD1C7",
                       linecolor="#4F6F5F", tickfont=dict(color="#4E554B", size=13),
                       title=dict(font=dict(color="#2E342A", size=15))),
            yaxis=dict(gridcolor="#CBD1C7", zerolinecolor="#CBD1C7",
                       linecolor="#4F6F5F", tickfont=dict(color="#4E554B", size=13),
                       title=dict(font=dict(color="#2E342A", size=15))),
            legend=dict(font=dict(color="#2E342A", size=12))
        )
        return fig

    # ── Reusable chart description box (the "box" under each title) ──
    def g3_desc(text):
        st.markdown(
            f"""<div style='background:#FFFFFF;border:1px solid #CBD1C7;
                        border-left:4px solid #4F6F5F;border-radius:8px;
                        padding:12px 16px;margin:2px 0 12px 0;
                        color:#2E342A;font-size:14px;line-height:1.6;'>
                {text}
            </div>""", unsafe_allow_html=True)

    # ── Use already-loaded users dataframe ────────────────────
    # ── Load G3 CSVs from shared Google Drive ─────────────────
    import glob as _g3glob

    G3_SEARCH_ROOTS = [
        "/content/drive/MyDrive/BTAxDashboardxAcsenda/Main CSV Files",
        "/content/drive/MyDrive/BTAxDashboardxAcsenda",
        "/content/drive/Shareddrives/BTAxDashboardxAcsenda/Main CSV Files",
        "/content/drive/Shareddrives/BTAxDashboardxAcsenda",
        "/content/drive/MyDrive",
    ]


    def g3_find_csv(filename):
        for root in G3_SEARCH_ROOTS:
            path = os.path.join(root, filename)
            if os.path.exists(path):
                return path
        results = _g3glob.glob(f"/content/drive/**/{filename}", recursive=True)
        return results[0] if results else None

    # Use the MAIN SIDEBAR filtered users only.
    # This makes G3 respond to Role, Province, Time Period, and Unknown/Internal filters.
    g3_users = filtered_users.copy()
    g3_users['lifetime_points_earned'] = pd.to_numeric(g3_users['lifetime_points_earned'], errors='coerce')
    g3_users['current_points_balance'] = pd.to_numeric(g3_users['current_points_balance'], errors='coerce')

    def g3_parse_rewards(value):
        if pd.isna(value): return []
        value = str(value).strip()
        if value.startswith('['):
            try: return json.loads(value)
            except: pass
        if value.startswith('a:'):
            try:
                r = re.findall(r's:\d+:\"([^\"]+)\"', value)
                if r: return r
            except: pass
        return [value]

    if 'preferred_rewards' in g3_users.columns:
        g3_users['preferred_rewards_parsed'] = g3_users['preferred_rewards'].apply(g3_parse_rewards)
    else:
        g3_users['preferred_rewards_parsed'] = [[] for _ in range(len(g3_users))]

    # ── Cleaning ──────────────────────────────────────────────
    g3_CAP = g3_users['lifetime_points_earned'].quantile(0.99)
    g3_users['lifetime_points_earned_clean'] = g3_users['lifetime_points_earned'].clip(upper=g3_CAP)
    g3_users['current_points_balance_clean'] = g3_users['current_points_balance'].clip(lower=0, upper=g3_CAP)
    g3_users['points_spent']            = g3_users['lifetime_points_earned_clean'] - g3_users['current_points_balance_clean']
    g3_users['utilization_rate']        = np.where(
        g3_users['lifetime_points_earned_clean'] > 0,
        g3_users['points_spent'] / g3_users['lifetime_points_earned_clean'], np.nan)
    g3_users['points_flag_implausible'] = g3_users['lifetime_points_earned'] > g3_CAP

    g3_bins_age   = [18,25,35,45,55,70]
    g3_labels_age = ['18-24','25-34','35-44','45-54','55+']
    g3_users['age_group'] = pd.cut(g3_users['age_years'], bins=g3_bins_age, labels=g3_labels_age, right=False)

    g3_earners = g3_users[
        (g3_users['lifetime_points_earned_clean'] > 0) &
        (~g3_users['points_flag_implausible'])
    ].copy()

    # ── GLOBAL SIDEBAR FILTERS ONLY ───────────────────────────
    # G3 uses filtered_users from the main sidebar.
    g3_e = g3_earners.copy()
    g3_uf = g3_users.copy()

    # ── KPI values ────────────────────────────────────────────
    g3_flagged_n    = int(g3_users['points_flag_implausible'].sum())
    g3_sys_util     = g3_e['points_spent'].sum() / g3_e['lifetime_points_earned_clean'].sum() * 100 if len(g3_e) else 0
    g3_mean_util    = g3_e['utilization_rate'].mean() * 100 if len(g3_e) else 0
    g3_total_users  = len(g3_uf)
    g3_active_users = int((g3_uf['lifetime_points_earned_clean'] > 0).sum())
    g3_avg_earned   = g3_uf.loc[g3_uf['lifetime_points_earned_clean'] > 0, 'lifetime_points_earned_clean'].mean()
    g3_avg_spent    = g3_uf.loc[g3_uf['lifetime_points_earned_clean'] > 0, 'points_spent'].mean()
    g3_pct_active   = g3_active_users / g3_total_users * 100 if g3_total_users else 0

    # ── KPI cards (exact G1 style) ────────────────────────────
    def g3_kpi(col, label, value, note, color, status, status_color):
        with col:
            st.markdown(f"""
            <div style='background:#F1F2EF;border-radius:16px;padding:20px 14px;
                        border-top:5px solid {color};
                        box-shadow:0 4px 14px rgba(0,0,0,0.07);min-height:160px;'>
                <div style='font-size:12px;font-weight:700;color:#4E554B;
                            text-transform:uppercase;letter-spacing:0.05em;'>{label}</div>
                <div style='font-size:28px;font-weight:900;color:{color};
                            margin:6px 0;line-height:1.1;'>{value}</div>
                <div style='font-size:11px;color:#888;font-style:italic;
                            margin-bottom:6px;'>{note}</div>
                <div style='font-size:10px;font-weight:700;
                            color:{status_color};'>● {status}</div>
            </div>""", unsafe_allow_html=True)

    # Row 1  (4 cards)
    k1,k2,k3,k4 = st.columns(4)
    g3_kpi(k1, "Total Users",        f"{g3_total_users:,}",   f"{g3_active_users:,} active ({g3_pct_active:.0f}%)", "#2E86AB", "Info",             "#2E86AB")
    g3_kpi(k2, "Active Users",       f"{g3_active_users:,}",  "Earned at least 1 point",                           "#27AE60", "On track" if g3_pct_active >= 40 else "Needs attention", "#27AE60" if g3_pct_active >= 40 else "#E07A30")
    g3_kpi(k3, "Avg Points Earned",  f"{g3_avg_earned:,.0f}", "Active users only",                                 "#8E44AD", "Info",             "#8E44AD")
    g3_kpi(k4, "System Utilization", f"{g3_sys_util:.1f}%",   "Weighted spent / earned",                           "#C0392B", "Needs attention" if g3_sys_util < 10 else "On track", "#C0392B")

    st.markdown("<br>", unsafe_allow_html=True)

    # Row 2  (4 cards — Total Points Earned card and Data-Quality Flag card removed)
    k5,k6,k7,k8 = st.columns(4)
    g3_kpi(k5, "Mean Utilization",   f"{g3_mean_util:.1f}%",                 "Per earner average",   "#E07A30", "Amber", "#E07A30")
    g3_kpi(k6, "Avg Points Spent",   f"{g3_avg_spent:,.0f}",                 "Active users only",    "#16A085", "Info",  "#16A085")
    g3_kpi(k7, "Total Points Spent", f"{g3_e['points_spent'].sum():,.0f}",   "Cleaned earners only", "#8E44AD", "Info",  "#8E44AD")
    g3_kpi(k8, "P99 Cap Applied",    f"{g3_CAP:,.0f}",                       "Points cap threshold", "#7F8C8D", "Info",  "#7F8C8D")

    st.markdown("<br>", unsafe_allow_html=True)

    # ── GLOBAL SIDEBAR FILTERS ONLY ───────────────────────────
    st.caption("G3 is controlled by the main sidebar filters above.")
    g3_chart_e = g3_e.copy()
    g3_chart_uf = g3_uf.copy()

    st.markdown("<br>", unsafe_allow_html=True)

    # ════════════════════════════════════════
    # CHARTS — matching G1 two-column layout
    # ════════════════════════════════════════

    # ── ROW 1: Utilization histogram + Mean util by role ──────
    col_l, col_r = st.columns(2)

    with col_l:
        st.markdown('<div class="chart-title">Utilization Distribution</div>', unsafe_allow_html=True)
        g3_desc(
            "Each bar counts how many earning users fall into a 5% utilization band (points spent ÷ points earned), "
            "with values clipped at 50% so a long tail doesn't flatten the view. The heavy pile-up on the left shows "
            "that most earners redeem almost none of their points. The dashed magenta line marks the median and the "
            "dotted orange line marks the mean — when the mean sits to the right of the median, a few heavy redeemers "
            "are pulling the average up."
        )

        g3_util_pct = g3_chart_e['utilization_rate'] * 100
        g3_bins     = list(range(0, 55, 5))
        g3_counts, _ = np.histogram(np.clip(g3_util_pct.dropna(), 0, 50), bins=g3_bins)
        g3_bin_labels = [f"{b}–{b+5}%" for b in g3_bins[:-1]]
        g3_bin_labels[-1] = "45–50%+"

        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=g3_bin_labels, y=g3_counts,
            marker=dict(color="#2E86AB", line=dict(color="#2E342A", width=0.8)),
            opacity=0.85, name="Earners",
            hovertemplate="Range: %{x}<br>Earners: %{y}<extra></extra>"
        ))

        # Correct vline positions using actual bin index
        g3_med = float(g3_util_pct.median())
        g3_mn  = float(g3_util_pct.mean())
        g3_med_idx = min(int(g3_med // 5), len(g3_bin_labels)-1)
        g3_mn_idx  = min(int(g3_mn  // 5), len(g3_bin_labels)-1)

        fig.add_vline(x=g3_med_idx, line_dash="dash", line_color="#A23B72", line_width=2,
            annotation_text=f"Median {g3_med:.1f}%",
            annotation_font=dict(color="#A23B72", size=11),
            annotation_position="top right")
        fig.add_vline(x=g3_mn_idx, line_dash="dot", line_color="#E07A30", line_width=2,
            annotation_text=f"Mean {g3_mn:.1f}%",
            annotation_font=dict(color="#E07A30", size=11),
            annotation_position="top left")

        fig = g3_style(fig, 380)
        fig.update_layout(title=dict(text="Most earners spend almost none of their points",
            font=dict(color="#2E342A", size=14)),
            xaxis_title="Utilization Rate %", yaxis_title="Number of Earners",
            showlegend=False)
        st.plotly_chart(fig, use_container_width=True, key="g3_util_hist")

    with col_r:
        st.markdown('<div class="chart-title">Mean Utilization by Role</div>', unsafe_allow_html=True)
        g3_desc(
            "The average utilization rate for each user role, shown only for roles with at least 5 earners (the "
            "sample size n is printed on every bar). A taller bar means that role redeems a larger share of the "
            "points it earns, which highlights who is most engaged with the rewards economy rather than simply "
            "hoarding points."
        )

        g3_by_role = (g3_chart_e
            .assign(role=g3_chart_e['bta_user_role'].fillna('Unknown'))
            .groupby('role')
            .agg(n=('user_key','size'), mean_u=('utilization_rate','mean'))
            .reset_index())
        g3_by_role = g3_by_role[g3_by_role['n'] >= 5].sort_values('mean_u', ascending=False)

        role_colors_list = [G3_ROLE_COLORS.get(r, '#8E9E96') for r in g3_by_role['role']]

        fig2 = go.Figure()
        for i, row in g3_by_role.iterrows():
            fig2.add_trace(go.Bar(
                x=[row['role']], y=[row['mean_u']*100],
                name=row['role'],
                marker=dict(color=G3_ROLE_COLORS.get(row['role'], '#8E9E96'),
                            line=dict(width=0)),
                text=[f"{row['mean_u']*100:.1f}%<br>n={row['n']}"],
                textposition='outside',
                textfont=dict(color="#2E342A", size=11),
                hovertemplate=f"<b>{row['role']}</b><br>Utilization: {row['mean_u']*100:.1f}%<br>n={row['n']}<extra></extra>"
            ))

        fig2 = g3_style(fig2, 380)
        fig2.update_layout(
            title=dict(text="Budtenders redeem the most",
                font=dict(color="#2E342A", size=14)),
            yaxis_title="Mean Utilization (%)",
            showlegend=False,
            yaxis_range=[0, g3_by_role['mean_u'].max()*130] if len(g3_by_role) else [0,100]
        )
        st.plotly_chart(fig2, use_container_width=True, key="g3_role_util")

    # ── ROW 2: Points by Role (3 charts) ─────────────────────
    st.markdown('<div class="chart-title">Points by User Role</div>', unsafe_allow_html=True)
    g3_desc(
        "Three side-by-side views per role — average points earned, average points spent, and average utilization "
        "rate. Comparing the earned and spent charts reveals which roles accumulate points but rarely cash them in. "
        "The table underneath lists the exact figures plus the activation % (the share of users in each role who "
        "earned at least one point)."
    )

    g3_seg_role = g3_chart_uf.groupby('bta_user_role').agg(
        total_users  = ('user_key',                      'count'),
        active_users = ('lifetime_points_earned_clean',  lambda x: (x>0).sum()),
        avg_earned   = ('lifetime_points_earned_clean',  'mean'),
        avg_spent    = ('points_spent',                  'mean'),
        avg_util     = ('utilization_rate',              'mean'),
    ).reset_index().dropna(subset=['bta_user_role'])
    g3_seg_role['util_%']       = (g3_seg_role['avg_util'] * 100).round(2)
    g3_seg_role['activation_%'] = (g3_seg_role['active_users'] / g3_seg_role['total_users'] * 100).round(1)

    c1,c2,c3 = st.columns(3)
    for col, y_col, title, fmt in [
        (c1, 'avg_earned', 'Avg Points Earned per User', lambda v: f"{v:,.0f}"),
        (c2, 'avg_spent',  'Avg Points Spent per User',  lambda v: f"{v:,.0f}"),
        (c3, 'util_%',     'Avg Utilization Rate (%)',   lambda v: f"{v:.1f}%"),
    ]:
        with col:
            fig = go.Figure()
            for _, row in g3_seg_role.iterrows():
                fig.add_trace(go.Bar(
                    x=[row['bta_user_role']], y=[row[y_col]],
                    name=row['bta_user_role'],
                    marker=dict(color=G3_ROLE_COLORS.get(row['bta_user_role'], '#8E9E96'),
                                line=dict(width=0)),
                    text=[fmt(row[y_col])], textposition='outside',
                    textfont=dict(color="#2E342A", size=11),
                    hovertemplate=f"<b>{row['bta_user_role']}</b><br>{title}: {fmt(row[y_col])}<extra></extra>"
                ))
            fig = g3_style(fig, 340)
            fig.update_layout(
                title=dict(text=title, font=dict(color="#2E342A", size=13)),
                showlegend=False,
                yaxis_range=[0, 100] if y_col == 'util_%' else None
            )
            st.plotly_chart(fig, use_container_width=True, key=f"g3_{y_col}_bar")

    # Role summary table
    g3_disp_role = g3_seg_role[['bta_user_role','total_users','active_users',
        'activation_%','avg_earned','avg_spent','util_%']].copy()
    g3_disp_role.columns = ['Role','Total Users','Active Users','Activation %',
        'Avg Earned','Avg Spent','Util %']
    g3_disp_role['Avg Earned'] = g3_disp_role['Avg Earned'].apply(lambda x: f"{x:,.0f}")
    g3_disp_role['Avg Spent']  = g3_disp_role['Avg Spent'].apply(lambda x: f"{x:,.0f}")
    st.dataframe(g3_disp_role, use_container_width=True, hide_index=True)

    # ── ROW 3: Age group ──────────────────────────────────────
    st.markdown('<div class="chart-title">Points by Age Group</div>', unsafe_allow_html=True)
    g3_desc(
        "On the left, grouped horizontal bars compare average points earned (blue) against average points spent "
        "(orange) for each age band — a wide gap between the two means that age group banks points without "
        "redeeming them. On the right, the average utilization rate per age band: bars above the overall average "
        "are highlighted in purple and those below in muted green, so you can quickly see which ages actually "
        "convert points into rewards."
    )

    g3_seg_age = g3_chart_uf.groupby('age_group', observed=True).agg(
        n=('user_key','count'), avg_earned=('lifetime_points_earned_clean','mean'),
        avg_spent=('points_spent','mean'), avg_util=('utilization_rate','mean')
    ).reset_index()
    g3_seg_age['util_%']    = (g3_seg_age['avg_util'] * 100).round(2)
    g3_seg_age['age_group'] = g3_seg_age['age_group'].astype(str)

    col_a, col_b = st.columns(2)
    with col_a:
        fig = go.Figure()
        fig.add_trace(go.Bar(
            y=g3_seg_age['age_group'], x=g3_seg_age['avg_earned'],
            name='Earned', orientation='h',
            marker=dict(color='#2E86AB', line=dict(width=0)),
            text=g3_seg_age['avg_earned'].apply(lambda x: f"{x:,.0f}"),
            textposition='outside', textfont=dict(color="#2E342A", size=10),
            hovertemplate="Age: %{y}<br>Avg Earned: %{x:,.0f}<extra></extra>"
        ))
        fig.add_trace(go.Bar(
            y=g3_seg_age['age_group'], x=g3_seg_age['avg_spent'],
            name='Spent', orientation='h',
            marker=dict(color='#E07A30', line=dict(width=0)),
            text=g3_seg_age['avg_spent'].apply(lambda x: f"{x:,.0f}"),
            textposition='outside', textfont=dict(color="#2E342A", size=10),
            hovertemplate="Age: %{y}<br>Avg Spent: %{x:,.0f}<extra></extra>"
        ))
        fig = g3_style(fig, 380)
        fig.update_layout(
            barmode='group',
            title=dict(text="Points Earned vs Spent by Age Group",
                font=dict(color="#2E342A", size=14)),
            xaxis_title="Average Points", yaxis_title="Age Group",
            legend=dict(orientation='h', y=1.12, font=dict(color="#2E342A"))
        )
        st.plotly_chart(fig, use_container_width=True, key="g3_age_bar")

    with col_b:
        age_colors = ['#8E44AD' if v >= g3_seg_age['util_%'].mean()
                      else '#B8CBAA' for v in g3_seg_age['util_%']]
        fig = go.Figure()
        for i, row in g3_seg_age.iterrows():
            fig.add_trace(go.Bar(
                x=[row['age_group']], y=[row['util_%']],
                marker=dict(color='#8E44AD' if row['util_%'] >= g3_seg_age['util_%'].mean()
                            else '#B8CBAA', line=dict(width=0)),
                text=[f"{row['util_%']:.1f}%"], textposition='outside',
                textfont=dict(color="#2E342A", size=11),
                name=row['age_group'],
                hovertemplate=f"Age: {row['age_group']}<br>Utilization: {row['util_%']:.1f}%<extra></extra>"
            ))
        fig = g3_style(fig, 380)
        fig.update_layout(
            title=dict(text="Utilization Rate by Age Group",
                font=dict(color="#2E342A", size=14)),
            xaxis_title="Age Group", yaxis_title="Utilization (%)",
            showlegend=False,
            yaxis_range=[0, g3_seg_age['util_%'].max()*1.35] if len(g3_seg_age) else [0,10]
        )
        st.plotly_chart(fig, use_container_width=True, key="g3_age_util")

    # Age summary table
    g3_disp_age = g3_seg_age[['age_group','n','avg_earned','avg_spent','util_%']].copy()
    g3_disp_age.columns = ['Age Group','Users','Avg Earned','Avg Spent','Util %']
    g3_disp_age['Avg Earned'] = g3_disp_age['Avg Earned'].apply(lambda x: f"{x:,.0f}")
    g3_disp_age['Avg Spent']  = g3_disp_age['Avg Spent'].apply(lambda x: f"{x:,.0f}")
    st.dataframe(g3_disp_age, use_container_width=True, hide_index=True)

    # ── ROW 4: Preferred Rewards ──────────────────────────────
    st.markdown('<div class="chart-title">Preferred Rewards</div>', unsafe_allow_html=True)
    g3_desc(
        "On the left, a heatmap of how many users in each role selected each reward — darker cells are more popular, "
        "so reading across a row shows which roles favour a reward and reading down a column profiles a single role. "
        "On the right, a donut of the overall reward preference across all users in the current filter, where bigger "
        "slices are the rewards the membership wants most regardless of role."
    )

    g3_reward_rows = []
    for _, row in g3_chart_uf.iterrows():
        for reward in row.get('preferred_rewards_parsed', []):
            g3_reward_rows.append({'bta_user_role': row['bta_user_role'], 'reward': reward})
    g3_df_rewards = pd.DataFrame(g3_reward_rows)

    if not g3_df_rewards.empty:
        g3_pivot      = g3_df_rewards.groupby(['reward','bta_user_role']).size().unstack(fill_value=0)
        g3_totals     = g3_df_rewards['reward'].value_counts().reset_index()
        g3_totals.columns = ['reward','count']
        g3_totals['reward_label'] = g3_totals['reward'].str.replace('_',' ').str.title()

        col_a, col_b = st.columns(2)
        with col_a:
            # Clean, Title-Case reward names so the y-axis matches the pie chart
            g3_pivot_disp = g3_pivot.copy()
            g3_pivot_disp.index = (g3_pivot_disp.index.astype(str)
                                   .str.replace('_', ' ').str.title())

            fig = px.imshow(g3_pivot_disp, color_continuous_scale='YlGnBu',
                title="Reward Popularity by Role", text_auto=True,
                labels=dict(x="Role", y="Reward", color="Users"), aspect="auto")
            fig.update_layout(
                paper_bgcolor="#F1F2EF", plot_bgcolor="#F1F2EF",
                font=dict(color="#2E342A", size=12), height=420,
                title_font=dict(color="#2E342A", size=14),
                margin=dict(l=10, r=20, t=55, b=45),
                coloraxis_colorbar=dict(
                    title=dict(text="Users", font=dict(color="#2E342A")),
                    tickfont=dict(color="#2E342A"))
            )
            # automargin lets the long reward names expand the axis instead of being clipped
            fig.update_xaxes(tickfont=dict(color="#2E342A", size=12),
                             title_font=dict(color="#2E342A"))
            fig.update_yaxes(tickfont=dict(color="#2E342A", size=12),
                             title_font=dict(color="#2E342A"), automargin=True)
            fig.update_traces(
                textfont=dict(size=11),
                hovertemplate="Reward: %{y}<br>Role: %{x}<br>Users: %{z}<extra></extra>")
            st.plotly_chart(fig, use_container_width=True, key="g3_reward_heat")

        with col_b:
            fig = go.Figure(go.Pie(
                labels=g3_totals['reward_label'], values=g3_totals['count'],
                hole=0.5,
                marker=dict(colors=["#556B5D","#6E8575","#8A9B8A","#B8CBAA",
                                    "#D6DDD1","#E07A30","#2E86AB","#8E44AD"]),
                textfont=dict(color="#2E342A"),
                hovertemplate="<b>%{label}</b><br>%{value:,} (%{percent})<extra></extra>"
            ))
            fig.update_layout(
                paper_bgcolor="#F1F2EF", plot_bgcolor="#F1F2EF",
                font=dict(color="#2E342A", size=12), height=420,
                title=dict(text="Overall Reward Preference", font=dict(color="#2E342A", size=14)),
                margin=dict(l=35, r=35, t=55, b=35),
                legend=dict(font=dict(color="#2E342A", size=11), orientation='v')
            )
            st.plotly_chart(fig, use_container_width=True, key="g3_reward_pie")

        st.dataframe(g3_totals[['reward_label','count']].rename(
            columns={'reward_label':'Reward','count':'Users'}),
            use_container_width=True, hide_index=True)
    else:
        st.info("No preferred rewards data found in the current filter.")

    # ── ROW 5: Activity Levels ────────────────────────────────
    st.markdown('<div class="chart-title">User Distribution by Points Activity Level</div>', unsafe_allow_html=True)
    g3_desc(
        "On the left, users grouped into lifetime-points buckets (No points → Very high 50K+), which reveals how the "
        "base splits between dormant accounts and power earners. On the right, a scatter where each dot is one active "
        "user, positioned by points earned (x) and points spent (y) and coloured by role: dots hugging the bottom "
        "earn a lot but spend little, while dots rising along the diagonal redeem in proportion to what they earn."
    )

    g3_cats = pd.cut(g3_chart_uf['lifetime_points_earned_clean'],
        bins=[-1,0,5000,20000,50000,float('inf')],
        labels=['No points','Low (1-5K)','Medium (5K-20K)','High (20K-50K)','Very high (50K+)'])
    g3_cat_counts = g3_cats.value_counts().sort_index().reset_index()
    g3_cat_counts.columns = ['Category','Count']
    g3_cat_palette = ['#8E9E96','#B8CBAA','#6E8575','#556B5D','#2E342A']

    col_a, col_b = st.columns(2)
    with col_a:
        fig = go.Figure()
        for i, row in g3_cat_counts.iterrows():
            fig.add_trace(go.Bar(
                x=[row['Category']], y=[row['Count']],
                marker=dict(color=g3_cat_palette[i % len(g3_cat_palette)], line=dict(width=0)),
                text=[str(row['Count'])], textposition='outside',
                textfont=dict(color="#2E342A", size=11),
                name=row['Category'],
                hovertemplate=f"<b>{row['Category']}</b><br>Users: {row['Count']}<extra></extra>"
            ))
        fig = g3_style(fig, 380)
        fig.update_layout(title=dict(text="Users by Points Activity Level",
            font=dict(color="#2E342A", size=14)),
            xaxis_tickangle=-20, showlegend=False)
        st.plotly_chart(fig, use_container_width=True, key="g3_activity_bar")

    with col_b:
        g3_active_sc = g3_chart_uf[g3_chart_uf['lifetime_points_earned_clean'] > 0].copy()
        g3_active_sc['role'] = g3_active_sc['bta_user_role'].fillna('Unknown')

        fig = go.Figure()
        for role in g3_active_sc['role'].unique():
            sub = g3_active_sc[g3_active_sc['role'] == role]
            fig.add_trace(go.Scatter(
                x=sub['lifetime_points_earned_clean'], y=sub['points_spent'],
                mode='markers', name=role,
                marker=dict(color=G3_ROLE_COLORS.get(role, '#8E9E96'),
                            size=7, opacity=0.65,
                            line=dict(color='white', width=0.5)),
                hovertemplate=f"<b>{role}</b><br>Earned: %{{x:,.0f}}<br>Spent: %{{y:,.0f}}<extra></extra>"
            ))
        fig = g3_style(fig, 380)
        fig.update_layout(title=dict(text="Points Earned vs Points Spent by Role",
            font=dict(color="#2E342A", size=14)),
            xaxis_title="Points Earned", yaxis_title="Points Spent",
            legend=dict(orientation='h', y=-0.25, font=dict(color="#2E342A", size=11)))
        st.plotly_chart(fig, use_container_width=True, key="g3_scatter")

    st.dataframe(g3_cat_counts, use_container_width=True, hide_index=True)

    # ── ROW 6: Full Dataset ───────────────────────────────────
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown('<div class="chart-title">Full Dataset</div>', unsafe_allow_html=True)

    # Preferred order — keep whichever columns actually exist
    g3_pref_cols = [
        'bta_user_role', 'province', 'age_years', 'age_group',
        'lifetime_points_earned', 'lifetime_points_earned_clean',
        'current_points_balance', 'current_points_balance_clean',
        'points_spent', 'utilization_rate', 'points_flag_implausible',
        'preferred_rewards',
    ]
    g3_full_cols = [c for c in g3_pref_cols if c in g3_chart_uf.columns]
    # Append any remaining columns not already listed (excludes user_key and the parsed-list helper)
    g3_full_cols += [c for c in g3_chart_uf.columns
                     if c not in g3_full_cols and c not in ('preferred_rewards_parsed', 'user_key')]

    # Privacy: never display personal contact info (phone / mobile / email / address).
    g3_contact_keywords = ('phone', 'mobile', 'cell', 'email', 'e-mail', 'address', 'contact')
    g3_full_cols = [c for c in g3_full_cols
                    if not any(kw in c.lower() for kw in g3_contact_keywords)]

    g3_full_df = g3_chart_uf[g3_full_cols].copy()
    if 'utilization_rate' in g3_full_df.columns:
        g3_full_df['utilization_rate'] = (g3_full_df['utilization_rate'] * 100).round(2)

    st.caption(f"{len(g3_full_df):,} rows × {len(g3_full_df.columns)} columns (current filter)")
    st.dataframe(g3_full_df, use_container_width=True, hide_index=True)

    st.download_button(
        label="⬇️ Download full dataset (CSV)",
        data=g3_full_df.to_csv(index=False).encode('utf-8'),
        file_name="g3_points_utilization_full.csv",
        mime="text/csv",
        key="g3_full_download",
    )

# =============================================================
# G4 · VARIANCE ANALYSIS
# TEAM MEMBERS: THARINDU, MATEO, KEENE
#
# ⬇️ G4 PASTE YOUR FINAL CHART CODE BELOW THIS SECTION ⬇️
# =============================================================
elif page == "G4":

    # =============================================================
    # G4 · VARIANCE ANALYSIS — FINAL STREAMLIT SECTION
    # Replace everything from: elif page == "G4":
    # up to right before: elif page == "G5":
    # =============================================================

    st.markdown("""
    <style>
    .g4-title {
        font-family: Georgia, serif !important;
        font-size: 46px;
        font-weight: 800;
        color: #e26655;
        margin-bottom: 8px;
    }

    .g4-subtitle {
        font-size: 15px;
        font-weight: 800;
        color: #000000;
        margin-bottom: 28px;
    }

    .g4-section-title {
        font-family: Georgia, serif !important;
        font-size: 24px;
        font-weight: 900;
        color: #e26655;
        margin-top: 30px;
        margin-bottom: 14px;
        text-transform: uppercase;
    }

    .g4-kpi-card {
        background: #fffdf7;
        border-radius: 18px;
        padding: 22px 24px;
        min-height: 150px;
        border: 1px solid rgba(0,0,0,0.08);
        box-shadow: 0px 8px 22px rgba(0,0,0,0.10);
        position: relative;
        overflow: hidden;
    }

    .g4-kpi-card::before {
        content: "";
        position: absolute;
        top: 0;
        left: 0;
        height: 6px;
        width: 100%;
        background: var(--card-color);
    }

    .g4-kpi-flex {
        display: flex;
        align-items: center;
        gap: 18px;
    }

    .g4-icon {
        width: 56px;
        height: 56px;
        min-width: 56px;
        border-radius: 16px;
        background: var(--card-color);
        color: #ffffff;
        display: flex;
        justify-content: center;
        align-items: center;
        font-size: 28px;
        font-weight: 900;
    }

    .g4-kpi-label {
        font-size: 13px;
        font-weight: 900;
        text-transform: uppercase;
        color: #000000;
        letter-spacing: .5px;
    }

    .g4-kpi-value {
        font-size: 40px;
        font-weight: 900;
        color: var(--card-color);
        margin-top: 8px;
        line-height: 1.1;
    }

    .g4-kpi-note {
        color: #000000;
        font-size: 13px;
        font-weight: 700;
        margin-top: 8px;
    }

    .g4-panel {
        background: #fffdf7;
        border-radius: 18px;
        padding: 22px;
        border: 1px solid rgba(0,0,0,0.08);
        box-shadow: 0px 8px 22px rgba(0,0,0,0.10);
    }

    .g4-final-note {
        background: #fffdf7;
        border: 1px solid #b0bfa5;
        border-left: 8px solid #b0bfa5;
        border-radius: 16px;
        padding: 22px 24px;
        margin-top: 24px;
        color: #000000;
        font-size: 15px;
        line-height: 1.6;
        box-shadow: 0px 8px 22px rgba(0,0,0,0.08);
    }

    .g4-final-note b {
        color: #4f6f46;
        font-size: 16px;
    }

    .g4-table-wrap {
        background: #fffdf7;
        border-radius: 18px;
        padding: 12px;
        border: 1px solid rgba(0,0,0,0.08);
        box-shadow: 0px 8px 22px rgba(0,0,0,0.10);
    }

    div[data-testid="stDataFrame"] {
        border-radius: 12px !important;
        overflow: hidden !important;
    }
    </style>
    """, unsafe_allow_html=True)

    st.markdown('<div class="g4-title">G4 · Variance Analysis</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="g4-subtitle">Straight-lining detection using variance across linear-scale survey answers.</div>',
        unsafe_allow_html=True
    )

    # --------------------------------------------------
    # Build G4 variance table
    # --------------------------------------------------

    g4 = fct.copy()

    if "question_type" not in g4.columns:
        g4 = g4.merge(
            questions[["question_key", "question_type"]],
            on="question_key",
            how="left"
        )

    g4_scale = g4[g4["question_type"] == "linear_scale"].copy()

    g4_scale["answer_numeric"] = pd.to_numeric(
        g4_scale["answer"],
        errors="coerce"
    )

    g4_scale = g4_scale.dropna(subset=["answer_numeric"])

    survey_col = "survey_key" if "survey_key" in g4_scale.columns else "survey_id"

    g4_blocks = (
        g4_scale
        .groupby(["user_key", survey_col])
        .agg(
            scale_answers=("answer_numeric", "count"),
            variance=("answer_numeric", "var"),
            mean_answer=("answer_numeric", "mean"),
            min_answer=("answer_numeric", "min"),
            max_answer=("answer_numeric", "max")
        )
        .reset_index()
    )

    g4_eligible = g4_blocks[g4_blocks["scale_answers"] >= 3].copy()
    g4_eligible["variance"] = g4_eligible["variance"].fillna(0)
    g4_eligible["straight_line_flag"] = g4_eligible["variance"] == 0

    if "bta_user_role" in users.columns:
        g4_eligible = g4_eligible.merge(
            users[["user_key", "bta_user_role"]],
            on="user_key",
            how="left"
        )
    else:
        g4_eligible["bta_user_role"] = "Unknown"

    if selected_role != "All":
        g4_eligible = g4_eligible[
            g4_eligible["bta_user_role"].astype(str) == selected_role
        ]

    flagged_blocks = int(g4_eligible["straight_line_flag"].sum())
    total_blocks = len(g4_eligible)
    flag_rate = (flagged_blocks / total_blocks * 100) if total_blocks > 0 else 0
    avg_variance = g4_eligible["variance"].mean() if total_blocks > 0 else 0

    # --------------------------------------------------
    # KPI Cards
    # --------------------------------------------------

    k1, k2, k3, k4 = st.columns(4)

    with k1:
        st.markdown(f"""
        <div class="g4-kpi-card" style="--card-color:#e26655;">
            <div class="g4-kpi-flex">
                <div class="g4-icon">👥</div>
                <div>
                    <div class="g4-kpi-label">Eligible Blocks</div>
                    <div class="g4-kpi-value">{total_blocks}</div>
                    <div class="g4-kpi-note">3+ scale answers</div>
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

    with k2:
        st.markdown(f"""
        <div class="g4-kpi-card" style="--card-color:#e6866f;">
            <div class="g4-kpi-flex">
                <div class="g4-icon">🚩</div>
                <div>
                    <div class="g4-kpi-label">Flagged Blocks</div>
                    <div class="g4-kpi-value">{flagged_blocks}</div>
                    <div class="g4-kpi-note">Zero variance blocks</div>
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

    with k3:
        st.markdown(f"""
        <div class="g4-kpi-card" style="--card-color:#f4b35e;">
            <div class="g4-kpi-flex">
                <div class="g4-icon">%</div>
                <div>
                    <div class="g4-kpi-label">Flag Rate</div>
                    <div class="g4-kpi-value">{flag_rate:.2f}%</div>
                    <div class="g4-kpi-note">Flagged ÷ eligible</div>
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

    with k4:
        st.markdown(f"""
        <div class="g4-kpi-card" style="--card-color:#b0bfa5;">
            <div class="g4-kpi-flex">
                <div class="g4-icon">↗</div>
                <div>
                    <div class="g4-kpi-label">Average Variance</div>
                    <div class="g4-kpi-value">{avg_variance:.2f}</div>
                    <div class="g4-kpi-note">Across eligible blocks</div>
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

    # --------------------------------------------------
    # Histogram
    # --------------------------------------------------

    st.markdown('<div class="g4-section-title">Variance Distribution Histogram</div>', unsafe_allow_html=True)

    g4_eligible["variance_type"] = np.where(
        g4_eligible["variance"] == 0,
        "Zero Variance - Flagged",
        "Positive Variance"
    )

    fig_hist = px.histogram(
        g4_eligible,
        x="variance",
        color="variance_type",
        nbins=35,
        title="Variance Distribution Across Eligible Survey Blocks",
        labels={
            "variance": "Variance",
            "count": "Number of Survey Blocks",
            "variance_type": "Variance Type"
        },
        color_discrete_map={
            "Zero Variance - Flagged": "#e26655",
            "Positive Variance": "#b0bfa5"
        }
    )

    fig_hist.update_layout(
        paper_bgcolor="#fffdf7",
        plot_bgcolor="#fffdf7",
        font=dict(color="#000000", size=14),
        title_font=dict(color="#000000", size=20),
        height=440,
        bargap=0.08,
        margin=dict(l=60, r=45, t=70, b=60),
        xaxis=dict(
            title="Variance",
            title_font=dict(color="#000000", size=14),
            tickfont=dict(color="#000000", size=12),
            gridcolor="rgba(0,0,0,0.10)",
            zerolinecolor="#000000",
            linecolor="#000000"
        ),
        yaxis=dict(
            title="Number of Survey Blocks",
            title_font=dict(color="#000000", size=14),
            tickfont=dict(color="#000000", size=12),
            gridcolor="rgba(0,0,0,0.10)",
            zerolinecolor="#000000",
            linecolor="#000000"
        ),
        legend=dict(
            font=dict(color="#000000", size=13),
            title_font=dict(color="#000000", size=13)
        )
    )

    fig_hist.add_annotation(
        x=0,
        y=max(flagged_blocks, 1),
        text=f"{flagged_blocks} flagged blocks<br>{flag_rate:.2f}% flag rate",
        showarrow=True,
        arrowhead=2,
        ax=75,
        ay=-65,
        font=dict(color="#000000", size=13),
        bgcolor="#FFFFFF",
        bordercolor="#e26655",
        borderwidth=1
    )

    st.markdown('<div class="g4-panel">', unsafe_allow_html=True)
    st.plotly_chart(fig_hist, use_container_width=True)
    st.markdown('</div>', unsafe_allow_html=True)

    # --------------------------------------------------
    # Role Chart + Interpretation
    # --------------------------------------------------

    left, right = st.columns([1, 1])

    with left:
        st.markdown('<div class="g4-section-title">Straight-Line Flag Rate by Role</div>', unsafe_allow_html=True)

        role_summary = (
            g4_eligible
            .groupby("bta_user_role")
            .agg(
                eligible_blocks=("straight_line_flag", "count"),
                flagged_blocks=("straight_line_flag", "sum")
            )
            .reset_index()
        )

        role_summary["flag_rate"] = (
            role_summary["flagged_blocks"] /
            role_summary["eligible_blocks"] * 100
        )

        fig_role = px.bar(
            role_summary,
            x="bta_user_role",
            y="flag_rate",
            text=role_summary.apply(
                lambda r: f"{r['flag_rate']:.2f}%<br>({int(r['flagged_blocks'])}/{int(r['eligible_blocks'])})",
                axis=1
            ),
            labels={
                "bta_user_role": "User Role",
                "flag_rate": "Flag Rate (%)"
            },
            color="bta_user_role",
            color_discrete_sequence=["#e26655", "#e6866f", "#b0bfa5"]
        )

        fig_role.update_traces(
            textposition="outside",
            marker_line_width=0,
            opacity=0.95
        )

        fig_role.update_layout(
            paper_bgcolor="#fffdf7",
            plot_bgcolor="#fffdf7",
            font=dict(color="#000000", size=14),
            height=395,
            showlegend=False,
            margin=dict(l=60, r=35, t=35, b=60),
            xaxis=dict(
                title="User Role",
                title_font=dict(color="#000000"),
                tickfont=dict(color="#000000"),
                linecolor="#000000"
            ),
            yaxis=dict(
                title="Flag Rate (%)",
                title_font=dict(color="#000000"),
                tickfont=dict(color="#000000"),
                gridcolor="rgba(0,0,0,0.10)",
                linecolor="#000000"
            )
        )

        st.markdown('<div class="g4-panel">', unsafe_allow_html=True)
        st.plotly_chart(fig_role, use_container_width=True)
        st.markdown('</div>', unsafe_allow_html=True)

    with right:
        st.markdown('<div class="g4-section-title">G4 Interpretation</div>', unsafe_allow_html=True)

        # Clean Streamlit version: no raw HTML is printed on the page.
        with st.container(border=True):
            st.markdown("**What this dashboard shows:**")
            st.write(
                "This section checks for possible straight-lining behavior. "
                "Straight-lining happens when someone keeps choosing the same answer "
                "across multiple scale questions."
            )

            st.markdown("**Main result:**")
            st.success(
                f"🚩 **{flagged_blocks}** out of **{total_blocks}** eligible survey blocks were flagged. "
                f"That means the flag rate is **{flag_rate:.2f}%**."
            )

            st.markdown("**Important note:**")
            st.write(
                "A flag does not mean the response is wrong. It only means the pattern "
                "should be reviewed because the answers had zero variance."
            )

    # --------------------------------------------------
    # Flagged Table
    # --------------------------------------------------

    st.markdown('<div class="g4-section-title">Flagged Survey Blocks for Review</div>', unsafe_allow_html=True)

    flagged_table = g4_eligible[g4_eligible["straight_line_flag"] == True].copy()

    display_cols = [
        "user_key",
        survey_col,
        "bta_user_role",
        "scale_answers",
        "variance",
        "mean_answer",
        "min_answer",
        "max_answer",
        "straight_line_flag"
    ]

    flagged_table = flagged_table[display_cols].copy()

    flagged_table["straight_line_flag"] = flagged_table["straight_line_flag"].map({
        True: "Yes",
        False: "No"
    })

    st.markdown('<div class="g4-table-wrap">', unsafe_allow_html=True)
    st.dataframe(
        flagged_table,
        use_container_width=True,
        hide_index=True,
        height=300
    )
    st.markdown('</div>', unsafe_allow_html=True)

    # --------------------------------------------------
    # Final Note
    # --------------------------------------------------

    st.markdown(f"""
    <div class="g4-final-note">
        <b>GREEN — Feasible for Straight-Lining Detection</b><br>
        The data supports variance-based detection on linear-scale questions.
        Blocks with fewer than 3 scale answers are excluded to reduce false positives.
        Flags are treated as review indicators, not accusations.<br><br>
        <b>Owners:</b> Tharindu · Mateo · Keene | Dashboard B
    </div>
    """, unsafe_allow_html=True)
# =============================================================
# G5 · POINTS ECONOMY
# TEAM MEMBERS: MARI ERIKO, BERNIE, JULIE
#
# ⬇️ G5 PASTE YOUR FINAL CHART CODE BELOW THIS SECTION ⬇️
# =============================================================

elif page == "G5":

    # ── BTA brand colours — applied throughout all G5 charts and cards ──
    # Source: BudtendersAssocColours.png provided by BTA
    G5_RED    = "#e26655"   # Primary red/coral — flagged items, attention needed
    G5_ORANGE = "#e38f55"   # Orange — burst events, moderate alerts
    G5_AMBER  = "#f4b35e"   # Amber — threshold lines, warning indicators
    G5_SAGE   = "#b0bfa5"   # Sage green — clean/ok items, coverage
    G5_BLUE   = "#838db9"   # Blue/lavender — neutral info metrics
    G5_GRAY   = "#9e9e9e"   # Gray — below-threshold items

    # Colour mapping for farming suspicion tiers
    # Used consistently across bar charts, donuts, and scatter plots
    SUSP_COLORS = {
        "High":                 G5_RED,     # most suspicious
        "Medium":               G5_AMBER,   # moderate concern
        "Low — likely genuine": G5_SAGE,    # likely genuine catch-up
        "Below threshold":      G5_GRAY,    # did not meet burst threshold
    }

    # ── Shared chart layout function ──────────────────────────
    # Returns a standard Plotly layout dict with BTA cream background,
    # warm gridlines, and consistent font colour. Called for every chart
    # to ensure visual consistency across all tabs.
    def g5_layout(title="", height=300):
        return dict(
            paper_bgcolor="#fdf8f3", plot_bgcolor="#fdf8f3",
            font=dict(color="#3d3530", size=12),
            margin=dict(l=20, r=40, t=40, b=30),
            height=height, showlegend=False,
            xaxis=dict(gridcolor="#e6d5c3", zerolinecolor="#e6d5c3"),
            yaxis=dict(gridcolor="#e6d5c3", zerolinecolor="#e6d5c3"),
            title=dict(text=title, font_size=13, font_color="#3d3530") if title else {}
        )

    # ── Page header ───────────────────────────────────────────
    st.markdown('<div class="main-title">G5 · Lazy-Text + Reward Farming</div>',
                unsafe_allow_html=True)
    st.markdown('<div class="main-subtitle">Reward-farming checks and lazy-text response quality indicators.</div>',
                unsafe_allow_html=True)

    # ── Session state initialisation ─────────────────────────
    # Streamlit re-runs the entire script on every interaction.
    # Session state preserves values across re-runs so pipeline
    # progress and loaded data are not lost when the user clicks a button.
    if "g5_step" not in st.session_state: st.session_state.g5_step = 1  # current pipeline step (1-5)
    if "g5_lt"   not in st.session_state: st.session_state.g5_lt   = None  # lazy-text dataframe
    if "g5_fm"   not in st.session_state: st.session_state.g5_fm   = None  # farming dataframe

    # ── Google Drive file paths ───────────────────────────────
    # DRIVE_BASE is defined by the team leader in app.py and points to
    # the root BTAxDashboardxAcsenda folder on Google Drive.
    # All G5 files live in the G5 subfolder.
    G5_FOLDER   = os.path.join(DRIVE_BASE, "G5")              # BTAxDashboardxAcsenda/G5/
    MAIN_FOLDER = os.path.join(DRIVE_BASE, "Main CSV Files")   # BTAxDashboardxAcsenda/Main CSV Files/
    LAZY_PATH   = os.path.join(G5_FOLDER, "g5_lazy_text_flags.csv")   # generated output file
    FARM_PATH   = os.path.join(G5_FOLDER, "g5_farming_flags.csv")     # generated output file

    # ── Locate raw BTA input files ────────────────────────────
    # Uses glob to handle the timestamp suffix in filenames (e.g. _202605121902.csv).
    # Searches the G5 folder first, then falls back to Main CSV Files.
    import glob
    def find_file(*folders, pattern):
        """Search multiple folders for a file matching a glob pattern.
        Returns the first match found, or None if not found anywhere."""
        for folder in folders:
            matches = glob.glob(os.path.join(folder, pattern))
            if matches:
                return matches[0]
        return None

    FCT_PATH = find_file(G5_FOLDER, MAIN_FOLDER, pattern="fct_survey_responses*.csv")
    QS_PATH  = find_file(G5_FOLDER, MAIN_FOLDER, pattern="dim_questions*.csv")

    # ── Check availability of files ───────────────────────────
    # g5_files_exist: True if the feature files from a previous run are already in Drive
    # raw_files_exist: True if the raw BTA input files are accessible
    g5_files_exist  = os.path.exists(LAZY_PATH) and os.path.exists(FARM_PATH)
    raw_files_exist = FCT_PATH is not None and QS_PATH is not None

    # ── Auto-advance step ─────────────────────────────────────
    # Only auto-load existing feature files if step is still at 1
    # (i.e. user hasn't started a fresh run)
    if "g5_rerun" not in st.session_state:
        st.session_state.g5_rerun = False

    if g5_files_exist and st.session_state.g5_step == 1 and not st.session_state.g5_rerun:
        # Feature files exist — offer choice: use existing or re-run
        st.session_state.g5_step = 5
    elif raw_files_exist and st.session_state.g5_step < 2:
        st.session_state.g5_step = 2

    g5_step = st.session_state.g5_step


    # ── Pipeline expander (Steps 1-4) ────────────────────────
    # ── Toggle button for pipeline steps ─────────────────────
    if "g5_show_pipeline" not in st.session_state:
        st.session_state.g5_show_pipeline = st.session_state.g5_step < 5

    is_complete = st.session_state.g5_step >= 5
    toggle_label = (
        "✔  Data pipeline complete — click to hide steps"
        if (is_complete and st.session_state.g5_show_pipeline)
        else "▼  Data pipeline complete — click to review steps"
        if is_complete
        else f"▼  Data pipeline — Step {st.session_state.g5_step} of 4 — click to expand"
        if not st.session_state.g5_show_pipeline
        else f"▲  Data pipeline — Step {st.session_state.g5_step} of 4 — click to collapse"
    )

    st.markdown("""
    <style>
    button[kind="secondary"].g5-toggle {
        background: #fdf8f3 !important;
        border: 1px solid #e6d5c3 !important;
        border-radius: 10px !important;
        color: #3d3530 !important;
        font-size: 14px !important;
        font-weight: 600 !important;
        width: 100% !important;
        text-align: left !important;
        padding: 12px 16px !important;
    }
    </style>
    """, unsafe_allow_html=True)

    if st.button(toggle_label, key="g5_toggle_pipeline", use_container_width=True):
        st.session_state.g5_show_pipeline = not st.session_state.g5_show_pipeline
        st.rerun()

    if st.session_state.g5_show_pipeline:

        # ── Re-run option when files already exist ────────────────
        if g5_files_exist and st.session_state.g5_step >= 5:
            col_info, col_btn = st.columns([3,1])
            with col_info:
                st.markdown(
                    f"<div style='font-size:12px;color:#7a6f68;padding:8px 0'>"
                    f"✅ Feature files found in Google Drive from a previous run. "
                    f"The dashboard is already populated with existing results.</div>",
                    unsafe_allow_html=True
                )
            with col_btn:
                if st.button("🔄 Re-run detectors", use_container_width=True, key="g5_rerun_btn"):
                    st.session_state.g5_rerun  = True
                    st.session_state.g5_step   = 2
                    st.session_state.g5_lt     = None
                    st.session_state.g5_fm     = None
                    st.rerun()

        # ── Progress indicators ───────────────────────────────────
        st.markdown("<br>", unsafe_allow_html=True)
        p1,p2,p3,p4,p5 = st.columns(5)
        step_labels = [
            ("1","Load from Drive",       G5_RED),
            ("2","Run Detectors",         G5_ORANGE),
            ("3","Save to Drive",         G5_AMBER),
            ("4","Load Feature Files",    G5_BLUE),
            ("5","View Dashboard",        G5_SAGE),
        ]
        for col, (num, label, color) in zip([p1,p2,p3,p4,p5], step_labels):
            s       = int(num)
            bg      = color if s <= g5_step else "#e0d5cc"
            txt     = "white" if s <= g5_step else "#9e9088"
            done    = "✓ " if s < g5_step else ""
            bold    = "700" if s == g5_step else "400"
            col.markdown(
                f"<div style='text-align:center;padding:8px 4px;"
                f"border-bottom:3px solid {'transparent' if s!=g5_step else color}'>"
                f"<div style='background:{bg};color:{txt};border-radius:50%;"
                f"width:28px;height:28px;line-height:28px;font-weight:700;"
                f"font-size:13px;margin:0 auto 4px'>{num}</div>"
                f"<div style='font-size:11px;color:{'#3d3530' if s<=g5_step else '#9e9088'};"
                f"font-weight:{bold}'>{done}{label}</div></div>",
                unsafe_allow_html=True
            )
        st.markdown("<br>", unsafe_allow_html=True)

        # ════════════════════════════════════════════════════════
        # STEP 1 — UPLOAD RAW BTA FILES
        # ════════════════════════════════════════════════════════
        step1_style = (
            f"background:#fdf0ee;border:2px solid {G5_RED};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step == 1 else
            f"background:#eef2ec;border:2px solid {G5_SAGE};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step > 1 else
            "background:#f5f2ee;border:2px dashed #c8b8a8;border-radius:12px;padding:16px;margin-bottom:12px;opacity:0.6"
        )
        st.markdown(f"<div style='{step1_style}'>", unsafe_allow_html=True)
        st.markdown(
            f"<div style='font-weight:700;font-size:14px;color:{G5_RED};margin-bottom:6px'>"
            f"📄 STEP 1 — Load BTA Raw Data from Google Drive</div>"
            "<div style='font-size:12px;color:#7a6f68;margin-bottom:12px'>"
            "G5 reads the 2 raw BTA CSV files directly from the Google Drive folder.</div>",
            unsafe_allow_html=True
        )
        c1, c2 = st.columns(2)
        with c1:
            st.markdown("**fct_survey_responses_*.csv**")
            if FCT_PATH:
                st.success(f"✅ Found in Google Drive")
                st.caption(f"{os.path.basename(FCT_PATH)} · 15,385 rows")
            else:
                st.error(f"❌ Not found in {MAIN_FOLDER}")
        with c2:
            st.markdown("**dim_questions_*.csv**")
            if QS_PATH:
                st.success(f"✅ Found in Google Drive")
                st.caption(f"{os.path.basename(QS_PATH)} · 223 rows")
            else:
                st.error(f"❌ Not found in {MAIN_FOLDER}")
        if not raw_files_exist:
            st.error(f"❌ Raw BTA files not found in {MAIN_FOLDER} or {G5_FOLDER}")
        st.markdown("</div>", unsafe_allow_html=True)

        # ════════════════════════════════════════════════════════
        # STEP 2 — RUN DETECTORS
        # ════════════════════════════════════════════════════════
        step2_style = (
            f"background:#fdf5e6;border:2px solid {G5_ORANGE};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step == 2 else
            f"background:#eef2ec;border:2px solid {G5_SAGE};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step > 2 else
            "background:#f5f2ee;border:2px dashed #c8b8a8;border-radius:12px;padding:16px;margin-bottom:12px;opacity:0.6"
        )
        st.markdown(f"<div style='{step2_style}'>", unsafe_allow_html=True)
        st.markdown(
            f"<div style='font-weight:700;font-size:14px;color:{G5_ORANGE};margin-bottom:6px'>"
            f"⚙️ STEP 2 — Run G5 Detectors</div>"
            "<div style='font-size:12px;color:#7a6f68;margin-bottom:12px'>"
            "Click the button to run both detectors on BTA raw data from Google Drive.</div>",
            unsafe_allow_html=True
        )
        if g5_step > 2 and st.session_state.g5_lt is not None:
            lt = st.session_state.g5_lt
            fm = st.session_state.g5_fm
            st.success(
                f"✅ Detectors complete! "
                f"Lazy-text: {int(lt['lazy_text_flagged'].sum()):,} flagged from {len(lt):,} responses  |  "
                f"Farming: {int(fm['flagged_for_review'].sum())} flagged burst events"
            )
        elif raw_files_exist:
            if st.button("▶  Run Lazy-Text + Farming Detectors",
                         type="primary", use_container_width=True, key="g5_run_btn"):
                progress = st.progress(0, text="Starting...")
                status_text = st.empty()

                status_text.text("📂 Step 1/6 — Loading fct_survey_responses from Google Drive...")
                progress.progress(5, text="Loading raw survey responses (15,385 rows)...")
                fct = pd.read_csv(FCT_PATH)
                fct["created_at_clean"] = pd.to_datetime(
                    fct["created_at"], utc=True, errors="coerce")

                progress.progress(15, text="Loading dim_questions (223 rows)...")
                status_text.text("📂 Step 2/6 — Loading dim_questions from Google Drive...")
                qs = pd.read_csv(QS_PATH)

                progress.progress(25, text="Filtering Textarea responses...")
                status_text.text("\U0001f524 Step 3/6 \u2014 Running Lazy-Text Detector...")

                # ── DETECTOR (a): LAZY-TEXT ───────────────────────────────────────────
                # Step 1: Isolate Textarea (open-ended) responses only.
                # Join on question_key to dim_questions to find Textarea types,
                # or use fct.question_type directly if that column exists.
                textarea_qs = qs[qs["response_type"] == "Textarea"]["question_key"].tolist() \
                    if "response_type" in qs.columns else []
                lt = fct[fct["question_type"] == "Textarea"].copy() \
                    if "question_type" in fct.columns \
                    else fct[fct["question_key"].isin(textarea_qs)].copy()

                # Step 2: Clean answer text.
                # Write to answer_clean — NEVER overwrite the original answer column.
                lt["answer_clean"] = lt["answer"].astype(str).str.strip().str.strip("\"'")

                # Step 3: Compute character count — primary detection signal.
                lt["char_count"] = lt["answer_clean"].str.len()

                # Step 4: Compute repeat-character ratio.
                # Measures what fraction of the text is the most frequent single character.
                # Ratio of 1.0 means the entire answer is one repeated character (e.g. "aaaa").
                def rr(text):
                    t = str(text).strip()
                    if not t: return 1.0  # empty string = fully repeated
                    s = pd.Series(list(t.lower())).value_counts()
                    return s.iloc[0] / len(t)

                lt["repeat_ratio"] = lt["answer_clean"].apply(rr)

                # Step 5: Apply five heuristic detection rules.
                # All thresholds are tunable via sidebar sliders — nothing is hard-coded.
                lt["flag_too_short"]      = lt["char_count"] < 10
                # Flags responses under 10 characters — too short to be meaningful.

                lt["flag_repeat_char"]    = lt["repeat_ratio"] >= 0.7
                # Flags if 70%+ of the answer is a single repeated character.

                lt["flag_keyboard_smash"] = lt["answer_clean"].apply(
                    lambda v: bool(re.search(r"(.)\1{5,}", str(v))))
                # Flags 6+ consecutive identical characters anywhere in the text.

                lt["flag_whitespace_only"] = lt["answer_clean"].apply(
                    lambda v: not bool(re.search(r"[a-zA-Z0-9]", str(v))))
                # Flags responses with no letters or numbers at all.

                lt["flag_gibberish"] = lt["answer_clean"].apply(
                    lambda v: len(re.sub(r"\s","",str(v))) >= 10 and
                    sum(1 for c in str(v).lower() if c in "aeiou") /
                    max(len(re.sub(r"\s","",str(v))),1) < 0.05)
                # Flags longer text (10+ chars) with fewer than 5% vowels.
                # Real English has ~35-40% vowels; very low ratio = random keystrokes.

                # Step 6: Composite flag — True if ANY of the five rules triggered.
                flag_cols = ["flag_too_short","flag_repeat_char","flag_keyboard_smash",
                             "flag_whitespace_only","flag_gibberish"]
                lt["lazy_text_flagged"] = lt[flag_cols].any(axis=1)
                progress.progress(55, text="Lazy-text detector complete...")
                status_text.text("\U0001f69c Step 4/6 \u2014 Running Reward Farming Detector...")

                # Step 7: Build human-readable reason string listing all triggered flags.
                # Helps BTA analysts understand why each response was flagged.
                def build_reason(row):
                    r = []
                    if row["flag_too_short"]:       r.append(f"too_short({row['char_count']})")
                    if row["flag_repeat_char"]:     r.append(f"repeat_char({row['repeat_ratio']:.2f})")
                    if row["flag_keyboard_smash"]:  r.append("keyboard_smash")
                    if row["flag_whitespace_only"]: r.append("whitespace_only")
                    if row["flag_gibberish"]:       r.append("gibberish")
                    return "; ".join(r)

                lt["lazy_text_reason"] = lt.apply(build_reason, axis=1)
                lt["user_short"]       = lt["user_key"].str[:12] + "…"

                # ── DETECTOR (b): REWARD FARMING ─────────────────────────
                # Step 1: Sort all survey responses by user and timestamp.
                # This creates a chronological activity timeline per user,
                # which is required for the LAG pattern below.
                fs = fct.sort_values(["user_key","created_at_clean"]).copy()

                # Step 2: Compute the gap between each response and the previous one
                # for the same user. This replicates the SQL LAG() window function:
                #   LAG(created_at) OVER (PARTITION BY user_key ORDER BY created_at)
                # pandas shift(1) within a groupby gives us the previous row's timestamp.
                fs["prev_at"]  = fs.groupby("user_key")["created_at_clean"].shift(1)
                fs["gap_days"] = (
                    fs["created_at_clean"] - fs["prev_at"]
                ).dt.total_seconds() / 86400  # convert seconds to days

                # Step 3: Identify burst start points.
                # A burst start is any response where the user was inactive for
                # 60+ days before this submission. This is the "dormant" period
                # that precedes a potential reward-farming burst.
                bursts = fs[fs["gap_days"] >= 60].copy()

                # Step 4: For each burst start, count responses in the 24-hour window.
                # We look at all responses by the same user within 24 hours of returning.
                # A high count in a short window is the core farming signal.
                records = []
                for _, row in bursts.iterrows():
                    ukey = row["user_key"]
                    t0   = row["created_at_clean"]  # timestamp of burst start

                    # Count all responses by this user within the 24-hour burst window
                    win = fs[(fs["user_key"] == ukey) &
                             (fs["created_at_clean"] >= t0) &
                             (fs["created_at_clean"] <= t0 + pd.Timedelta(hours=24))]

                    n  = len(win)                         # total responses in burst
                    ns = win["survey_key"].nunique()                          if "survey_key" in win.columns else 0  # distinct surveys answered

                    # Step 5: Flag if response count meets the burst threshold (default 20).
                    fl = n >= 20

                    # Step 6: Assign a suspicion tier using survey diversity as the key signal.
                    # A user who completes many different surveys is more likely genuinely
                    # catching up. A user who repeats the same 1-2 surveys many times
                    # is more consistent with gaming the reward system.
                    sp = (
                        "Below threshold" if not fl else          # under the response threshold
                        "High"            if ns <= 1 and n > 50   # 1 survey + 50+ responses
                        else "Medium"     if ns <= 2              # 1-2 surveys in burst
                        else "Low — likely genuine"               # 3+ surveys = likely genuine
                    )

                    records.append({
                        "user_key":            ukey,
                        "burst_start":         t0,
                        "dormant_days":        round(row["gap_days"], 1),
                        "responses_in_burst":  n,
                        "surveys_in_burst":    ns,
                        "farming_suspicion":   sp,
                        "flagged_for_review":  fl,
                        "user_short":          ukey[:12] + "…",  # truncated for display only
                        "burst_date":          t0.date() if pd.notna(t0) else None,
                    })

                # Build the farming feature dataframe from all burst event records
                fm = pd.DataFrame(records)
                progress.progress(80, text="Saving feature files to Google Drive...")
                status_text.text("💾 Step 5/6 — Saving to Google Drive...")
                try:
                    os.makedirs(G5_FOLDER, exist_ok=True)
                    lt.drop(columns=['user_short'], errors='ignore').to_csv(LAZY_PATH, index=False)
                    fm.drop(columns=['user_short','burst_date'], errors='ignore').to_csv(FARM_PATH, index=False)
                except Exception as e:
                    st.warning(f'Auto-save to Drive failed: {e}')

                progress.progress(95, text='Loading feature files into dashboard...')
                status_text.text('📊 Step 6/6 — Preparing dashboard...')
                st.session_state.g5_lt    = lt
                st.session_state.g5_fm    = fm
                st.session_state.g5_step  = 5
                st.session_state.g5_rerun = False
                progress.progress(100, text='✅ Complete!')
                status_text.text('✅ All done! KPI dashboard is ready below.')
                st.rerun()
        else:
            st.error(f"❌ Raw BTA files not found — check {MAIN_FOLDER}")
        st.markdown("</div>", unsafe_allow_html=True)

        # ════════════════════════════════════════════════════════
        # STEP 3 — SAVE FEATURE FILES TO GOOGLE DRIVE
        # ════════════════════════════════════════════════════════
        step3_style = (
            f"background:#fdf8f0;border:2px solid {G5_AMBER};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step == 3 else
            f"background:#eef2ec;border:2px solid {G5_SAGE};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step > 3 else
            "background:#f5f2ee;border:2px dashed #c8b8a8;border-radius:12px;padding:16px;margin-bottom:12px;opacity:0.6"
        )
        st.markdown(f"<div style='{step3_style}'>", unsafe_allow_html=True)
        st.markdown(
            f"<div style='font-weight:700;font-size:14px;color:{G5_AMBER};margin-bottom:6px'>"
            f"💾 STEP 3 — Save Feature Files to Google Drive</div>"
            "<div style='font-size:12px;color:#7a6f68;margin-bottom:12px'>"
            "Feature files are saved to the G5 folder in Google Drive — "
            "available for download and loaded automatically next session.</div>",
            unsafe_allow_html=True
        )
        if g5_step >= 3 and st.session_state.g5_lt is not None:
            lt = st.session_state.g5_lt
            fm = st.session_state.g5_fm
            cd1, cd2 = st.columns(2)
            with cd1:
                st.markdown("**g5_lazy_text_flags.csv**")
                st.caption(f"{len(lt):,} rows · one per Textarea response with flags")
                st.download_button(
                    "⬇ Download g5_lazy_text_flags.csv",
                    lt.drop(columns=["user_short"], errors="ignore").to_csv(index=False).encode(),
                    "g5_lazy_text_flags.csv", "text/csv",
                    use_container_width=True, key="g5_dl_lt"
                )
            with cd2:
                st.markdown("**g5_farming_flags.csv**")
                st.caption(f"{len(fm):,} rows · one per burst event with suspicion tier")
                st.download_button(
                    "⬇ Download g5_farming_flags.csv",
                    fm.drop(columns=["user_short","burst_date"], errors="ignore").to_csv(index=False).encode(),
                    "g5_farming_flags.csv", "text/csv",
                    use_container_width=True, key="g5_dl_fm"
                )
            st.markdown("<br>", unsafe_allow_html=True)
            if g5_step == 3:
                try:
                    os.makedirs(G5_FOLDER, exist_ok=True)
                    lt.drop(columns=["user_short"], errors="ignore").to_csv(LAZY_PATH, index=False)
                    fm.drop(columns=["user_short","burst_date"], errors="ignore").to_csv(FARM_PATH, index=False)
                    st.success(f"✅ Saved to Google Drive: {G5_FOLDER}")
                    st.session_state.g5_step = 4
                    st.rerun()
                except Exception as e:
                    st.warning(f"⚠ Auto-save failed: {e}")
                    if st.button("Continue to Step 4", use_container_width=True, key="g5_step4_btn"):
                        st.session_state.g5_step = 4
                        st.rerun()
            else:
                st.success(f"✅ Files saved to Google Drive: {G5_FOLDER}")
        else:
            st.info("Complete Steps 1 & 2 first.")
        st.markdown("</div>", unsafe_allow_html=True)

        # ════════════════════════════════════════════════════════
        # STEP 4 — LOAD FEATURE FILES FROM GOOGLE DRIVE
        # ════════════════════════════════════════════════════════
        step4_style = (
            f"background:#eef0f8;border:2px solid {G5_BLUE};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step == 4 else
            f"background:#eef2ec;border:2px solid {G5_SAGE};border-radius:12px;padding:16px;margin-bottom:12px"
            if g5_step > 4 else
            "background:#f5f2ee;border:2px dashed #c8b8a8;border-radius:12px;padding:16px;margin-bottom:12px;opacity:0.6"
        )
        st.markdown(f"<div style='{step4_style}'>", unsafe_allow_html=True)
        st.markdown(
            f"<div style='font-weight:700;font-size:14px;color:{G5_BLUE};margin-bottom:6px'>"
            f"📂 STEP 4 — Load Feature Files from Google Drive</div>"
            "<div style='font-size:12px;color:#7a6f68;margin-bottom:12px'>"
            "Feature files are loaded from Google Drive to power the KPI dashboard below.</div>",
            unsafe_allow_html=True
        )
        if g5_step >= 4:
            if g5_step > 4:
                st.success(f"✅ Feature files loaded from Google Drive — dashboard is live below.")
            else:
                if os.path.exists(LAZY_PATH) and os.path.exists(FARM_PATH):
                    with st.spinner("Loading feature files from Google Drive..."):
                        lt_up = pd.read_csv(LAZY_PATH)
                        fm_up = pd.read_csv(FARM_PATH)
                        lt_up["user_short"] = lt_up["user_key"].str[:12] + "…"
                        fm_up["user_short"] = fm_up["user_key"].str[:12] + "…"
                        if "burst_start" in fm_up.columns:
                            fm_up["burst_start"] = pd.to_datetime(
                                fm_up["burst_start"], utc=True, errors="coerce")
                            fm_up["burst_date"] = fm_up["burst_start"].dt.date
                        st.session_state.g5_lt   = lt_up
                        st.session_state.g5_fm   = fm_up
                        st.session_state.g5_step = 5
                    st.success(f"✅ Loaded from {G5_FOLDER}")
                    st.rerun()
                else:
                    st.info("⏳ Waiting for Step 3 to save files to Google Drive...")
        else:
            st.info("Complete Steps 1–3 first.")
        st.markdown("</div>", unsafe_allow_html=True)


    # ════════════════════════════════════════════════════════
    # STEP 5 — LIVE KPI DASHBOARD (always visible, updates after pipeline)
    # ════════════════════════════════════════════════════════
    st.markdown(
        f"<div style='font-weight:700;font-size:14px;color:{G5_SAGE};"
        f"border-top:3px solid {G5_SAGE};padding-top:14px;margin:16px 0 12px'>"
        f"📊 KPI Dashboard</div>",
        unsafe_allow_html=True
    )

    # Show empty KPI cards by default, fill with data once pipeline is complete
    has_data = st.session_state.g5_step >= 5 and st.session_state.g5_lt is not None

    # KPI values — zeros if no data yet
    if has_data:
        lt_kpi = st.session_state.g5_lt.copy()
        fm_kpi = st.session_state.g5_fm.copy()
        n_textarea  = len(lt_kpi)
        n_lazy_flag = int(lt_kpi["lazy_text_flagged"].sum()) if "lazy_text_flagged" in lt_kpi.columns else 0
        n_bursts    = len(fm_kpi)
        n_farm_flag = int(fm_kpi["flagged_for_review"].sum()) if "flagged_for_review" in fm_kpi.columns else 0
        n_high      = int((fm_kpi["farming_suspicion"] == "High").sum()) if "farming_suspicion" in fm_kpi.columns else 0
        pct_lazy    = n_lazy_flag / max(n_textarea, 1) * 100
        kpi_vals = [f"{n_textarea:,}", f"{n_lazy_flag:,}", f"{n_bursts:,}",
                    f"{n_farm_flag:,}", f"{n_high:,}", "224 / 539"]
        kpi_subs = [f"For lazy-text detection", f"{pct_lazy:.1f}% of total",
                    "Dormant-then-burst", "Flagged for review",
                    "No high-risk events" if n_high==0 else "Requires review",
                    "41.6% response rate"]
        kpi_statuses = ["Textarea responses", "Needs attention", "Total detected",
                        "Needs attention", "On track" if n_high==0 else "Review", "Coverage"]
    else:
        kpi_vals     = ["—", "—", "—", "—", "—", "—"]
        kpi_subs     = ["Run pipeline to populate", "Run pipeline to populate",
                        "Run pipeline to populate", "Run pipeline to populate",
                        "Run pipeline to populate", "Run pipeline to populate"]
        kpi_statuses = ["Pending","Pending","Pending","Pending","Pending","Pending"]

    kpi_meta = [
        ("Free-text responses",  G5_BLUE,   "Open-ended Textarea answers from BTA survey data"),
        ("Lazy-text flagged",    G5_RED,    "Responses flagged by at least one lazy-text rule"),
        ("Burst events",         G5_ORANGE, "Users who returned after 60+ days dormant"),
        ("Farming flagged",      G5_RED,    "Burst events exceeding the response threshold"),
        ("High suspicion",       G5_AMBER,  "Single survey + 50+ responses in burst"),
        ("Coverage",             G5_SAGE,   "Profiled BTA members who answered at least one survey"),
    ]

    k1,k2,k3,k4,k5,k6 = st.columns(6)
    for col,(label,accent,tip),(val,sub,status) in zip(
        [k1,k2,k3,k4,k5,k6], kpi_meta,
        zip(kpi_vals, kpi_subs, kpi_statuses)
    ):
        sc = (G5_RED    if status in ("Needs attention","Review") else
              G5_SAGE   if status in ("On track","Coverage","Textarea responses") else
              "#9e9e9e" if status == "Pending" else G5_ORANGE)
        with col:
            st.markdown(f"""
            <div style='background:#fdf8f3;border-radius:12px;padding:18px 14px 14px;
                border-top:4px solid {accent if has_data else "#e0d5cc"};
                box-shadow:0 2px 8px rgba(0,0,0,0.08);
                min-height:160px;box-sizing:border-box;
                {"opacity:0.6;" if not has_data else ""}' title='{tip}'>
                <div style='font-size:13px;font-weight:600;color:#3d3530;margin-bottom:8px;
                    font-family:Inter,sans-serif'>{label}
                    <span style='color:{G5_BLUE};font-size:11px' title='{tip}'> ⓘ</span>
                </div>
                <div style='font-size:{"30px" if has_data else "24px"};font-weight:800;
                    color:{accent if has_data else "#c8b8a8"};
                    font-family:Inter,sans-serif;line-height:1.1;margin-bottom:6px'>{val}</div>
                <div style='font-size:11px;color:#7a6f68;font-style:italic;margin-bottom:10px'>{sub}</div>
                <div style='font-size:11px;font-weight:600;color:{sc}'>● {status}</div>
            </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    if not has_data:
        st.markdown(
            "<div style='background:#f5f2ee;border-radius:8px;padding:12px 16px;"
            "text-align:center;color:#9e9088;font-size:12px'>"
            "⬆ Complete the data pipeline above to populate the KPI cards and charts."
            "</div>", unsafe_allow_html=True
        )

    if st.session_state.g5_step >= 5 and st.session_state.g5_lt is not None:
        lt = st.session_state.g5_lt.copy()
        # Sidebar threshold controls
        st.sidebar.markdown("---")
        st.sidebar.markdown("**G5 Thresholds**")
        min_len      = st.sidebar.slider("Min length (chars)",     1,  50,  10, key="g5_min_len")
        repeat_ratio = st.sidebar.slider("Repeat-char ratio",      0.5, 1.0, 0.7, 0.05, key="g5_repeat")
        dormant_days = st.sidebar.slider("Dormant days threshold", 7,  180, 60, key="g5_dormant")
        burst_min    = st.sidebar.slider("Min burst responses",    5,  200, 20, key="g5_burst")

        # Apply thresholds to data
        lt = st.session_state.g5_lt.copy()
        fm = st.session_state.g5_fm.copy()

        # Re-score with sidebar thresholds
        for c in ["flag_too_short","flag_repeat_char","flag_keyboard_smash",
                  "flag_whitespace_only","flag_gibberish"]:
            if c not in lt.columns: lt[c] = False
        lt["flag_too_short"]    = lt["char_count"] < min_len
        lt["flag_repeat_char"]  = lt["repeat_ratio"] >= repeat_ratio
        lt["lazy_text_flagged"] = lt[["flag_too_short","flag_repeat_char","flag_keyboard_smash",
                                      "flag_whitespace_only","flag_gibberish"]].any(axis=1)
        fm["flagged_for_review"] = fm["responses_in_burst"] >= burst_min

        def g5_suspicion(row):
            if not row["flagged_for_review"]: return "Below threshold"
            if row.get("surveys_in_burst",2) <= 1 and row["responses_in_burst"] > 50: return "High"
            if row.get("surveys_in_burst",2) <= 2: return "Medium"
            return "Low — likely genuine"
        fm["farming_suspicion"] = fm.apply(g5_suspicion, axis=1)

        # KPI values
        n_textarea  = len(lt)
        n_lazy_flag = int(lt["lazy_text_flagged"].sum())
        n_bursts    = len(fm)
        n_farm_flag = int(fm["flagged_for_review"].sum())
        n_high      = int((fm["farming_suspicion"] == "High").sum())
        pct_lazy    = n_lazy_flag / max(n_textarea, 1) * 100


        # Feasibility verdict
        st.markdown(
            f"<div style='background:#eef2ec;border:1px solid {G5_SAGE};"
            f"border-left:4px solid {G5_SAGE};border-radius:8px;"
            f"padding:12px 16px;color:#3d3530;font-size:13px;margin-bottom:16px'>"
            f"<b>🟢 FEASIBILITY VERDICT — GREEN</b><br>"
            f"{n_textarea:,} free-text responses confirmed from BTA raw data. "
            f"Burst behaviour confirmed. Both detectors are fully supported."
            f"</div>", unsafe_allow_html=True
        )

        # Tabs
        g5_tab1, g5_tab2, g5_tab3, g5_tab4 = st.tabs([
            "🔤  Lazy-Text Detector",
            "🚜  Reward Farming Detector",
            "📋  Raw Data & Export",
            "📂  Data Sources",
        ])

        with g5_tab1:
            c1, c2 = st.columns(2)
            flag_cols   = ["flag_too_short","flag_repeat_char","flag_keyboard_smash",
                           "flag_whitespace_only","flag_gibberish"]
            flag_labels = ["Too short","Repeat char","Keyboard smash","Whitespace only","Gibberish"]
            flag_vals   = [int(lt[c].sum()) for c in flag_cols]

            with c1:
                fig = go.Figure(go.Bar(
                    x=flag_labels, y=flag_vals, text=flag_vals, textposition="outside",
                    marker=dict(color=[G5_RED if v>0 else G5_SAGE for v in flag_vals],
                                line=dict(width=0)),
                    hovertemplate="<b>%{x}</b><br>%{y} flagged<extra></extra>"))
                fig.update_layout(**g5_layout("Flag counts", 280))
                st.plotly_chart(fig, use_container_width=True, key="g5_flags")
            with c2:
                fig2 = go.Figure(go.Pie(
                    labels=["Flagged","Clean"],
                    values=[n_lazy_flag, n_textarea-n_lazy_flag],
                    hole=0.55, marker=dict(colors=[G5_RED, G5_SAGE])))
                fig2.update_layout(**g5_layout("Flagged vs clean", 280))
                fig2.update_layout(showlegend=True,
                    legend=dict(orientation="h", y=-0.1, font_color="#3d3530"))
                st.plotly_chart(fig2, use_container_width=True, key="g5_pie")

            st.markdown(
                "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                "<b>What this means:</b> The bar chart shows how many responses triggered each flag type. <b>\u201cToo short\u201d dominates</b> \u2014 meaning most flagged users wrote too little, not gibberish or random text. The donut shows the overall split: most responses are clean, but the flagged minority is worth reviewing."
                "</div>", unsafe_allow_html=True
            )

            # Length histogram
            fig3 = go.Figure()
            fig3.add_histogram(x=lt[~lt["lazy_text_flagged"]]["char_count"].clip(upper=300),
                nbinsx=40, name="Clean",
                marker=dict(color=G5_SAGE, opacity=0.7))
            fig3.add_histogram(x=lt[lt["lazy_text_flagged"]]["char_count"].clip(upper=300),
                nbinsx=40, name="Flagged",
                marker=dict(color=G5_RED, opacity=0.8))
            fig3.add_vline(x=min_len, line_dash="dash", line_color=G5_AMBER)
            fig3.add_annotation(x=min_len, y=1.05, xref="x", yref="paper",
                text=f"Min length ({min_len})", font=dict(color=G5_AMBER, size=11),
                showarrow=False, xanchor="left")
            fig3.update_layout(**g5_layout("Response length distribution", 280))
            fig3.update_layout(showlegend=True, barmode="overlay",
                legend=dict(orientation="h", y=1.12, font_color="#3d3530"))
            st.plotly_chart(fig3, use_container_width=True, key="g5_hist")

            st.markdown(
                "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                "<b>What this means:</b> Each bar shows how many responses fall within that character-count range. The amber dashed line marks the minimum-length threshold. Responses to the left of that line are flagged \u201ctoo short\u201d. Clean responses (sage) spread across a wide range; flagged ones (red) cluster near zero \u2014 confirming the threshold is catching genuinely brief answers."
                "</div>", unsafe_allow_html=True
            )

            # Word clouds
            wc1, wc2 = st.columns(2)
            with wc1:
                st.caption("Word cloud — all responses")
                all_text = " ".join(lt["answer_clean"].dropna().astype(str).tolist())
                if all_text.strip():
                    wc = WordCloud(width=600, height=260, background_color="white",
                        colormap="Dark2", max_words=100, collocations=False,
                        stopwords={"the","a","and","is","in","it","of","to","was","for",
                                   "on","are","with","that","this","be","at","by","an",
                                   "we","as","or"}).generate(all_text)
                    fig_wc, ax = plt.subplots(figsize=(6,2.6))
                    ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
                    plt.tight_layout(pad=0)
                    st.pyplot(fig_wc, use_container_width=True); plt.close(fig_wc)
            with wc2:
                st.caption("Word cloud — flagged only")
                flag_text = " ".join(
                    lt[lt["lazy_text_flagged"]]["answer_clean"].dropna().astype(str).tolist())
                if flag_text.strip():
                    wc2c = WordCloud(width=600, height=260, background_color="white",
                        colormap="Reds", max_words=100, collocations=False,
                        stopwords={"the","a","and","is","in","it","of","to","was","for",
                                   "on","are","with","that","this","be","at","by","an",
                                   "we","as","or"}).generate(flag_text)
                    fig_wc2, ax2 = plt.subplots(figsize=(6,2.6))
                    ax2.imshow(wc2c, interpolation="bilinear"); ax2.axis("off")
                    plt.tight_layout(pad=0)
                    st.pyplot(fig_wc2, use_container_width=True); plt.close(fig_wc2)


            st.markdown(
                "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                "<b>What this means:</b> The left cloud shows the most common words across all responses. The right cloud shows words that appear most in flagged responses only. Comparing the two helps distinguish filler/lazy patterns from genuinely common vocabulary."
                "</div>", unsafe_allow_html=True
            )

            # ── Scatter: length vs repeat-char ratio ─────────────
            st.markdown("**Length vs repeat-character ratio**")
            fig_sc = go.Figure()
            for flagged, color, name in [(False, G5_SAGE, "Clean"), (True, G5_RED, "Flagged")]:
                sub = lt[lt["lazy_text_flagged"] == flagged]
                fig_sc.add_scatter(
                    x=sub["char_count"].clip(upper=300), y=sub["repeat_ratio"],
                    mode="markers", name=name,
                    marker=dict(color=color, size=5, opacity=0.6),
                    hovertemplate=f"<b>{name}</b><br>Length: %{{x}}<br>Ratio: %{{y:.2f}}<extra></extra>")
            fig_sc.add_hline(y=repeat_ratio, line_dash="dash", line_color=G5_AMBER)
            fig_sc.add_annotation(x=1.01, y=repeat_ratio, xref="paper", yref="y",
                text=f"Threshold ({repeat_ratio})",
                font=dict(color=G5_AMBER, size=11), showarrow=False, xanchor="left")
            fig_sc.add_vline(x=min_len, line_dash="dot", line_color=G5_BLUE)
            fig_sc.add_annotation(x=min_len, y=1.05, xref="x", yref="paper",
                text=f"Min len ({min_len})",
                font=dict(color=G5_BLUE, size=11), showarrow=False, xanchor="left")
            fig_sc.update_layout(**g5_layout("", 320))
            fig_sc.update_layout(showlegend=True,
                legend=dict(orientation="h", y=1.12, font_color="#3d3530"))
            fig_sc.update_xaxes(title_text="Character count")
            fig_sc.update_yaxes(title_text="Repeat-char ratio")
            st.plotly_chart(fig_sc, use_container_width=True, key="g5_scatter")

            st.markdown(
                "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                "<b>What this means:</b> Each dot is one response. The amber horizontal line is the repeat-ratio threshold; the blue dotted vertical line is the minimum-length threshold. Points in the bottom-left (short, low repeat) are flagged for length. Points high on the y-axis are flagged for repetition (e.g. \u201caaaaaaa\u201d). This scatter helps you judge whether the thresholds need adjusting."
                "</div>", unsafe_allow_html=True
            )

            # ── Flagged responses table ───────────────────────────
            st.markdown("**Flagged responses — for review only**")
            st.caption("⚠ These are flagged for review only — not confirmed low-effort. "
                       "Adjust sidebar thresholds to recalibrate.")

            show_flagged = st.checkbox("Show flagged only", value=True, key="g5_show_flagged")
            search_lt    = st.text_input("🔍 Search answers...", placeholder="Type to filter",
                                          key="g5_search_lt")

            view_lt = lt[lt["lazy_text_flagged"]] if show_flagged else lt
            if search_lt:
                view_lt = view_lt[
                    view_lt["answer_clean"].astype(str).str.contains(
                        search_lt, case=False, na=False)]

            disp_lt = view_lt[["user_short","answer_clean","char_count",
                                "repeat_ratio","lazy_text_reason"]].copy()
            disp_lt.columns = ["User","Answer","Chars","Repeat ratio","Flags"]
            st.caption(f"Showing {len(disp_lt):,} rows")
            st.dataframe(disp_lt, use_container_width=True, hide_index=True, height=320)
            st.download_button("⬇ Download lazy-text flags CSV",
                lt.drop(columns=["user_short"],errors="ignore").to_csv(index=False).encode(),
                "g5_lazy_text_flagged.csv","text/csv", key="g5_tab_dl_lt")

        with g5_tab2:
            if len(fm) > 0:
                susp_order  = ["High","Medium","Low — likely genuine","Below threshold"]
                susp_counts = fm["farming_suspicion"].value_counts().reindex(
                    [s for s in susp_order if s in fm["farming_suspicion"].values]).fillna(0)

                ca, cb = st.columns(2)
                with ca:
                    fig4 = go.Figure(go.Bar(
                        x=susp_counts.index, y=susp_counts.values,
                        text=susp_counts.values.astype(int), textposition="outside",
                        marker=dict(color=[SUSP_COLORS.get(s,G5_GRAY) for s in susp_counts.index],
                                    line=dict(width=0))))
                    fig4.update_layout(**g5_layout("Suspicion tiers", 280))
                    st.plotly_chart(fig4, use_container_width=True, key="g5_susp")
                with cb:
                    fig5 = go.Figure(go.Pie(
                        labels=susp_counts.index, values=susp_counts.values, hole=0.5,
                        marker=dict(colors=[SUSP_COLORS.get(s,G5_GRAY) for s in susp_counts.index])))
                    fig5.update_layout(**g5_layout("", 280))
                    fig5.update_layout(showlegend=True,
                        legend=dict(orientation="h", y=-0.12, font_color="#3d3530"))
                    st.plotly_chart(fig5, use_container_width=True, key="g5_susp_pie")

                st.markdown(
                    "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                    "<b>What this means:</b> The bar chart counts burst events per suspicion tier. The donut shows their proportional split. Focus review effort on <b>High</b> and <b>Medium</b> tier events first \u2014 these combine long dormancy with large bursts, the strongest farming signal. \u201cBelow threshold\u201d events did not meet the minimum burst size and are shown for completeness only."
                    "</div>", unsafe_allow_html=True
                )

                # Scatter
                fig6 = go.Figure()
                for tier in susp_order:
                    sub = fm[fm["farming_suspicion"] == tier]
                    if len(sub) == 0: continue
                    fig6.add_scatter(
                        x=sub["dormant_days"].clip(upper=365),
                        y=sub["responses_in_burst"],
                        mode="markers", name=tier,
                        marker=dict(color=SUSP_COLORS.get(tier,G5_GRAY), size=8, opacity=0.75),
                        hovertemplate=f"<b>{tier}</b><br>Dormant: %{{x:.0f}}d<br>Responses: %{{y}}<extra></extra>")
                fig6.add_hline(y=burst_min, line_dash="dash", line_color=G5_AMBER)
                fig6.add_annotation(x=1.01, y=burst_min, xref="paper", yref="y",
                    text=f"Burst min ({burst_min})", font=dict(color=G5_AMBER,size=11),
                    showarrow=False, xanchor="left")
                fig6.update_layout(**g5_layout("Dormant days vs burst size", 300))
                fig6.update_layout(showlegend=True,
                    legend=dict(orientation="h", y=1.12, font_color="#3d3530"))
                fig6.update_xaxes(title_text="Days dormant before burst")
                fig6.update_yaxes(title_text="Responses in burst window")
                st.plotly_chart(fig6, use_container_width=True, key="g5_sc2")

                st.markdown(
                    "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                    "<b>What this means:</b> Each dot is a burst event. The x-axis shows how many days the user was dormant before the burst; the y-axis shows how many responses they submitted in that window. The amber dashed line is the minimum burst threshold. Points in the top-right quadrant \u2014 long dormancy + large burst \u2014 are the strongest farming signals and appear red (High suspicion)."
                    "</div>", unsafe_allow_html=True
                )

                # Timeline
                if "burst_date" in fm.columns:
                    tl = fm.groupby("burst_date").agg(
                        events=("user_key","count"),
                        flagged=("flagged_for_review","sum")).reset_index()
                    fig7 = go.Figure()
                    fig7.add_bar(x=tl["burst_date"], y=tl["events"],
                        name="All bursts", marker=dict(color=G5_BLUE, line=dict(width=0)))
                    fig7.add_bar(x=tl["burst_date"], y=tl["flagged"],
                        name="Flagged", marker=dict(color=G5_RED, line=dict(width=0)))
                    fig7.update_layout(**g5_layout("Burst events over time", 240))
                    fig7.update_layout(showlegend=True, barmode="overlay",
                        legend=dict(orientation="h", y=1.12, font_color="#3d3530"))
                    st.plotly_chart(fig7, use_container_width=True, key="g5_timeline")

                st.markdown(
                    "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                    "<b>What this means:</b> This timeline shows when burst events occurred. Blue bars are all burst events; red overlay marks flagged ones. Clusters on specific dates may reflect survey campaigns that triggered a wave of fast responses \u2014 useful for distinguishing coordinated farming from genuine catch-up periods."
                    "</div>", unsafe_allow_html=True
                )

                # RPD histogram
                fig8 = go.Figure()
                fig8.add_histogram(x=fm["responses_in_burst"].clip(upper=300),
                    nbinsx=30, name="All burst events",
                    marker=dict(color=G5_BLUE, opacity=0.7))
                fig8.add_histogram(x=fm[fm["flagged_for_review"]]["responses_in_burst"].clip(upper=300),
                    nbinsx=30, name="Flagged farming",
                    marker=dict(color=G5_RED, opacity=0.8))
                fig8.add_vline(x=burst_min, line_dash="dash", line_color=G5_AMBER)
                fig8.add_annotation(x=burst_min, y=1.05, xref="x", yref="paper",
                    text=f"Burst threshold ({burst_min})",
                    font=dict(color=G5_AMBER, size=11), showarrow=False, xanchor="left")
                fig8.update_layout(**g5_layout("Responses-per-user-per-day distribution", 280))
                fig8.update_layout(showlegend=True, barmode="overlay",
                    legend=dict(orientation="h", y=1.12, font_color="#3d3530"))
                st.plotly_chart(fig8, use_container_width=True, key="g5_rpd")

                st.markdown(
                    "<div style='background:#ffffff;border-radius:10px;padding:16px 20px;border-left:4px solid #e38f55;color:#3d3530;font-size:13px;margin-top:12px;margin-bottom:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06)'>"
                    "<b>What this means:</b> Distribution of burst sizes (responses per user per day) across all events. Blue = all burst events; red overlay = flagged farming events. The amber line marks the burst threshold \u2014 only events to the right of it were considered bursts. A heavy red tail on the right indicates a small number of users submitting extremely high counts in a single day."
                    "</div>", unsafe_allow_html=True
                )

                # Farming table
                st.markdown("**Flagged burst events — for review only**")
                tier_filter = st.multiselect("Filter by tier", options=susp_order,
                    default=[s for s in susp_order if s != "Below threshold"], key="g5_tier")
                view_fm = fm[fm["farming_suspicion"].isin(tier_filter)] if tier_filter else fm
                disp_fm = view_fm[["user_short","burst_start","dormant_days",
                    "responses_in_burst","surveys_in_burst","farming_suspicion"]].copy()
                disp_fm.columns = ["User","Burst start","Dormant days",
                    "Responses","Surveys","Suspicion"]
                st.caption(f"{len(disp_fm):,} events — flags are for review only")
                st.dataframe(disp_fm, use_container_width=True, hide_index=True, height=320)
                st.download_button("⬇ Download farming flags CSV",
                    fm.drop(columns=["user_short","burst_date"],errors="ignore").to_csv(index=False).encode(),
                    "g5_farming_flagged.csv","text/csv", key="g5_tab_dl_fm")
            else:
                st.info("No farming data available.")

        with g5_tab3:
            st.markdown("**Lazy-text feature table**")
            raw_lt = lt.drop(columns=["user_short"], errors="ignore")
            st.caption(f"{len(raw_lt):,} rows")
            st.dataframe(raw_lt, use_container_width=True, hide_index=True, height=280)

            st.markdown("**Farming feature table**")
            raw_fm = fm.drop(columns=["user_short","burst_date"], errors="ignore")
            st.caption(f"{len(raw_fm):,} rows")
            st.dataframe(raw_fm, use_container_width=True, hide_index=True, height=280)

        with g5_tab4:
            st.markdown("**Where does G5 data come from?**")
            st.markdown(
                "<div style='background:#fdf8f3;border:1px solid #f4b35e;"
                "border-left:4px solid #e26655;border-radius:8px;"
                "padding:14px 18px;color:#3d3530;font-size:13px;margin-bottom:16px'>"
                "<b>Data pipeline:</b> BTA raw files → Part B Colab notebook → "
                "G5 feature files → This dashboard.<br><br>"
                "G5 uses only 2 of the 5 BTA raw files:<br>"
                "• <b>fct_survey_responses_*.csv</b> (15,385 rows) — core survey answers<br>"
                "• <b>dim_questions_*.csv</b> (223 rows) — to identify Textarea question type<br><br>"
                "The notebook filters, computes flags, detects burst events, and saves the results. "
                "The dashboard reads those pre-processed files — making it fast and lightweight."
                "</div>", unsafe_allow_html=True
            )
            raw_src = pd.DataFrame({
                "File": ["fct_survey_responses_*.csv","dim_questions_*.csv",
                         "dim_surveys_*.csv","dim_users_*.csv","wp_users_*.csv"],
                "Rows": ["15,385","223","33","539","478"],
                "Used by G5?": ["✅ Yes","✅ Yes","❌ No","❌ No","❌ No"],
                "Purpose": [
                    "Core data — Textarea filter + farming timeline",
                    "Identifies Textarea question type",
                    "Not needed by G5",
                    "Not needed by G5",
                    "Not needed by G5",
                ]
            })
            st.dataframe(raw_src, use_container_width=True, hide_index=True)

        # Reset button
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("🔄 Reset — start over with new data", key="g5_reset"):
            for k in ["g5_step","g5_fct","g5_questions","g5_lt","g5_fm"]:
                st.session_state[k] = None if k != "g5_step" else 1
            st.rerun()

    elif st.session_state.g5_step < 5:
        st.markdown(
            "<div style='background:#f5f2ee;border-radius:10px;padding:20px;"
            "text-align:center;color:#7a6f68;font-size:13px;margin-top:8px'>"
            "📊 Complete Steps 1–4 above to unlock the live KPI dashboard."
            "</div>", unsafe_allow_html=True
        )

Overwriting app.py


In [26]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

# Step 0: Ensure all previous Streamlit and ngrok processes are killed
print("Attempting to kill existing Streamlit and ngrok processes...")
!pkill -f streamlit || true # `|| true` prevents error if no process is found
ngrok.kill() # Forcefully kill any pyngrok-managed ngrok process
!pkill -f ngrok || true # Forcefully kill any other ngrok process not managed by pyngrok

# Add a longer delay after ngrok termination to ensure the endpoint is released
print("Waiting 90 seconds for processes to fully terminate and endpoint to release...")
time.sleep(90) # Increased to 90 seconds for more time for endpoint release

# Step 1: Start Streamlit in the background
print("Starting Streamlit application...")
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.headless", "true"])

# Step 2: Wait for Streamlit to fully start
print("Waiting 15 seconds for Streamlit to start...")
time.sleep(15)

# Step 3: Connect ngrok with retry logic
# IMPORTANT: Replace '3F0wLKWEgvktzNUIKpiSu4VFKHi_3nEaR2swo5ToKgDhaYnYU' with your actual ngrok authtoken.
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('3F3DlzOjI8t7AJnjdnPIgaBtV76_5pmFy6sxhjEoGujY2FdKS')
public_url = None
max_retries = 5 # Increased retries for robustness
retry_delay = 30 # Increased to 30 seconds for more time for endpoint release between retries

print(f"Attempting to connect ngrok (max {max_retries} retries)...")
for i in range(max_retries):
    try:
        public_url = ngrok.connect(8501)
        print("✅ Dashboard is live at:", public_url)
        break # Connection successful, exit loop
    except PyngrokNgrokHTTPError as e:
        print(f"Ngrok connection attempt {i+1}/{max_retries} failed: {e}")
        if "already online" in str(e) and i < max_retries - 1:
            print(f"Endpoint reported as 'already online'. Retrying in {retry_delay} seconds...")
            ngrok.kill() # Kill pyngrok-managed process before retry
            !pkill -f ngrok || true # Ensure all ngrok processes are killed again at OS level
            time.sleep(retry_delay)
        else:
            print("Failed to establish ngrok tunnel after multiple attempts.")
            raise # Re-raise the exception if it's the last retry or a different error
    except Exception as e:
        print(f"An unexpected error occurred during ngrok connection attempt {i+1}/{max_retries}: {e}")
        raise # Re-raise for unexpected errors

if public_url is None:
    print("❌ Failed to get ngrok public URL.")

In [ ]:
!pkill -f streamlit
!pkill -f ngrok
!ngrok kill

In [ ]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.kill()  # closes old tunnels inside this notebook

ngrok.set_auth_token("3F3DlzOjI8t7AJnjdnPIgaBtV76_5pmFy6sxhjEoGujY2FdKS")

public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

# Step 0: Ensure all previous Streamlit and ngrok processes are killed
print("Attempting to kill existing Streamlit and ngrok processes...")
!pkill -f streamlit || true # `|| true` prevents error if no process is found
ngrok.kill() # Forcefully kill any pyngrok-managed ngrok process
!pkill -f ngrok || true # Forcefully kill any other ngrok process not managed by pyngrok

# Add a longer delay after ngrok termination to ensure the endpoint is released
print("Waiting 60 seconds for processes to fully terminate and endpoint to release...")
time.sleep(60) # Increased to 60 seconds for more time for endpoint release

# Step 2: Start Streamlit in the background
print("Starting Streamlit application...")
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.headless", "true"])

# Step 3: Wait for Streamlit to fully start
print("Waiting 15 seconds for Streamlit to start...")
time.sleep(15)

# Step 4: Connect ngrok with retry logic
# IMPORTANT: Replace 'YOUR_AUTH_TOKEN_HERE' with your actual ngrok authtoken.
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('3F3DlzOjI8t7AJnjdnPIgaBtV76_5pmFy6sxhjEoGujY2FdKS') # Changed auth token to a dummy to avoid exposing real one. User should replace this.
public_url = None
max_retries = 3
retry_delay = 30 # Increased to 30 seconds (Increased retry delay)

print(f"Attempting to connect ngrok (max {max_retries} retries)...")
for i in range(max_retries):
    try:
        # Add --log=stdout to get more detailed logging from ngrok
        public_url = ngrok.connect(8501, options={'log': 'stdout'})
        print("✅ Dashboard is live at:", public_url)
        break # Connection successful, exit loop
    except PyngrokNgrokHTTPError as e:
        print(f"Ngrok connection attempt {i+1}/{max_retries} failed: {e}")
        if "already online" in str(e) and i < max_retries - 1:
            print(f"Endpoint reported as 'already online'. Retrying in {retry_delay} seconds...")
            ngrok.kill() # Kill ngrok process before retry
            !pkill -f ngrok || true # Ensure all ngrok processes are killed again
            time.sleep(retry_delay)
        else:
            print("Failed to establish ngrok tunnel after multiple attempts.")
            raise # Re-raise the exception if it's the last retry or a different error
    except Exception as e:
        print(f"An unexpected error occurred during ngrok connection attempt {i+1}/{max_retries}: {e}")
        raise # Re-raise for unexpected errors

if public_url is None:
    print("❌ Failed to get ngrok public URL.")

In [ ]:
## I just remove this fortune temporarily to check whether it will run on my end!


## from pyngrok import ngrok

# IMPORTANT: Replace 'YOUR_AUTH_TOKEN_HERE' with your actual ngrok authtoken.
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
## ngrok.set_auth_token('3F3DlzOjI8t7AJnjdnPIgaBtV76_5pmFy6sxhjEoGujY2FdKS')

## public_url = ngrok.connect(8501)
## print(public_url)

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

# Step 0: Ensure all previous Streamlit and ngrok processes are killed
print("Attempting to kill existing Streamlit and ngrok processes...")
!pkill -f streamlit
ngrok.kill() # Forcefully kill any pyngrok-managed ngrok process
!pkill -f ngrok # Forcefully kill any other ngrok process not managed by pyngrok

# Add a longer delay after ngrok termination to ensure the endpoint is released
print("Waiting 45 seconds for processes to fully terminate and endpoint to release...")
time.sleep(45) # Increased to 45 seconds for more time for endpoint release

# Step 2: Start Streamlit in the background
print("Starting Streamlit application...")
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.headless", "true"])

# Step 3: Wait for Streamlit to fully start
print("Waiting 15 seconds for Streamlit to start...")
time.sleep(15)

# Step 4: Connect ngrok with retry logic
ngrok.set_auth_token('3F3DlzOjI8t7AJnjdnPIgaBtV76_5pmFy6sxhjEoGujY2FdKS')
public_url = None
max_retries = 3
retry_delay = 10 # seconds (Increased retry delay)

print(f"Attempting to connect ngrok (max {max_retries} retries)...")
for i in range(max_retries):
    try:
        public_url = ngrok.connect(8501)
        print("✅ Dashboard is live at:", public_url)
        break # Connection successful, exit loop
    except PyngrokNgrokHTTPError as e:
        print(f"Ngrok connection attempt {i+1}/{max_retries} failed: {e}")
        if "already online" in str(e) and i < max_retries - 1:
            print(f"Endpoint reported as 'already online'. Retrying in {retry_delay} seconds...")
            ngrok.kill() # Kill ngrok process before retry
            !pkill -f ngrok # Ensure all ngrok processes are killed again
            time.sleep(retry_delay)
        else:
            print("Failed to establish ngrok tunnel after multiple attempts.")
            raise # Re-raise the exception if it's the last retry or a different error
    except Exception as e:
        print(f"An unexpected error occurred during ngrok connection attempt {i+1}/{max_retries}: {e}")
        raise # Re-raise for unexpected errors

if public_url is None:
    print("❌ Failed to get ngrok public URL.")

In [ ]:
!pkill -f streamlit
!pkill -f ngrok

from pyngrok import ngrok
ngrok.kill()

In [27]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Kill any existing tunnels
ngrok.kill()

# Step 2: Start Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501",
                  "--server.headless", "true"])

# Step 3: Wait for Streamlit to fully start
time.sleep(15)

# Step 4: Connect ngrok
ngrok.set_auth_token('3CxJdmutmnlp5s8XkRuXWIxHJxo_4AHv8mq4XjXWSHiJ1qGMd')
public_url = ngrok.connect(8501)
print("✅ Dashboard is live at:", public_url)

✅ Dashboard is live at: NgrokTunnel: "https://arousal-catnip-synapse.ngrok-free.dev" -> "http://localhost:8501"
